# GridSentinel AI - run everything
Run the cells top to bottom (Runtime > Run all). Simulation on IEEE 14 / 30 / 57 / 118 pandapower grids; true voltage is used only for scoring.

In [ ]:
!pip -q install pandapower==3.5.6 scikit-learn scipy matplotlib streamlit pytest

In [ ]:
import os
os.makedirs('/content/gridsentinel', exist_ok=True)
os.chdir('/content/gridsentinel')

In [ ]:
%%writefile gridsentinel_core.py
"""
GridSentinel AI - core pipeline
================================
Trust-aware decision layer for smart-grid state estimation under false-data
injection attacks (FDIA), IEEE 14-bus system (pandapower).

DESIGN RULE (the "no oracle" rule)
----------------------------------
* `run_pipeline(model, z, cfg, calibration)` is the LIVE path. It receives only
  what a control-centre operator sees: the measurement vector `z`, the public
  network model, and an offline-calibrated gate. It never receives the true
  state, the attack label or the attacked meter list.
* Everything that needs the truth (scenario generation, attacks, error
  metrics, recovery tables) lives in clearly separated functions:
  `make_scenarios`, `apply_attack`, `recovery_table`, `evaluate_run`.

Measurement set (full): bus voltage magnitudes (14), bus P/Q injections (28),
branch P/Q flows at BOTH ends (80) = 122 measurements for 27 states.

Pipeline: robust (Huber) AC-WLS fit -> trust score per measurement from
normalized residuals with a median/MAD robust scale -> isolation with an
observability guard -> re-estimation on the trusted set -> runtime-only gate
features -> calibrated risk bound (split-conformal, Mondrian bins) ->
PROCEED / CAUTION / VALIDATE.
"""
from __future__ import annotations

import copy
import json
import math
import time
from dataclasses import dataclass, field

import numpy as np

KINDS = ("V", "Pinj", "Qinj", "Pf", "Qf", "Pt", "Qt")
FEATURE_NAMES = (
    "log_J",          # log10 of chi-square statistic per degree of freedom (retained set)
    "log_zmax",       # log10 of largest remaining robust z-score
    "frac_iso",       # fraction of measurements isolated
    "log_shift",      # log10 of max change in voltage estimate caused by recovery
    "log_disagree",   # log10 of max disagreement between WLS recovery and Huber estimate
    "log_sdV",        # log10 of model-based std of the voltage estimate (from gain matrix)
    "n_crit",         # number of critical (undetectable-error) measurements retained
    "obs_blocked",    # 1 if suspicious measurements could not be removed (observability)
    "log_obs",        # log10 of observability margin (relative smallest eigenvalue)
    "log_J_raw",      # log10 chi-square per dof of the RAW (uncleaned) WLS fit: conflict before cleaning
    "log_zfirst",     # log10 of the largest z-score seen BEFORE any isolation
)
K_MAX = 8  # threat model: attacker can compromise at most K_MAX meters
IN_MODEL_MIX = {
    "none": 0.15,
    "random_single": 0.15,
    "single_small": 0.15,
    "multi_coordinated": 0.15,
    "stealth_partial": 0.20,
    "adaptive": 0.20,
}
ATTACK_MODES = (
    "none", "random_single", "single_small", "multi_coordinated",
    "stealth_partial", "adaptive", "stealth_full", "custom_voltage", "custom_stealth",
)


# =============================================================================
# 1. Network / AC measurement model
# =============================================================================
@dataclass
class Fit:
    x: np.ndarray
    converged: bool
    iters: int


@dataclass
class Analysis:
    rN: np.ndarray        # normalized residuals (0 for critical / unavailable)
    crit: np.ndarray      # critical measurements (error undetectable)
    res: np.ndarray       # raw residuals z - h(x)
    Ginv: np.ndarray
    lam_min: float


class GridModel:
    """IEEE-14 AC measurement model h(x), Jacobian, WLS / Huber estimators."""

    def __init__(self, meas_set: str = "full", sigma_v: float = 0.004, sigma_pq: float = 1.0):
        import pandapower as pp
        import pandapower.networks as pn

        net = pn.case14()
        pp.runpp(net, calculate_voltage_angles=True, numba=False)
        ppc = net._ppc
        lk = np.asarray(net._pd2ppc_lookups["bus"])
        nb = len(net.bus)
        if not np.array_equal(lk[:nb], np.arange(nb)):
            raise RuntimeError("Unexpected bus ordering in pandapower ppc")
        internal = ppc["internal"]
        self.net = net
        self.base = float(ppc["baseMVA"])
        self.Ybus = np.asarray(internal["Ybus"].todense(), dtype=complex)
        self.Yf = np.asarray(internal["Yf"].todense(), dtype=complex)
        self.Yt = np.asarray(internal["Yt"].todense(), dtype=complex)
        br = ppc["branch"]
        self.f = br[:, 0].real.astype(int)
        self.t = br[:, 1].real.astype(int)
        self.nb = nb
        self.nl = len(self.f)
        self.slack = int(net.ext_grid.bus.iloc[0])
        self.nonref = np.array([b for b in range(nb) if b != self.slack])
        self.n_state = 2 * nb - 1
        self.Cf = np.zeros((self.nl, nb))
        self.Ct = np.zeros((self.nl, nb))
        self.Cf[np.arange(self.nl), self.f] = 1.0
        self.Ct[np.arange(self.nl), self.t] = 1.0

        self.kinds = np.array(["V"] * nb + ["Pinj"] * nb + ["Qinj"] * nb
                              + ["Pf"] * self.nl + ["Qf"] * self.nl + ["Pt"] * self.nl + ["Qt"] * self.nl)
        self.elem = np.concatenate([np.arange(nb)] * 3 + [np.arange(self.nl)] * 4)
        self.loc = np.concatenate([np.arange(nb)] * 3 + [self.f, self.f, self.t, self.t])
        self.m = len(self.kinds)
        self.sigma = np.where(self.kinds == "V", sigma_v, sigma_pq).astype(float)
        self.meas_set = meas_set
        if meas_set == "full":
            self.avail = np.ones(self.m, bool)
        elif meas_set == "basic":  # voltages + injections only (ablation)
            self.avail = np.isin(self.kinds, ["V", "Pinj", "Qinj"])
        else:
            raise ValueError("meas_set must be 'full' or 'basic'")
        self.names = []
        for k, e in zip(self.kinds, self.elem):
            if k == "V":
                self.names.append(f"V@B{e + 1}")
            elif k in ("Pinj", "Qinj"):
                self.names.append(f"{k[0]}_inj@B{e + 1}")
            else:
                a, b = (self.f[e], self.t[e]) if k.endswith("f") else (self.t[e], self.f[e])
                self.names.append(f"{k[0]}_flow B{a + 1}->B{b + 1}")
        self.iV = np.arange(nb)  # measurement indices of voltage meters
        self.stateV = (nb - 1) + np.arange(nb)  # state indices of voltage magnitudes

        V0 = net.res_bus.vm_pu.values.copy()
        th0 = np.deg2rad(net.res_bus.va_degree.values)
        self.x_base = self.pack(V0, th0)
        self.lam_ref = self.obs_lambda(self.x_base, self.avail)

    # -- state packing ---------------------------------------------------
    def pack(self, V, th):
        return np.concatenate([np.asarray(th)[self.nonref], np.asarray(V)])

    def unpack(self, x):
        th = np.zeros(self.nb)
        th[self.nonref] = x[: self.nb - 1]
        return np.asarray(x[self.nb - 1:]), th

    def flat_start(self):
        return self.pack(np.ones(self.nb), np.zeros(self.nb))

    # -- measurement function and Jacobian ---------------------------------
    def h(self, x):
        V, th = self.unpack(x)
        Vc = V * np.exp(1j * th)
        Sb = Vc * np.conj(self.Ybus @ Vc) * self.base
        Sf = Vc[self.f] * np.conj(self.Yf @ Vc) * self.base
        St = Vc[self.t] * np.conj(self.Yt @ Vc) * self.base
        return np.concatenate([V, Sb.real, Sb.imag, Sf.real, Sf.imag, St.real, St.imag])

    def jac(self, x):
        V, th = self.unpack(x)
        nb = self.nb
        Vc = V * np.exp(1j * th)
        Vn = np.exp(1j * th)
        Ib = self.Ybus @ Vc
        dV = np.diag(Vc)
        dVn = np.diag(Vn)
        dSb_dVm = dV @ np.conj(self.Ybus @ dVn) + np.conj(np.diag(Ib)) @ dVn
        dSb_dVa = 1j * dV @ np.conj(np.diag(Ib) - self.Ybus @ dV)
        If = self.Yf @ Vc
        It = self.Yt @ Vc
        dVf = np.diag(Vc[self.f])
        dVt = np.diag(Vc[self.t])
        dSf_dVa = 1j * (np.conj(np.diag(If)) @ self.Cf @ dV - dVf @ np.conj(self.Yf @ dV))
        dSf_dVm = dVf @ np.conj(self.Yf @ dVn) + np.conj(np.diag(If)) @ self.Cf @ dVn
        dSt_dVa = 1j * (np.conj(np.diag(It)) @ self.Ct @ dV - dVt @ np.conj(self.Yt @ dV))
        dSt_dVm = dVt @ np.conj(self.Yt @ dVn) + np.conj(np.diag(It)) @ self.Ct @ dVn
        nr = self.nonref
        b = self.base

        def blk(dVa, dVm, part):
            return np.hstack([part(dVa)[:, nr], part(dVm)]) * b

        HV = np.hstack([np.zeros((nb, nb - 1)), np.eye(nb)])
        return np.vstack([
            HV,
            blk(dSb_dVa, dSb_dVm, np.real), blk(dSb_dVa, dSb_dVm, np.imag),
            blk(dSf_dVa, dSf_dVm, np.real), blk(dSf_dVa, dSf_dVm, np.imag),
            blk(dSt_dVa, dSt_dVm, np.real), blk(dSt_dVa, dSt_dVm, np.imag),
        ])

    # -- estimators ----------------------------------------------------------
    def wls(self, z, active, sig_eff=None, x0=None, max_iter=30, tol=1e-6) -> Fit:
        sig = self.sigma if sig_eff is None else sig_eff
        idx = np.where(active)[0]
        w = 1.0 / sig[idx] ** 2
        zi = z[idx]
        x = self.flat_start() if x0 is None else np.array(x0, float)

        def cost(xx):
            r = zi - self.h(xx)[idx]
            return float(np.sum(w * r * r))

        c = cost(x)
        for it in range(max_iter):
            r = zi - self.h(x)[idx]
            H = self.jac(x)[idx]
            G = H.T @ (H * w[:, None])
            g = H.T @ (w * r)
            try:
                dx = np.linalg.solve(G, g)
            except np.linalg.LinAlgError:
                return Fit(x, False, it)
            step = 1.0
            xn, cn = x, c
            for _ in range(10):
                xn = x + step * dx
                xn[self.nb - 1:] = np.maximum(xn[self.nb - 1:], 0.3)
                cn = cost(xn)
                if np.isfinite(cn) and cn <= c * (1 + 1e-12):
                    break
                step *= 0.5
            x, c = xn, cn
            if np.max(np.abs(step * dx)) < tol:
                return Fit(x, True, it + 1)
        return Fit(x, False, max_iter)

    def huber(self, z, active, x0=None, k=1.5, iters=8) -> Fit:
        sig = self.sigma
        sig_eff = sig.copy()
        x = x0
        fit = None
        for _ in range(iters):
            fit = self.wls(z, active, sig_eff, x0=x)
            if not fit.converged:
                return fit
            x = fit.x
            r = np.abs(z - self.h(x)) / sig
            wgt = np.minimum(1.0, k / np.maximum(r, 1e-9))
            new = sig / np.sqrt(wgt)
            if np.max(np.abs(new - sig_eff) / sig_eff) < 1e-3:
                break
            sig_eff = new
        return fit

    # -- diagnostics -------------------------------------------------------------
    def analyze(self, z, x, active) -> Analysis:
        H = self.jac(x)
        w = 1.0 / self.sigma ** 2
        Ha = H[active]
        G = Ha.T @ (Ha * w[active][:, None])
        try:
            Ginv = np.linalg.inv(G)
        except np.linalg.LinAlgError:
            Ginv = np.linalg.pinv(G)
        lam = float(np.linalg.eigvalsh(G)[0])
        Q = np.einsum("ij,jk,ik->i", H, Ginv, H)
        res = z - self.h(x)
        s2 = self.sigma ** 2
        var = np.where(active, s2 - Q, s2 + Q)
        crit = active & (var < 0.02 * s2)
        rN = res / np.sqrt(np.maximum(var, 1e-12))
        rN = np.where(crit | ~self.avail, 0.0, rN)
        return Analysis(rN=rN, crit=crit, res=res, Ginv=Ginv, lam_min=lam)

    def obs_lambda(self, x, active) -> float:
        H = self.jac(x)[active]
        G = H.T @ (H / (self.sigma[active] ** 2)[:, None])
        return float(np.linalg.eigvalsh(G)[0])

    def obs_ratio(self, x, active) -> float:
        return max(0.0, self.obs_lambda(x, active)) / self.lam_ref


# =============================================================================
# 2. Scenarios (TRUTH LIVES HERE - never passed to run_pipeline)
# =============================================================================
@dataclass
class Scenario:
    x_true: np.ndarray
    z_true: np.ndarray   # noise-free measurements
    noise: np.ndarray
    load_factor: float

    @property
    def z_clean(self):
        return self.z_true + self.noise


def make_scenarios(model: GridModel, n: int, rng, load_range=(0.8, 1.2), per_load_sd=0.05):
    """n operating points: global load swing +-20%, per-load variation, AC power flow."""
    import pandapower as pp

    net = copy.deepcopy(model.net)
    p0 = net.load.p_mw.values.copy()
    q0 = net.load.q_mvar.values.copy()
    out, tries = [], 0
    while len(out) < n and tries < 6 * n + 20:
        tries += 1
        g = rng.uniform(*load_range)
        fct = g * np.clip(1 + rng.normal(0, per_load_sd, len(p0)), 0.7, 1.3)
        net.load["p_mw"] = p0 * fct
        net.load["q_mvar"] = q0 * fct
        try:
            pp.runpp(net, calculate_voltage_angles=True, init="results", numba=False)
        except Exception:
            try:
                pp.runpp(net, calculate_voltage_angles=True, init="flat", numba=False)
            except Exception:
                continue
        V = net.res_bus.vm_pu.values.copy()
        th = np.deg2rad(net.res_bus.va_degree.values)
        x = model.pack(V, th)
        out.append(Scenario(x, model.h(x), rng.normal(0.0, model.sigma), float(g)))
    return out


# =============================================================================
# 3. Attacks (evaluation / demo only)
# =============================================================================
@dataclass
class AttackInfo:
    mode: str
    bias: np.ndarray
    idx: np.ndarray
    target_bus: int | None = None
    meta: dict = field(default_factory=dict)

    @property
    def attacked_buses(self):
        return self.meta.get("attacked_buses", [])


def _stealth_bias(model, x_true, bus, dV, dth):
    """a = h(x+c) - h(x): consistent with the AC model, invisible to residual tests
    if the attacker controls every meter in the support of `a`."""
    V, th = model.unpack(x_true)
    V, th = V.copy(), th.copy()
    V[bus] += dV
    if bus != model.slack:
        th[bus] += dth
    return model.h(model.pack(V, th)) - model.h(x_true)


def _attacked_buses(model, idx):
    return sorted({int(model.loc[i]) for i in idx})


def _cfg_default():
    return PipelineConfig()


def quick_detect(model, z, cfg=None) -> bool:
    """White-box detection stage used by the adaptive attacker: would the first
    isolation round flag anything?"""
    cfg = cfg or _cfg_default()
    avail = model.avail
    plain = model.wls(z, avail)
    hub = model.huber(z, avail, x0=plain.x if plain.converged else None, k=cfg.huber_k)
    if not hub.converged:
        return True
    ana = model.analyze(z, hub.x, avail)
    s = _robust_scale(ana.rN, avail & ~ana.crit)
    zs = np.abs(ana.rN) / s
    return bool(np.any(avail & ~ana.crit & (zs >= cfg.z_iso)))


SUPPORT_EPS = 0.1  # a meter is "in the support" if the attack must move it by >= 0.1 sigma


def apply_attack(model: GridModel, sc: Scenario, mode: str, rng, cfg=None, bus=None, size=None,
                 protected=None):
    """Returns (z_attacked, AttackInfo). `bus` (0-based) and `size` for custom modes.
    `protected`: meter indices the attacker cannot touch (hardened meters)."""
    protected = np.zeros(model.m, bool) if protected is None else np.asarray(protected, bool)
    z = sc.z_clean.copy()
    avail_idx = np.where(model.avail)[0]
    sig = model.sigma
    bias = np.zeros(model.m)
    meta: dict = {}
    target = None

    if mode == "none":
        pass
    elif mode == "random_single":
        i = int(rng.choice(avail_idx))
        bias[i] = rng.choice([-1, 1]) * rng.uniform(10, 60) * sig[i]
    elif mode == "single_small":
        i = int(rng.choice(avail_idx))
        bias[i] = rng.choice([-1, 1]) * rng.uniform(3, 8) * sig[i]
    elif mode == "multi_coordinated":
        k = int(rng.integers(2, 6))
        center = int(rng.integers(model.nb))
        nbrs = set([center]) | {int(model.t[e]) for e in np.where(model.f == center)[0]} \
            | {int(model.f[e]) for e in np.where(model.t == center)[0]}
        pool = [i for i in avail_idx if int(model.loc[i]) in nbrs]
        pick = rng.choice(pool, size=min(k, len(pool)), replace=False)
        sign_by_kind = {kd: rng.choice([-1, 1]) for kd in KINDS}
        for i in pick:
            bias[int(i)] = sign_by_kind[model.kinds[i]] * rng.uniform(3, 15) * sig[i]
        target = center
    elif mode in ("stealth_partial", "adaptive", "stealth_full", "custom_stealth"):
        b = int(rng.integers(model.nb)) if bus is None else int(bus)
        target = b
        sign = rng.choice([-1, 1])
        dV0 = sign * (rng.uniform(0.03, 0.08) if size is None else abs(size))
        if size is not None and size < 0:
            dV0 = -abs(size)
        dth0 = 0.0
        a_full = _stealth_bias(model, sc.x_true, b, dV0, dth0) * model.avail
        support = np.where(np.abs(a_full) / sig > SUPPORT_EPS)[0]
        order = np.array([i for i in support[np.argsort(-np.abs(a_full[support]) / sig[support])]
                          if not protected[i]], dtype=int)
        meta.update(support_size=int(support.size), stealth_cost=int(support.size),
                    protected_in_support=int(protected[support].sum()))
        if mode in ("stealth_full", "custom_stealth"):
            bias[order] = a_full[order]          # controls everything it is allowed to touch
            meta["controlled"] = int(len(order))
        elif mode == "stealth_partial":
            ctrl = order[: min(K_MAX, order.size)]
            bias[ctrl] = a_full[ctrl]
            meta["controlled"] = int(len(ctrl))
        else:  # adaptive: white-box against the trust rule, budget K_MAX
            ctrl = order[: min(K_MAX, order.size)]
            mask = np.zeros(model.m)
            mask[ctrl] = 1.0
            lo, hi = 0.0, 1.0
            for _ in range(7):
                s = 0.5 * (lo + hi)
                a_s = _stealth_bias(model, sc.x_true, b, s * dV0, 0.0) * mask * model.avail
                if quick_detect(model, z + a_s, cfg):
                    hi = s
                else:
                    lo = s
            bias = _stealth_bias(model, sc.x_true, b, lo * dV0, 0.0) * mask * model.avail
            meta.update(controlled=int(len(ctrl)), adaptive_scale=float(lo), dV=float(lo * dV0))
    elif mode == "custom_voltage":
        b = int(rng.integers(model.nb)) if bus is None else int(bus)
        amount = float(rng.uniform(0.05, 0.15) if size is None else size)
        bias[model.iV[b]] = amount
        target = b
    else:
        raise ValueError(f"unknown attack mode {mode}")

    bias = np.where(protected, 0.0, bias)
    idx = np.where(np.abs(bias) > 0)[0]
    meta["attacked_buses"] = _attacked_buses(model, idx)
    return z + bias, AttackInfo(mode, bias, idx, target, meta)


# =============================================================================
# 4. LIVE PIPELINE (operator-visible inputs only)
# =============================================================================
@dataclass
class PipelineConfig:
    z_iso: float = 4.5         # robust z-score at which a measurement is isolated (trust 50%)
    z_full: float = 2.5        # z-score up to which trust is 100%
    z_zero: float = 6.5        # z-score at which trust is 0%
    z_reinstate: float = 2.5
    max_rounds: int = 16       # one-at-a-time removal (largest normalized residual first)
    iso_per_round: int = 1
    max_iso_frac: float = 0.25
    huber_k: float = 1.5
    use_trust: bool = True     # ablation switch
    use_obs_guard: bool = True  # ablation switch
    obs_ratio_min: float = 0.02
    tol_pu: float = 0.01       # voltage tolerance defining an unsafe recovery
    alpha: float = 0.10        # target risk level of the calibrated bound
    caution_conf: float = 0.50
    v_limits: tuple = (0.95, 1.05)


def decide_level(bound, conf, hard_fail, cfg):
    """Gate rule (shared by the live pipeline and the batch evaluation)."""
    if hard_fail:
        return "red", "VALIDATE - operator review required"
    if bound <= cfg.tol_pu:
        return "green", "PROCEED with recovered state"
    if conf >= 100.0 * cfg.caution_conf:
        return "yellow", "CAUTION - proceed only with extra checks"
    return "red", "VALIDATE - operator review required"


def _robust_scale(rN, mask):
    v = rN[mask]
    if v.size < 5:
        return 1.0
    return max(1.0, 1.4826 * float(np.median(np.abs(v - np.median(v)))))


@dataclass
class PipelineResult:
    converged: bool
    plain_V: np.ndarray
    huber_V: np.ndarray
    rec_V: np.ndarray
    rec_x: np.ndarray
    iso: np.ndarray                 # bool mask over measurements
    blocked: list
    z_score: np.ndarray
    trust: np.ndarray               # per measurement, 0-100
    bus_trust: np.ndarray           # per bus, 0-100 (min over meters located there)
    crit: np.ndarray
    features: dict
    fvec: np.ndarray
    obs_ok: bool
    conf: float | None              # calibrated P(max voltage error <= tol), 0-100
    bound_pu: float | None          # calibrated upper bound on max voltage error
    level: str                      # green / yellow / red / grey
    label: str
    reasons: list
    ms: float
    hard_fail: bool = False
    names: list = field(default_factory=list)

    @property
    def iso_idx(self):
        return np.where(self.iso)[0]

    @property
    def iso_names(self):
        return [self.names[i] for i in self.iso_idx]


def run_pipeline(model: GridModel, z: np.ndarray, cfg: PipelineConfig | None = None,
                 calibration: "GateCalibration | None" = None) -> PipelineResult:
    """LIVE PATH. Inputs: measurements, public model, offline-calibrated gate. Nothing else."""
    cfg = cfg or PipelineConfig()
    t0 = time.perf_counter()
    avail = model.avail
    m_av = int(avail.sum())

    plain = model.wls(z, avail)
    hub = model.huber(z, avail, x0=plain.x if plain.converged else None, k=cfg.huber_k)
    converged = bool(hub.converged)
    iso = np.zeros(model.m, bool)
    z_at_iso = np.zeros(model.m)
    blocked: list = []
    z_first = None
    fit = hub if converged else plain

    if cfg.use_trust and converged:
        max_iso = int(cfg.max_iso_frac * m_av)
        for _ in range(cfg.max_rounds):
            act = avail & ~iso
            ana = model.analyze(z, fit.x, act)
            s = _robust_scale(ana.rN, act & ~ana.crit)
            zs = np.abs(ana.rN) / s
            if z_first is None:
                z_first = float(zs[act & ~ana.crit].max())
            cand = np.where(act & ~ana.crit & (zs >= cfg.z_iso))[0]
            if cand.size == 0:
                break
            changed = False
            took = 0
            for i in cand[np.argsort(-zs[cand])]:
                if iso.sum() >= max_iso or took >= cfg.iso_per_round:
                    break
                trial = iso.copy()
                trial[i] = True
                if cfg.use_obs_guard and model.obs_ratio(fit.x, avail & ~trial) < cfg.obs_ratio_min:
                    if int(i) not in blocked:
                        blocked.append(int(i))
                    continue
                iso = trial
                z_at_iso[i] = zs[i]
                changed = True
                took += 1
            if not changed:
                break
            new = model.wls(z, avail & ~iso, x0=fit.x)
            if not new.converged:
                break
            fit = new
        # reinstate measurements that agree with the cleaned estimate
        if iso.any():
            act = avail & ~iso
            ana = model.analyze(z, fit.x, act)
            s = _robust_scale(ana.rN, act & ~ana.crit)
            back = [i for i in np.where(iso)[0] if abs(ana.rN[i]) / s < cfg.z_reinstate]
            if back:
                iso[back] = False
                new = model.wls(z, avail & ~iso, x0=fit.x)
                if new.converged:
                    fit = new
                else:
                    iso[back] = True

    act = avail & ~iso
    ana = model.analyze(z, fit.x, act)
    s_f = _robust_scale(ana.rN, act & ~ana.crit)
    zs_f = np.abs(ana.rN) / s_f
    z_score = np.where(iso & (z_at_iso > 0), z_at_iso, zs_f)
    trust = 100.0 * np.clip((cfg.z_zero - z_score) / (cfg.z_zero - cfg.z_full), 0.0, 1.0)
    trust = np.where(model.avail, trust, np.nan)
    bus_trust = np.array([np.nanmin(trust[(model.loc == b) & model.avail]) for b in range(model.nb)])

    r = (z - model.h(fit.x))[act] / model.sigma[act]
    df = max(int(act.sum()) - model.n_state, 1)
    J_norm = float(np.sum(r * r) / df)
    Vp, _ = model.unpack(plain.x)
    Vh, _ = model.unpack(hub.x)
    Vr, _ = model.unpack(fit.x)
    shift = float(np.max(np.abs(Vr - Vp))) if plain.converged else 1.0
    disagree = float(np.max(np.abs(Vr - Vh))) if converged else 1.0
    sdV = float(np.sqrt(np.max(np.diag(ana.Ginv)[model.stateV])))
    ratio = max(ana.lam_min, 0.0) / model.lam_ref
    obs_ok = ratio >= cfg.obs_ratio_min
    if plain.converged:
        r_raw = (z - model.h(plain.x))[avail] / model.sigma[avail]
        J_raw = float(np.sum(r_raw * r_raw) / max(m_av - model.n_state, 1))
    else:
        J_raw = 1e3
    if z_first is None:
        z_first = float(np.max(zs_f[act & ~ana.crit])) if np.any(act & ~ana.crit) else 0.0
    feats = {
        "log_J": math.log10(J_norm + 1e-3),
        "log_zmax": math.log10(float(np.max(zs_f[act & ~ana.crit])) + 0.1) if np.any(act & ~ana.crit) else 0.0,
        "frac_iso": float(iso.sum() / m_av),
        "log_shift": math.log10(shift + 1e-5),
        "log_disagree": math.log10(disagree + 1e-5),
        "log_sdV": math.log10(sdV + 1e-6),
        "n_crit": float(ana.crit.sum()),
        "obs_blocked": 1.0 if blocked else 0.0,
        "log_obs": math.log10(ratio + 1e-6),
        "log_J_raw": math.log10(J_raw + 1e-3),
        "log_zfirst": math.log10(z_first + 0.1),
    }
    fvec = np.array([feats[n] for n in FEATURE_NAMES])

    conf = bound = None
    level, label, reasons = "grey", "NOT CALIBRATED", []
    hard_fail = (not converged) or (not fit.converged) or (not obs_ok) or bool(blocked)
    if calibration is not None:
        conf = 100.0 * calibration.prob_within(fvec, cfg.tol_pu)
        bound = calibration.bound(fvec, cfg.alpha)
        level, label = decide_level(bound, conf, hard_fail, cfg)
    if not converged:
        reasons.append("State estimation did not converge: input data is inconsistent.")
    if blocked:
        reasons.append(f"{len(blocked)} suspicious measurement(s) could not be removed without losing observability.")
    if not obs_ok:
        reasons.append("Remaining measurements do not make the grid observable.")
    if iso.any():
        reasons.append("Isolated: " + ", ".join(model.names[i] for i in np.where(iso)[0][:6])
                       + (" ..." if iso.sum() > 6 else ""))
    else:
        reasons.append("No measurement isolated (all residuals within the noise level).")
    reasons.append(f"Recovery changed the voltage estimate by up to {shift:.4f} pu versus the raw estimate.")
    if bound is not None:
        reasons.append(f"Calibrated risk bound on the voltage error: {bound:.4f} pu "
                       f"(tolerance {cfg.tol_pu} pu, target risk {cfg.alpha:.0%}).")

    return PipelineResult(
        converged=converged, plain_V=Vp, huber_V=Vh, rec_V=Vr, rec_x=fit.x, iso=iso, blocked=blocked,
        z_score=z_score, trust=trust, bus_trust=bus_trust, crit=ana.crit, features=feats, fvec=fvec,
        obs_ok=bool(obs_ok), conf=conf, bound_pu=bound, level=level, label=label, reasons=reasons,
        ms=(time.perf_counter() - t0) * 1000.0, hard_fail=bool(hard_fail), names=model.names,
    )


# =============================================================================
# 5. Calibrated gate (boosted trees + split-conformal, Mondrian bins on the prediction)
# =============================================================================
class BoostedTrees:
    """Tiny gradient-boosted regression trees (numpy only, JSON-serialisable)."""

    def __init__(self, n_rounds=120, lr=0.08, depth=3, min_leaf=10, n_thr=20, subsample=0.8, seed=0):
        self.n_rounds, self.lr, self.depth = n_rounds, lr, depth
        self.min_leaf, self.n_thr, self.subsample, self.seed = min_leaf, n_thr, subsample, seed
        self.init = 0.0
        self.trees: list = []

    def _build(self, X, r, depth):
        n = len(r)
        if depth == 0 or n < 2 * self.min_leaf:
            return {"v": float(r.mean()) if n else 0.0}
        best = None
        tot, tot_n = r.sum(), n
        for j in range(X.shape[1]):
            col = X[:, j]
            thr = np.unique(np.quantile(col, np.linspace(0, 1, self.n_thr + 2)[1:-1]))
            for t in thr:
                left = col <= t
                nl = int(left.sum())
                if nl < self.min_leaf or n - nl < self.min_leaf:
                    continue
                sl = r[left].sum()
                gain = sl * sl / nl + (tot - sl) ** 2 / (tot_n - nl)
                if best is None or gain > best[0]:
                    best = (gain, j, float(t))
        if best is None:
            return {"v": float(r.mean())}
        _, j, t = best
        left = X[:, j] <= t
        return {"f": j, "t": t, "l": self._build(X[left], r[left], depth - 1),
                "r": self._build(X[~left], r[~left], depth - 1)}

    @staticmethod
    def _walk(node, x):
        while "v" not in node:
            node = node["l"] if x[node["f"]] <= node["t"] else node["r"]
        return node["v"]

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        rng = np.random.default_rng(self.seed)
        self.init = float(y.mean())
        pred = np.full(len(y), self.init)
        self.trees = []
        for _ in range(self.n_rounds):
            idx = rng.choice(len(y), max(int(self.subsample * len(y)), 2 * self.min_leaf), replace=False)
            tree = self._build(X[idx], (y - pred)[idx], self.depth)
            self.trees.append(tree)
            pred += self.lr * np.array([self._walk(tree, x) for x in X])
        return self

    def predict_one(self, x):
        x = np.asarray(x, float)
        return self.init + self.lr * sum(self._walk(t, x) for t in self.trees)

    def to_dict(self):
        return {"init": self.init, "lr": self.lr, "trees": self.trees}

    @classmethod
    def from_dict(cls, d):
        m = cls(lr=d["lr"])
        m.init, m.trees = d["init"], d["trees"]
        return m


class GateCalibration:
    """Offline: boosted trees predict log10(max voltage error) from RUNTIME features;
    split-conformal residual quantiles (per bin of the prediction) turn the prediction
    into a bound U with P(error <= U) >= 1 - alpha under the calibration distribution.
    Online: only the feature vector is needed - no truth."""

    def __init__(self, model, edges, bin_res, tol, alpha, meta=None):
        self.model = model
        self.edges = np.asarray(edges, float)
        self.bin_res = [np.sort(np.asarray(r, float)) for r in bin_res]
        self.tol = float(tol)
        self.alpha = float(alpha)
        self.meta = meta or {}

    @classmethod
    def fit(cls, X_train, err_train, X_cal, err_cal, tol=0.01, alpha=0.10, n_bins=3, seed=0):
        X_train, X_cal = np.asarray(X_train, float), np.asarray(X_cal, float)
        y_tr = np.log10(np.maximum(np.asarray(err_train, float), 1e-6))
        y_ca = np.log10(np.maximum(np.asarray(err_cal, float), 1e-6))
        bt = BoostedTrees(seed=seed).fit(X_train, y_tr)
        yhat_cal = np.array([bt.predict_one(x) for x in X_cal])
        res = y_ca - yhat_cal
        n_bins = max(1, min(n_bins, len(res) // 80))
        edges = np.quantile(yhat_cal, np.linspace(0, 1, n_bins + 1)[1:-1]) if n_bins > 1 else np.array([])
        bidx = np.searchsorted(edges, yhat_cal)
        bin_res = [res[bidx == b] for b in range(n_bins)]
        meta = {"n_train": int(len(y_tr)), "n_cal": int(len(y_ca)), "n_bins": int(n_bins)}
        return cls(bt, edges, bin_res, tol, alpha, meta)

    def predict_log(self, fvec):
        return float(self.model.predict_one(fvec))

    def _bin(self, yhat):
        return int(np.searchsorted(self.edges, yhat))

    def prob_within(self, fvec, tol=None):
        tol = self.tol if tol is None else tol
        yhat = self.predict_log(fvec)
        R = self.bin_res[self._bin(yhat)]
        return float(np.sum(R <= math.log10(tol) - yhat) / (len(R) + 1))

    def bound(self, fvec, alpha=None):
        alpha = self.alpha if alpha is None else alpha
        yhat = self.predict_log(fvec)
        R = self.bin_res[self._bin(yhat)]
        k = math.ceil((len(R) + 1) * (1 - alpha))
        if k > len(R):
            return float("inf")
        return float(10 ** (yhat + R[k - 1]))

    def to_json(self):
        return json.dumps({
            "model": self.model.to_dict(), "edges": self.edges.tolist(),
            "bin_res": [r.tolist() for r in self.bin_res], "tol": self.tol, "alpha": self.alpha,
            "meta": self.meta, "features": list(FEATURE_NAMES),
        })

    @classmethod
    def from_json(cls, s):
        d = json.loads(s)
        if d.get("features") != list(FEATURE_NAMES):
            raise ValueError("Calibration file does not match the current feature set; re-run batch_eval.py")
        return cls(BoostedTrees.from_dict(d["model"]), d["edges"], d["bin_res"], d["tol"], d["alpha"], d.get("meta"))


# =============================================================================
# 6. Evaluation helpers (use the truth: demo / batch only)
# =============================================================================
def true_error(model, x_true, V_est) -> float:
    Vt, _ = model.unpack(x_true)
    return float(np.max(np.abs(np.asarray(V_est) - Vt)))


def limit_violations(V, limits=(0.95, 1.05)):
    V = np.asarray(V)
    return sorted(int(b) for b in np.where((V < limits[0]) | (V > limits[1]))[0])


def recovery_table(model, res: PipelineResult, z, x_true, info: AttackInfo | None, cfg=None):
    """Recovery Accuracy Evaluation (EVALUATION ONLY - never used by the live path).
    Columns: true voltage, attacked reading, naive WLS estimate, reconstructed voltage and errors."""
    cfg = cfg or PipelineConfig()
    Vt, _ = model.unpack(x_true)
    attacked = set(info.attacked_buses) if info is not None else set()
    iso_buses = {int(model.loc[i]) for i in res.iso_idx}
    rows = []
    for b in range(model.nb):
        role = "ATTACKED+ISOLATED" if (b in attacked and b in iso_buses) else \
            "ATTACKED" if b in attacked else "ISOLATED (false alarm)" if b in iso_buses else "-"
        rows.append({
            "Bus": f"B{b + 1}",
            "True voltage (pu)": round(float(Vt[b]), 4),
            "Attacked reading (pu)": round(float(z[model.iV[b]]), 4),
            "Naive estimate (pu)": round(float(res.plain_V[b]), 4),
            "Reconstructed (pu)": round(float(res.rec_V[b]), 4),
            "Reconstruction error (pu)": round(float(abs(res.rec_V[b] - Vt[b])), 4),
            "Naive error (pu)": round(float(abs(res.plain_V[b] - Vt[b])), 4),
            "Role": role,
        })
    err = np.abs(res.rec_V - Vt)
    naive = np.abs(res.plain_V - Vt)
    mx = float(err.max())
    if mx <= cfg.tol_pu:
        note = f"Recovery is close to the reference (max error {mx:.4f} pu <= {cfg.tol_pu} pu)."
    elif mx <= 3 * cfg.tol_pu:
        note = f"Recovery is acceptable but not tight (max error {mx:.4f} pu)."
    else:
        note = f"Recovery deviates strongly from the reference (max error {mx:.4f} pu): do not rely on it."
    return rows, {"max_error": mx, "mean_error": float(err.mean()), "naive_max_error": float(naive.max()),
                  "naive_mean_error": float(naive.mean()), "note": note}


def consequence_view(model, res: PipelineResult, x_true, cfg=None):
    """What the operator would conclude from voltage-limit checks on each state."""
    cfg = cfg or PipelineConfig()
    Vt, _ = model.unpack(x_true)
    t = limit_violations(Vt, cfg.v_limits)
    n = limit_violations(res.plain_V, cfg.v_limits)
    r = limit_violations(res.rec_V, cfg.v_limits)
    lab = lambda s: ", ".join(f"B{b + 1}" for b in s) or "none"
    return {
        "true": lab(t), "naive": lab(n), "recovered": lab(r),
        "naive_wrong_action": n != t, "recovered_wrong_action": r != t,
    }


def run_gridsentinel(seed=None, attack_mode="random_single", model=None, calibration=None,
                     cfg=None, bus=None, size=None):
    """Convenience API for demos and notebooks. Returns operator-view and evaluation-view keys."""
    model = model or GridModel()
    cfg = cfg or PipelineConfig()
    rng = np.random.default_rng(seed)
    sc = make_scenarios(model, 1, rng)[0]
    z, info = apply_attack(model, sc, attack_mode, rng, cfg, bus=bus, size=size)
    if calibration is None:
        calibration = get_default_calibration(model, cfg=cfg)
    res = run_pipeline(model, z, cfg, calibration)        # <- live path: z only
    rows, summary = recovery_table(model, res, z, sc.x_true, info, cfg)
    return {
        # operator view
        "level": res.level, "label": res.label, "conf": res.conf, "bound_pu": res.bound_pu,
        "isolated": res.iso_names, "success": res.converged, "residual_norm": res.features["log_J"],
        "reasons": res.reasons, "bus_trust": res.bus_trust,
        # evaluation view (demo only)
        "attack_mode": attack_mode, "attack_buses": [b + 1 for b in info.attacked_buses],
        "recovery_rows": rows, "recovery": summary, "consequence": consequence_view(model, res, sc.x_true, cfg),
        "true_error": true_error(model, sc.x_true, res.rec_V),
        "result": res, "scenario": sc, "attack": info, "z": z,
    }


# =============================================================================
# 7. Calibration builder + hardening ranking
# =============================================================================
def build_calibration(model, n_train=300, n_cal=300, seed=7, cfg=None, mix=None, progress=None):
    """Generate attacked scenarios, run the LIVE pipeline, and fit the gate offline."""
    cfg = cfg or PipelineConfig()
    mix = mix or IN_MODEL_MIX
    rng = np.random.default_rng(seed)
    modes = list(mix.keys())
    probs = np.array([mix[k] for k in modes], float)
    probs /= probs.sum()
    feats, errs = [], []
    for k, n in enumerate((n_train, n_cal)):
        F, E = [], []
        for sc in make_scenarios(model, n, rng):
            mode = str(rng.choice(modes, p=probs))
            z, _ = apply_attack(model, sc, mode, rng, cfg)
            res = run_pipeline(model, z, cfg)
            F.append(res.fvec)
            E.append(true_error(model, sc.x_true, res.rec_V))
            if progress:
                progress(len(F) + k * n_train, n_train + n_cal)
        feats.append(np.array(F))
        errs.append(np.array(E))
    return GateCalibration.fit(feats[0], errs[0], feats[1], errs[1], cfg.tol_pu, cfg.alpha)


def get_default_calibration(model, path="gate_calibration.json", cfg=None, quick_n=240, seed=7):
    import os
    if os.path.exists(path):
        try:
            return GateCalibration.from_json(open(path).read())
        except Exception:
            pass
    cal = build_calibration(model, quick_n, quick_n, seed, cfg)
    try:
        open(path, "w").write(cal.to_json())
    except OSError:
        pass
    return cal


def stealth_costs(model: GridModel, x=None):
    """For each bus: meters an attacker must control to shift that bus's voltage
    stealthily (single-bus, voltage-targeting). Lower = easier to attack."""
    x = model.x_base if x is None else x
    out = []
    for b in range(model.nb):
        a = _stealth_bias(model, x, b, 0.03, 0.0) * model.avail
        out.append(np.where(np.abs(a) / model.sigma > SUPPORT_EPS)[0])
    return out


def hardening_plan(model: GridModel):
    """Greedy hitting set: fewest meters whose protection blocks every single-bus
    voltage-targeting stealth attack (the attacker must control ALL meters in the support)."""
    sets = [set(map(int, s)) for s in stealth_costs(model)]
    remaining = list(range(len(sets)))
    chosen = []
    while remaining:
        counts = {}
        for b in remaining:
            for i in sets[b]:
                counts[i] = counts.get(i, 0) + 1
        best = max(counts, key=lambda i: (counts[i], -i))
        chosen.append((best, sorted(b for b in remaining if best in sets[b])))
        remaining = [b for b in remaining if best not in sets[b]]
    return [(int(i), model.names[i], [f"B{b + 1}" for b in covered]) for i, covered in chosen]


In [ ]:
%%writefile gridsentinel_app.py
"""GridSentinel AI - Streamlit demo.  Run:  streamlit run gridsentinel_app.py"""
import os
import numpy as np
import pandas as pd
import streamlit as st

from gridsentinel_core import (GridModel, PipelineConfig, GateCalibration, make_scenarios, apply_attack,
                               run_pipeline, recovery_table, consequence_view, hardening_plan,
                               get_default_calibration, true_error, K_MAX)

st.set_page_config(page_title="GridSentinel AI", page_icon="⚡", layout="wide")
CAL_PATH = os.environ.get("GS_CAL", "gate_calibration.json")


@st.cache_resource(show_spinner="Loading grid model and calibrated gate (first run builds it, ~1 min)...")
def load():
    model, cfg = GridModel(), PipelineConfig()
    return model, cfg, get_default_calibration(model, path=CAL_PATH, cfg=cfg)


model, cfg, cal = load()
COLORS = {"green": "#1b8a3a", "yellow": "#c98a00", "red": "#c0392b", "grey": "#666"}

st.title("⚡ GridSentinel AI")
st.caption("Self-protecting state estimation for smart grids: detect, isolate, recover - and say when NOT to trust the result. "
           "IEEE 14-bus simulation, 122 AC measurements.")

with st.sidebar:
    st.header("Operator panel")
    seed = st.number_input("Seed", min_value=0, max_value=10**9, value=42, step=1)
    fixed = st.checkbox("Fixed seed", value=True, help="Unchecked = a new random scenario on every run")
    scenario = st.selectbox("Scenario", ["none", "random single", "multi", "stealth (partial)",
                                         "adaptive attacker", "custom voltage injection", "custom stealth"])
    bus = size = None
    if scenario.startswith("custom"):
        bus = st.number_input("Target bus", 1, model.nb, 9) - 1
        size = st.number_input("Size (pu)", -0.2, 0.2, 0.08, 0.01)
    run = st.button("Run scenario", type="primary")

MODE = {"none": "none", "random single": "random_single", "multi": "multi_coordinated",
        "stealth (partial)": "stealth_partial", "adaptive attacker": "adaptive",
        "custom voltage injection": "custom_voltage", "custom stealth": "custom_stealth"}[scenario]

if "case" not in st.session_state or run:
    s = int(seed) if fixed else int(np.random.default_rng().integers(10**9))
    rng = np.random.default_rng(s)
    sc = make_scenarios(model, 1, rng)[0]
    z, info = apply_attack(model, sc, MODE, rng, cfg, bus=bus, size=size)
    res = run_pipeline(model, z, cfg, cal)          # LIVE path: z only
    st.session_state.case = dict(seed=s, mode=MODE, sc=sc, z=z, info=info, res=res)
c = st.session_state.case
res, sc, z, info = c["res"], c["sc"], c["z"], c["info"]

# --- Attack status (separate from the Trust Gate) -------------------------------------------
from scipy.stats import chi2 as _chi2
from gridsentinel_core import FEATURE_NAMES
_dof = int(model.avail.sum() - len(model.flat_start()))
_j_raw = 10 ** res.features["log_J_raw"] if "log_J_raw" in res.features else 10 ** res.fvec[FEATURE_NAMES.index("log_J_raw")]
_j_thr = _chi2.ppf(0.99, _dof) / _dof
if len(res.iso_idx) or res.blocked:
    st_txt, st_col = f"ATTACK / BAD DATA DETECTED - {len(res.iso_idx)} meter(s) isolated", "#c0392b"
elif not res.converged or _j_raw > _j_thr:
    st_txt, st_col = "SUSPICIOUS DATA - measurements disagree, no single meter to blame", "#c98a00"
else:
    st_txt, st_col = "NO ATTACK INDICATORS - all measurements are consistent", "#1b8a3a"
st.markdown(f"<div style='background:{st_col};color:white;padding:10px 18px;border-radius:10px;font-size:1.15rem;margin-bottom:8px'>"
            f"<b>Attack status:</b> {st_txt}</div>", unsafe_allow_html=True)

col = COLORS[res.level]
st.markdown(f"<div style='background:{col};color:white;padding:14px 18px;border-radius:10px;font-size:1.3rem'>"
            f"<b>{res.label}</b> &nbsp;|&nbsp; P(error ≤ {cfg.tol_pu} pu) = {res.conf:.0f}% &nbsp;|&nbsp; "
            f"calibrated error bound = {res.bound_pu:.4f} pu</div>", unsafe_allow_html=True)
if st_col == "#1b8a3a" and res.level != "green":
    st.info("Why not PROCEED? No alarm was raised, but a perfectly stealthy attack would look exactly like this. "
            "The Trust Gate only says PROCEED when it can bound the error from calibrated evidence - that is the point: "
            "it refuses to be overconfident.")
m1, m2, m3, m4 = st.columns(4)
m1.metric("Estimator converged", "yes" if res.converged else "NO")
m2.metric("Meters isolated", len(res.iso_idx))
m3.metric("Residual (chi²/dof, log10)", f"{res.features['log_J']:.2f}")
m4.metric("Runtime", f"{res.ms:.0f} ms")
st.caption(f"Scenario: {c['mode']} | seed {c['seed']}  (the operator view above uses measurements only)")

t1, t2, t3, t4 = st.tabs(["Sensor trust", "Recovery Accuracy Evaluation", "Consequence", "Hardening"])

with t1:
    st.subheader("Trust per sensor station (S01-S14)")
    tb = pd.DataFrame({"Sensor": [f"S{b + 1:02d}" for b in range(model.nb)],
                       "Bus": [f"B{b + 1}" for b in range(model.nb)],
                       "Trust %": np.round(res.bus_trust, 0)})
    tb["Status"] = np.where(tb["Trust %"] >= 75, "trusted", np.where(tb["Trust %"] >= 40, "suspect", "isolated"))
    st.dataframe(tb, hide_index=True, use_container_width=True)
    if len(res.iso_idx):
        st.write("Isolated measurements:", ", ".join(res.iso_names))
    with st.expander("Measurement-level detail"):
        d = pd.DataFrame({"Measurement": res.names, "Trust %": np.round(res.trust, 0), "z-score": np.round(res.z_score, 2),
                          "Isolated": res.iso})
        st.dataframe(d.sort_values("Trust %").head(40), hide_index=True, use_container_width=True)

with t2:
    st.subheader("Recovery Accuracy Evaluation")
    st.warning("EVALUATION ONLY. The true voltage is stored only to score the result. "
               "It is never used by the system in any live decision.")
    rows, summ = recovery_table(model, res, z, sc.x_true, info, cfg)
    df = pd.DataFrame(rows)
    st.dataframe(df, hide_index=True, use_container_width=True)
    a, b, cc = st.columns(3)
    a.metric("Max reconstruction error", f"{summ['max_error']:.4f} pu", f"naive {summ['naive_max_error']:.4f}", delta_color="off")
    b.metric("Mean reconstruction error", f"{summ['mean_error']:.4f} pu")
    cc.metric("Improvement vs naive", f"{summ['naive_max_error'] / max(summ['max_error'], 1e-9):.1f}x")
    (st.success if summ["max_error"] <= cfg.tol_pu else st.info if summ["max_error"] <= 3 * cfg.tol_pu else st.error)(summ["note"])
    st.line_chart(df.set_index("Bus")[["True voltage (pu)", "Attacked reading (pu)", "Reconstructed (pu)"]])

with t3:
    cv = consequence_view(model, res, sc.x_true, cfg)
    st.subheader("What the operator would conclude (voltage limits 0.95-1.05 pu)")
    st.table(pd.DataFrame({"State": ["True grid", "Naive estimator", "GridSentinel recovered"],
                           "Buses outside limits": [cv["true"], cv["naive"], cv["recovered"]],
                           "Wrong action?": ["-", "YES" if cv["naive_wrong_action"] else "no",
                                             "YES" if cv["recovered_wrong_action"] else "no"]}))

with t4:
    st.subheader("Hardening ranking")
    st.write(f"Attacker budget modelled: up to {K_MAX} compromised meters. Protecting these meters makes every single-bus "
             "stealth attack require control of a protected meter:")
    st.dataframe(pd.DataFrame([{"Meter to protect": n, "Covers buses": ", ".join(cv_)} for _, n, cv_ in hardening_plan(model)]),
                 hide_index=True, use_container_width=True)
    st.caption("Limits: perfect stealth beyond the budget cannot be detected by any estimator-side method.")


In [ ]:
%%writefile batch_eval.py
#!/usr/bin/env python
"""GridSentinel AI - batch evaluation (reproducible numbers for the booth / Q&A).

Usage:
    python batch_eval.py --n 300 --seed 7 --out results          # full run
    python batch_eval.py --quick --out results_quick             # smoke test (~ few minutes)
    python batch_eval.py --n 300 --no-ablation

Everything here is SIMULATION on the IEEE 14-bus system with synthetic attacks.
Truth (x_true) is used ONLY to score results - never inside run_pipeline().
Outputs (in --out): results.csv, results_summary.md, summary.json, gate_calibration.json,
                    plots: calibration.png, risk_coverage.png, error_before_after.png, ablation.png
"""
import argparse
import json
import os
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")

from gridsentinel_core import (  # noqa: E402
    IN_MODEL_MIX, FEATURE_NAMES, GateCalibration, GridModel, PipelineConfig,
    apply_attack, decide_level, hardening_plan, limit_violations, make_scenarios,
    run_pipeline, true_error,
)

TYPES = list(IN_MODEL_MIX)


# --------------------------------------------------------------------------- data generation
def gen_cases(model, cfg, n, rng, mix=None, protected=None, keep_z=False, fixed_mode=None):
    """Attack -> LIVE pipeline (z only) -> record truth-based scores separately."""
    mix = mix or IN_MODEL_MIX
    modes = list(mix)
    p = np.array([mix[k] for k in modes], float)
    p /= p.sum()
    out = []
    for sc in make_scenarios(model, n, rng):
        mode = fixed_mode or str(rng.choice(modes, p=p))
        z, info = apply_attack(model, sc, mode, rng, cfg, protected=protected)
        res = run_pipeline(model, z, cfg)
        Vt, _ = model.unpack(sc.x_true)
        att = set(int(i) for i in info.idx)
        iso = set(int(i) for i in res.iso_idx)
        out.append({
            "mode": mode, "z": z if keep_z else None, "res": res,
            "err_rec": true_error(model, sc.x_true, res.rec_V),
            "err_plain": true_error(model, sc.x_true, res.plain_V),
            "err_huber": true_error(model, sc.x_true, res.huber_V),
            "n_att": len(att), "hits": len(att & iso), "n_iso": len(iso), "false_iso": len(iso - att),
            "wrong_rec": limit_violations(res.rec_V, cfg.v_limits) != limit_violations(Vt, cfg.v_limits),
            "wrong_naive": limit_violations(res.plain_V, cfg.v_limits) != limit_violations(Vt, cfg.v_limits),
            "z": z, "Vt": Vt, "ms": res.ms,
        })
    return out


def feats(cases):
    return np.array([c["res"].fvec for c in cases])


def errs(cases):
    return np.array([c["err_rec"] for c in cases])


# --------------------------------------------------------------------------- baselines
def chi2_stats(model, z, res):
    """Plain-WLS chi-square and largest normalized residual (the textbook detectors)."""
    from scipy.stats import chi2
    fit = model.wls(z, model.avail)
    ana = model.analyze(z, fit.x, model.avail)
    r = (z - model.h(fit.x)) / model.sigma
    J = float(np.sum(r[model.avail] ** 2))
    dof = int(model.avail.sum() - len(fit.x))
    return J > chi2.ppf(0.99, dof), float(np.max(np.abs(ana.rN[~ana.crit]))) if (~ana.crit).any() else 0.0


def detection_table(model, cases, iforest):
    rows = {}
    for name in ("GridSentinel (flags/isolates)", "WLS + chi-square", "WLS + LNR (>3)", "IsolationForest (raw z)"):
        rows[name] = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
    for c in cases:
        attacked = c["mode"] != "none"
        res = c["res"]
        chi_flag, lnr = chi2_stats(model, c["z"], res)
        flags = {
            "GridSentinel (flags/isolates)": res.hard_fail or c["n_iso"] > 0,
            "WLS + chi-square": bool(chi_flag),
            "WLS + LNR (>3)": lnr > 3.0,
            "IsolationForest (raw z)": bool(iforest.predict(c["z"][None, :])[0] == -1) if iforest is not None else False,
        }
        for k, f in flags.items():
            key = ("tp" if f else "fn") if attacked else ("fp" if f else "tn")
            rows[k][key] += 1
    out = {}
    for k, d in rows.items():
        prec = d["tp"] / max(d["tp"] + d["fp"], 1)
        rec = d["tp"] / max(d["tp"] + d["fn"], 1)
        out[k] = {"precision": prec, "recall": rec, "f1": 2 * prec * rec / max(prec + rec, 1e-9),
                  "fpr": d["fp"] / max(d["fp"] + d["tn"], 1)}
    return out


# --------------------------------------------------------------------------- gate metrics
def gate_metrics(cal, cases, cfg):
    F = feats(cases)
    E = errs(cases)
    U = np.array([cal.bound(f, cfg.alpha) for f in F])
    conf = np.array([100 * cal.prob_within(f, cfg.tol_pu) for f in F])
    levels = np.array([decide_level(u, cf, c["res"].hard_fail, cfg)[0] for u, cf, c in zip(U, conf, cases)])
    proceed = levels == "green"
    unsafe = E > cfg.tol_pu
    wrong = np.array([c["wrong_rec"] for c in cases])
    cov_bound = float(np.mean(E <= U))
    return {
        "n": len(cases),
        "proceed_rate": float(proceed.mean()),
        "caution_rate": float((levels == "yellow").mean()),
        "validate_rate": float((levels == "red").mean()),
        "unsafe_given_proceed": float(unsafe[proceed].mean()) if proceed.any() else float("nan"),
        "unsafe_proceed_joint": float(np.mean(proceed & unsafe)),
        "wrong_action_given_proceed": float(wrong[proceed].mean()) if proceed.any() else float("nan"),
        "bound_coverage": cov_bound,
        "unsafe_rate_all": float(unsafe.mean()),
        "per_type": {t: {"n": int(sum(c["mode"] == t for c in cases)),
                         "proceed": float(np.mean([p for p, c in zip(proceed, cases) if c["mode"] == t] or [np.nan])),
                         "unsafe_proceed": float(np.mean([(p and u) for p, u, c in zip(proceed, unsafe, cases) if c["mode"] == t] or [np.nan]))}
                     for t in sorted(set(c["mode"] for c in cases))},
        "_U": U, "_E": E, "_proceed": proceed, "_unsafe": unsafe,
    }


def risk_coverage(score, unsafe):
    """Accept cases from lowest score (most trusted) upward. Returns coverage, selective risk, AURC."""
    order = np.argsort(score)
    u = unsafe[order].astype(float)
    k = np.arange(1, len(u) + 1)
    risk = np.cumsum(u) / k
    cov = k / len(u)
    return cov, risk, float(np.mean(risk))


# --------------------------------------------------------------------------- plots
def make_plots(out, cal_stats, rc, before_after, ablation):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    E, U = cal_stats["_E"], cal_stats["_U"]
    fin = np.isfinite(U)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.loglog(np.maximum(U[fin], 1e-5), np.maximum(E[fin], 1e-5), ".", alpha=0.4, ms=4)
    lim = [1e-5, 1]
    ax.plot(lim, lim, "k--", lw=1, label="error = bound")
    ax.axhline(0.01, color="r", lw=0.8, ls=":")
    ax.set_xlabel("Calibrated bound U (pu)")
    ax.set_ylabel("True max voltage error (pu)")
    ax.set_title(f"Calibration: {100 * cal_stats['bound_coverage']:.1f}% of cases under the bound (target >= 90%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out, "calibration.png"), dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, (cov, risk, aurc) in rc.items():
        ax.plot(cov, risk, label=f"{name} (AURC {aurc:.3f})")
    ax.set_xlabel("Coverage (fraction of cases auto-accepted)")
    ax.set_ylabel("Unsafe rate among accepted")
    ax.set_title("Risk-coverage: lower is better")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "risk_coverage.png"), dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ts = list(before_after)
    x = np.arange(len(ts))
    ax.bar(x - 0.2, [before_after[t][0] for t in ts], 0.4, label="Naive WLS")
    ax.bar(x + 0.2, [before_after[t][1] for t in ts], 0.4, label="GridSentinel recovered")
    ax.axhline(0.01, color="r", ls=":", lw=0.8)
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(ts, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("Median max voltage error (pu)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out, "error_before_after.png"), dpi=140)
    plt.close(fig)

    if ablation:
        fig, ax = plt.subplots(figsize=(7, 4))
        names = list(ablation)
        ax.bar(range(len(names)), [ablation[k]["p90"] for k in names])
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=25, ha="right", fontsize=8)
        ax.set_yscale("log")
        ax.set_ylabel("90th percentile max voltage error (pu)")
        ax.set_title("Ablation: what each component buys")
        fig.tight_layout()
        fig.savefig(os.path.join(out, "ablation.png"), dpi=140)
        plt.close(fig)


# --------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=300, help="scenarios per split (train / calibration / test)")
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--out", default="results")
    ap.add_argument("--quick", action="store_true", help="small run for a smoke test")
    ap.add_argument("--no-ablation", action="store_true")
    a = ap.parse_args()
    n = 80 if a.quick else a.n
    os.makedirs(a.out, exist_ok=True)
    t0 = time.time()
    cfg = PipelineConfig()
    model = GridModel()
    rng = np.random.default_rng(a.seed)
    log = lambda s: print(f"[{time.time() - t0:6.0f}s] {s}", flush=True)

    # ---- deployed grid
    log(f"generating train/cal/test ({n} each) for the deployed grid")
    tr = gen_cases(model, cfg, n, rng)
    ca = gen_cases(model, cfg, n, rng)
    te = gen_cases(model, cfg, n, rng)
    cal = GateCalibration.fit(feats(tr), errs(tr), feats(ca), errs(ca), cfg.tol_pu, cfg.alpha)
    with open(os.path.join(a.out, "gate_calibration.json"), "w") as f:
        f.write(cal.to_json())
    gm = gate_metrics(cal, te, cfg)
    log(f"deployed: proceed {gm['proceed_rate']:.2f}, unsafe|proceed {gm['unsafe_given_proceed']:.3f}, bound cov {gm['bound_coverage']:.3f}")

    # ---- hardened grid (5 protected meters)
    plan = hardening_plan(model)
    mask = np.zeros(model.m, bool)
    mask[[i for i, _, _ in plan]] = True
    log(f"hardened grid: protecting {len(plan)} meters")
    htr = gen_cases(model, cfg, n, rng, protected=mask)
    hca = gen_cases(model, cfg, n, rng, protected=mask)
    hte = gen_cases(model, cfg, n, rng, protected=mask)
    hcal = GateCalibration.fit(feats(htr), errs(htr), feats(hca), errs(hca), cfg.tol_pu, cfg.alpha)
    hm = gate_metrics(hcal, hte, cfg)
    log(f"hardened: proceed {hm['proceed_rate']:.2f}, unsafe|proceed {hm['unsafe_given_proceed']:.3f}")

    # ---- out-of-model stealth_full (limit of the method)
    sf = gen_cases(model, cfg, max(n // 3, 30), rng, fixed_mode="stealth_full")
    sfm = gate_metrics(cal, sf, cfg)
    sf_h = gen_cases(model, cfg, max(n // 3, 30), rng, fixed_mode="stealth_full", protected=mask)
    sfm_h = gate_metrics(hcal, sf_h, cfg)

    # ---- baselines: detection
    iforest = None
    try:
        from sklearn.ensemble import IsolationForest
        clean = [sc.z_clean for sc in make_scenarios(model, n, rng)]
        iforest = IsolationForest(n_estimators=200, contamination=0.01, random_state=a.seed).fit(np.array(clean))
    except Exception as e:  # sklearn missing
        log(f"IsolationForest skipped ({e})")
    det = detection_table(model, te, iforest)

    # ---- recovery error: before vs after, per attack type
    before_after, rows = {}, []
    for t in TYPES:
        cs = [c for c in te if c["mode"] == t]
        if not cs:
            continue
        ep = np.array([c["err_plain"] for c in cs])
        eh = np.array([c["err_huber"] for c in cs])
        er = np.array([c["err_rec"] for c in cs])
        before_after[t] = (float(np.median(ep)), float(np.median(er)))
        na = sum(c["n_att"] for c in cs)
        hits = sum(c["hits"] for c in cs)
        niso = sum(c["n_iso"] for c in cs)
        rows.append({
            "attack": t, "n": len(cs),
            "rmse_naive_pu": float(np.sqrt(np.mean(ep ** 2))), "rmse_huber_pu": float(np.sqrt(np.mean(eh ** 2))),
            "rmse_recovered_pu": float(np.sqrt(np.mean(er ** 2))),
            "median_naive_pu": float(np.median(ep)), "median_recovered_pu": float(np.median(er)),
            "p90_recovered_pu": float(np.quantile(er, 0.9)),
            "iso_recall": hits / na if na else float("nan"), "iso_precision": hits / niso if niso else float("nan"),
            "wrong_action_naive": float(np.mean([c["wrong_naive"] for c in cs])),
            "wrong_action_recovered": float(np.mean([c["wrong_rec"] for c in cs])),
            "proceed_rate": gm["per_type"][t]["proceed"], "unsafe_proceed": gm["per_type"][t]["unsafe_proceed"],
        })

    # ---- risk-coverage vs baselines (selector = score; lower = more trusted)
    E = gm["_E"]
    unsafe = E > cfg.tol_pu
    Fte = feats(te)
    j_raw = Fte[:, FEATURE_NAMES.index("log_J_raw")]
    rc = {
        "GridSentinel gate": risk_coverage(gm["_U"], unsafe),
        "chi-square (raw J)": risk_coverage(j_raw, unsafe),
        "always-proceed": (np.array([1.0]), np.array([unsafe.mean()]), float(unsafe.mean())),
    }

    # ---- ablations (recovery error, 90th percentile, on the same attack mix)
    ablation = {}
    if not a.no_ablation:
        log("ablations")
        variants = {
            "Full GridSentinel": (model, cfg),
            "No trust weighting": (model, PipelineConfig(use_trust=False)),
            "No observability guard": (model, PipelineConfig(use_obs_guard=False)),
            "Basic meter set (no flows)": (GridModel("basic"), cfg),
        }
        for name, (m, c) in variants.items():
            cs = gen_cases(m, c, max(n, 100), np.random.default_rng(a.seed + 100))
            e = np.array([x["err_rec"] for x in cs])
            ablation[name] = {"median": float(np.median(e)), "p90": float(np.quantile(e, 0.9)),
                              "naive_p90": float(np.quantile([x["err_plain"] for x in cs], 0.9)),
                              "iso_recall": sum(x["hits"] for x in cs) / max(sum(x["n_att"] for x in cs), 1),
                              "false_iso": sum(x["false_iso"] for x in cs) / len(cs)}
        # always-proceed ("no gate")
        ablation["No gate (always proceed)"] = {"unsafe_rate": float(unsafe.mean()), "p90": ablation["Full GridSentinel"]["p90"]}

    runtime = {"median_ms": float(np.median([c["ms"] for c in te])), "p95_ms": float(np.quantile([c["ms"] for c in te], 0.95))}

    # ---- outputs
    import csv
    with open(os.path.join(a.out, "results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    make_plots(a.out, gm, rc, before_after, ablation)

    strip = lambda d: {k: v for k, v in d.items() if not k.startswith("_")}
    summary = {"seed": a.seed, "n_per_split": n, "deployed": strip(gm), "hardened": strip(hm),
               "stealth_full_deployed": strip(sfm), "stealth_full_hardened": strip(sfm_h),
               "detection": det, "per_attack": rows, "ablation": ablation, "runtime_ms": runtime,
               "aurc": {k: v[2] for k, v in rc.items()}, "protected_meters": [p[1] for p in plan],
               "alpha": cfg.alpha, "tol_pu": cfg.tol_pu}
    with open(os.path.join(a.out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2, default=float)

    P = lambda x: f"{100 * x:.1f}%" if np.isfinite(x) else "n/a"
    md = [f"# GridSentinel AI - batch evaluation (seed {a.seed}, {n} scenarios per split)",
          "", "Simulation on IEEE 14-bus (122 AC measurements). Synthetic attacks, attacker budget K_MAX=8 meters. "
          "Truth is used only to score results.", "",
          "## Headline (deployed grid, in-model attacks)",
          f"- Gate target: P(max voltage error <= {cfg.tol_pu} pu) >= {100 * (1 - cfg.alpha):.0f}% per bound",
          f"- Bound coverage on held-out test: **{P(gm['bound_coverage'])}**",
          f"- PROCEED share: **{P(gm['proceed_rate'])}**; unsafe among PROCEED: **{P(gm['unsafe_given_proceed'])}**; "
          f"unsafe-PROCEED over all cases: **{P(gm['unsafe_proceed_joint'])}** (always-proceed would be {P(gm['unsafe_rate_all'])})",
          f"- Wrong voltage-limit action among PROCEED: {P(gm['wrong_action_given_proceed'])}",
          f"- Runtime per snapshot: median {runtime['median_ms']:.0f} ms, p95 {runtime['p95_ms']:.0f} ms",
          "", "## Hardened grid (5 protected meters: " + ", ".join(p[1] for p in plan) + ")",
          f"- PROCEED {P(hm['proceed_rate'])}; unsafe among PROCEED {P(hm['unsafe_given_proceed'])}; bound coverage {P(hm['bound_coverage'])}",
          "", "## Limit: out-of-model full stealth (attacker controls the whole stealth support)",
          f"- Deployed: unsafe rate {P(sfm['unsafe_rate_all'])}, PROCEED {P(sfm['proceed_rate'])}, unsafe-PROCEED {P(sfm['unsafe_proceed_joint'])}",
          f"- Hardened: unsafe rate {P(sfm_h['unsafe_rate_all'])}, PROCEED {P(sfm_h['proceed_rate'])}, unsafe-PROCEED {P(sfm_h['unsafe_proceed_joint'])}",
          "- Out-of-model: the attacker controls far more than K_MAX meters. Perfect stealth leaves every residual unchanged, so no estimator-side method can detect it; "
          "the guarantee holds only inside the stated attacker budget. Hardening did NOT fix this case in our tests - we report the number as measured.",
          "", "## Detection vs baselines (attacked vs clean, deployed test set)",
          "| Method | Precision | Recall | F1 | FPR |", "|---|---|---|---|---|"]
    md += [f"| {k} | {P(v['precision'])} | {P(v['recall'])} | {P(v['f1'])} | {P(v['fpr'])} |" for k, v in det.items()]
    md += ["", "## Recovery error per attack type (max voltage error, pu)",
           "| Attack | n | RMSE naive | RMSE Huber | RMSE recovered | p90 recovered | Iso recall | Iso precision | Wrong action naive | Wrong action recovered | PROCEED | Unsafe-PROCEED |",
           "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    md += [f"| {r['attack']} | {r['n']} | {r['rmse_naive_pu']:.4f} | {r['rmse_huber_pu']:.4f} | {r['rmse_recovered_pu']:.4f} | "
           f"{r['p90_recovered_pu']:.4f} | {P(r['iso_recall'])} | {P(r['iso_precision'])} | {P(r['wrong_action_naive'])} | "
           f"{P(r['wrong_action_recovered'])} | {P(r['proceed_rate'])} | {P(r['unsafe_proceed'])} |" for r in rows]
    md += ["", "## Risk-coverage (AURC, lower is better)"] + [f"- {k}: {v[2]:.4f}" for k, v in rc.items()]
    if ablation:
        md += ["", "## Ablations (p90 max voltage error, pu)", "| Variant | median | p90 | iso recall | false isolations / case |", "|---|---|---|---|---|"]
        for k, v in ablation.items():
            if "median" in v:
                md.append(f"| {k} | {v['median']:.4f} | {v['p90']:.4f} | {P(v['iso_recall'])} | {v['false_iso']:.2f} |")
            else:
                md.append(f"| {k} | - | - | - | unsafe rate {P(v['unsafe_rate'])} |")
    md += ["", "Files: results.csv, summary.json, gate_calibration.json, calibration.png, risk_coverage.png, "
           "error_before_after.png, ablation.png", "",
           "Note: 'wrong action' compares which buses violate [0.95, 1.05] pu; it is also non-zero on clean data because several buses sit near a limit, so read it relative to the 'none' row.", "",
           "Caveats: simulated measurements and attacks on one network; results hold for the stated attack model and load range "
           "(+-20%). Re-run with your own seed and report the numbers you measure."]
    with open(os.path.join(a.out, "results_summary.md"), "w") as f:
        f.write("\n".join(md))
    log("done -> " + a.out)
    print("\n".join(md))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile test_gridsentinel.py
"""Run: python -m pytest -q test_gridsentinel.py   (or: python test_gridsentinel.py)"""
import ast, inspect
import numpy as np
import gridsentinel_core as g

M = g.GridModel()
CFG = g.PipelineConfig()


def _sc(seed=1):
    rng = np.random.default_rng(seed)
    return rng, g.make_scenarios(M, 1, rng)[0]


def test_jacobian_matches_finite_differences():
    x = M.x_base.copy()
    J = M.jac(x)
    rng = np.random.default_rng(0)
    for j in rng.choice(len(x), 6, replace=False):
        e = np.zeros(len(x)); e[j] = 1e-6
        fd = (M.h(x + e) - M.h(x - e)) / 2e-6
        assert np.allclose(J[:, j], fd, atol=1e-4, rtol=1e-4)


def test_clean_recovery_is_accurate():
    rng, sc = _sc(2)
    res = g.run_pipeline(M, sc.z_clean, CFG)
    assert res.converged and g.true_error(M, sc.x_true, res.rec_V) < 0.01


def test_big_attack_isolated_and_recovered():
    rng, sc = _sc(3)
    z, info = g.apply_attack(M, sc, "random_single", rng, CFG)
    res = g.run_pipeline(M, z, CFG)
    assert set(info.idx) <= set(res.iso_idx)
    assert g.true_error(M, sc.x_true, res.rec_V) < 0.01   # back to the noise floor after isolation


def test_full_stealth_leaves_chi_square_unchanged():
    rng, sc = _sc(4)
    z, _ = g.apply_attack(M, sc, "stealth_full", rng, CFG, bus=8)
    a = M.wls(sc.z_clean, M.avail); b = M.wls(z, M.avail)
    ra = np.sum(((sc.z_clean - M.h(a.x)) / M.sigma)[M.avail] ** 2)
    rb = np.sum(((z - M.h(b.x)) / M.sigma)[M.avail] ** 2)
    assert abs(ra - rb) / ra < 1e-3


def test_live_path_never_touches_truth():
    src = inspect.getsource(g.run_pipeline)
    tree = ast.parse(src)
    names = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name)} | {n.attr for n in ast.walk(tree) if isinstance(n, ast.Attribute)}
    assert not (names & {"x_true", "z_clean", "true_error", "Scenario", "apply_attack", "AttackInfo"})
    assert list(inspect.signature(g.run_pipeline).parameters)[:2] == ["model", "z"]


def test_seed_reproducibility():
    a = g.run_gridsentinel(seed=5, attack_mode="multi_coordinated", model=M, calibration=_cal())
    b = g.run_gridsentinel(seed=5, attack_mode="multi_coordinated", model=M, calibration=_cal())
    assert a["true_error"] == b["true_error"] and a["isolated"] == b["isolated"]


_CAL = None
def _cal():
    global _CAL
    if _CAL is None:
        _CAL = g.build_calibration(M, 120, 120, seed=3)
    return _CAL


def test_gate_bound_is_finite_and_ordered():
    r = g.run_gridsentinel(seed=6, attack_mode="none", model=M, calibration=_cal())
    assert r["bound_pu"] > 0 and 0 <= r["conf"] <= 100
    assert r["level"] in ("green", "yellow", "red")


def test_conformal_coverage_sanity():
    rng = np.random.default_rng(11)
    cases = []
    for sc in g.make_scenarios(M, 120, rng):
        z, _ = g.apply_attack(M, sc, str(rng.choice(list(g.IN_MODEL_MIX))), rng, CFG)
        res = g.run_pipeline(M, z, CFG)
        cases.append((res.fvec, g.true_error(M, sc.x_true, res.rec_V)))
    cov = np.mean([e <= _cal().bound(f) for f, e in cases])
    assert cov > 0.75   # small calibration set -> loose check; batch_eval reports the real figure


def test_calibration_json_roundtrip():
    c = _cal(); c2 = g.GateCalibration.from_json(c.to_json())
    f = np.zeros(len(g.FEATURE_NAMES))
    assert abs(c.predict_log(f) - c2.predict_log(f)) < 1e-9


if __name__ == "__main__":
    import pytest, sys
    sys.exit(pytest.main(["-q", __file__]))


In [ ]:
!python -m pytest -q test_gridsentinel.py 2>&1 | tail -3

In [ ]:
from gridsentinel_core import run_gridsentinel
r = run_gridsentinel(seed=42, attack_mode='random_single')
print(r['label'], '| conf', round(r['conf']), '% | bound', round(r['bound_pu'],4), 'pu | isolated:', r['isolated'])
print(r['recovery']['note'])

## Batch evaluation (~2 min) - these are the numbers you quote

In [ ]:
!python batch_eval.py --n 300 --seed 7 --out results | tail -60
!cp results/gate_calibration.json .

In [ ]:
from IPython.display import Image, display
for f in ['calibration','risk_coverage','error_before_after','ablation']:
    display(Image('results/'+f+'.png'))

## Multi-grid experiments: IEEE 14, 30, 57 and 118 (simulation only)
The same pipeline, same settings, re-calibrated on each grid. Saved full results (5 seeds for 14/30/57, 3 for 118):

| Grid | Bound coverage (target 0.90) | Unsafe given PROCEED | Clean false alarm | Median error naive -> recovered (pu) |
|---|---|---|---|---|
| IEEE 14 | 0.907 | 4.4% | 2.3% | 0.0063 -> 0.0032 |
| IEEE 30 | 0.918 | 4.1% | 3.7% | 0.0067 -> 0.0046 |
| IEEE 57 | 0.909 | 3.8% | 7.1% | 0.0063 -> 0.0053 |
| IEEE 118 | 0.895 | 5.6% | 17.2% | 0.0030 -> 0.0024 |

Recovery helps less on bigger grids and clean false alarms rise with size. Full details: `docs/EXPERIMENTS.md`. The cell below re-runs a quick version (about 1 minute).

In [ ]:
%%writefile multi_grid_eval.py
#!/usr/bin/env python
"""GridSentinel AI - multi-grid, multi-seed evaluation (NEW file; no existing file is modified).

Runs the SAME pipeline (no re-tuning) on IEEE 14 / 30 / 57 / 118 test systems from pandapower.
For each grid and each seed it builds its own train / calibration / test split, fits the
conformal trust gate on that grid, and scores it on held-out test scenarios.

How the grid is switched without touching gridsentinel_core.py:
    gridsentinel_core builds its network with pandapower.networks.case14(); here that
    function is temporarily pointed at the requested case. Nothing else changes.

Usage:
    python multi_grid_eval.py --cases case14 case30 case57 --n 150 --seeds 7 11 13 --out results_multigrid
    python multi_grid_eval.py --quick            # smoke test

SIMULATION ONLY: synthetic attacks on pandapower benchmark grids. Truth is used only for scoring.
"""
import argparse
import json
import os
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")

import pandapower.networks as pn  # noqa: E402

_ORIG = {c: getattr(pn, c) for c in ("case14", "case30", "case57", "case118") if hasattr(pn, c)}

from gridsentinel_core import GateCalibration, GridModel, PipelineConfig  # noqa: E402
from batch_eval import feats, errs, gen_cases, gate_metrics  # noqa: E402

ATTACK_TYPES = None  # use the repo's default in-model mix


def make_model(case):
    """Point core's case14() call at the requested grid, build the model, restore."""
    saved = pn.case14
    pn.case14 = _ORIG[case]
    try:
        return GridModel()
    finally:
        pn.case14 = saved


def ci95(x):
    x = np.asarray([v for v in x if v == v], float)
    if len(x) < 2:
        return float(x.mean()) if len(x) else float("nan"), float("nan")
    return float(x.mean()), float(1.96 * x.std(ddof=1) / np.sqrt(len(x)))


def one_run(case, seed, n, cfg):
    model = make_model(case)
    rng = np.random.default_rng(seed)
    t0 = time.time()
    tr = gen_cases(model, cfg, n, rng)
    ca = gen_cases(model, cfg, n, rng)
    te = gen_cases(model, cfg, n, rng)
    cal = GateCalibration.fit(feats(tr), errs(tr), feats(ca), errs(ca), cfg.tol_pu, cfg.alpha)
    gm = gate_metrics(cal, te, cfg)
    att = [c for c in te if c["mode"] != "none"]
    clean = [c for c in te if c["mode"] == "none"]
    hits = sum(c["hits"] for c in att)
    n_att = sum(c["n_att"] for c in att)
    out = {
        "case": case, "seed": seed, "n_test": len(te), "n_bus": model.nb,
        "bound_coverage": gm["bound_coverage"],
        "proceed_rate": gm["proceed_rate"],
        "unsafe_given_proceed": gm["unsafe_given_proceed"],
        "unsafe_proceed_joint": gm["unsafe_proceed_joint"],
        "unsafe_rate_all": gm["unsafe_rate_all"],
        "isolation_recall": hits / max(n_att, 1),
        "clean_false_alarm": float(np.mean([c["n_iso"] > 0 or c["res"].hard_fail for c in clean])) if clean else float("nan"),
        "n_clean": len(clean),
        "median_err_naive": float(np.median([c["err_plain"] for c in att])) if att else float("nan"),
        "median_err_recovered": float(np.median([c["err_rec"] for c in att])) if att else float("nan"),
        "median_ms_pipeline": float(np.median([c["ms"] for c in te])),
        "wall_s": time.time() - t0,
    }
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cases", nargs="+", default=["case14", "case30", "case57"])
    ap.add_argument("--n", type=int, default=150, help="scenarios per split (train/cal/test)")
    ap.add_argument("--seeds", nargs="+", type=int, default=[7, 11, 13])
    ap.add_argument("--out", default="results_multigrid")
    ap.add_argument("--quick", action="store_true")
    a = ap.parse_args()
    if a.quick:
        a.n, a.seeds = 40, [7]
    os.makedirs(a.out, exist_ok=True)
    cfg = PipelineConfig()
    rows = []
    for case in a.cases:
        for s in a.seeds:
            r = one_run(case, s, a.n, cfg)
            rows.append(r)
            print(f"{case} seed={s}: cov={r['bound_coverage']:.3f} proceed={r['proceed_rate']:.2f} "
                  f"unsafe|proceed={r['unsafe_given_proceed']:.3f} iso_rec={r['isolation_recall']:.2f} "
                  f"clean_FA={r['clean_false_alarm']:.2f} err {r['median_err_naive']:.4f}->{r['median_err_recovered']:.4f} "
                  f"{r['median_ms_pipeline']:.0f}ms/case wall={r['wall_s']:.0f}s", flush=True)
            with open(os.path.join(a.out, "runs.json"), "w") as f:
                json.dump(rows, f, indent=1)
    keys = ["bound_coverage", "proceed_rate", "unsafe_given_proceed", "unsafe_proceed_joint", "isolation_recall",
            "clean_false_alarm", "median_err_naive", "median_err_recovered", "median_ms_pipeline"]
    summ = {}
    for case in a.cases:
        rr = [r for r in rows if r["case"] == case]
        summ[case] = {"n_bus": rr[0]["n_bus"], "seeds": [r["seed"] for r in rr], "n_test_per_seed": rr[0]["n_test"]}
        for k in keys:
            m, h = ci95([r[k] for r in rr])
            summ[case][k] = {"mean": m, "ci95": h}
    with open(os.path.join(a.out, "summary.json"), "w") as f:
        json.dump(summ, f, indent=1)
    L = ["# Multi-grid results (simulation only)", "",
         f"n per split = {a.n}, seeds = {a.seeds}. Mean +/- 95% CI over seeds (small number of seeds: CI is rough).", "",
         "| Grid | Buses | Bound coverage (target 0.90) | PROCEED rate | Unsafe given PROCEED | Isolation recall | Clean false alarm | Median err naive -> recovered (pu) | ms/case |",
         "|---|---|---|---|---|---|---|---|---|"]
    f2 = lambda d: f"{d['mean']:.3f} +/- {d['ci95']:.3f}" if d['ci95'] == d['ci95'] else f"{d['mean']:.3f}"
    for case, s in summ.items():
        L.append(f"| {case} | {s['n_bus']} | {f2(s['bound_coverage'])} | {f2(s['proceed_rate'])} | {f2(s['unsafe_given_proceed'])} | "
                 f"{f2(s['isolation_recall'])} | {f2(s['clean_false_alarm'])} | "
                 f"{s['median_err_naive']['mean']:.4f} -> {s['median_err_recovered']['mean']:.4f} | {s['median_ms_pipeline']['mean']:.0f} |")
    open(os.path.join(a.out, "results_multigrid.md"), "w").write("\n".join(L) + "\n")
    print("\n".join(L))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile robustness_eval.py
#!/usr/bin/env python
"""GridSentinel AI - robustness tests (NEW file; no existing file is modified).

1) Distribution shift: calibrate the gate on the default generator, then test it on
   shifted load ranges and shifted attack mixes. Reports how far coverage drops.
2) Deep-learning baseline: an autoencoder trained on clean measurements (reconstruction
   error as the alarm) versus GridSentinel, same test scenarios, detection only.

SIMULATION ONLY (IEEE 14, pandapower). Truth is used only for scoring.
Usage: python robustness_eval.py --seeds 7 11 13 17 19 --n 300 --out results_robustness
"""
import argparse, json, os, warnings
import numpy as np
warnings.filterwarnings("ignore")
import batch_eval as be
from gridsentinel_core import GateCalibration, GridModel, PipelineConfig, IN_MODEL_MIX
from batch_eval import feats, errs, gen_cases, gate_metrics, chi2_stats

_orig_ms = be.make_scenarios

def with_load(lr, fn):
    be.make_scenarios = lambda m, n, r: _orig_ms(m, n, r, load_range=lr)
    try:
        return fn()
    finally:
        be.make_scenarios = _orig_ms

HEAVY = {"none": 0.10, "adaptive": 0.45, "stealth_partial": 0.45}

def conds(model, cfg, n, rng):
    return {
        "in-distribution (control)": lambda: gen_cases(model, cfg, n, rng),
        "load range 0.6-1.4 (was 0.8-1.2)": lambda: with_load((0.6, 1.4), lambda: gen_cases(model, cfg, n, rng)),
        "attack mix: 90% adaptive/stealth-partial": lambda: gen_cases(model, cfg, n, rng, mix=HEAVY),
        "both shifts together": lambda: with_load((0.6, 1.4), lambda: gen_cases(model, cfg, n, rng, mix=HEAVY)),
        "out-of-model: stealth_full only": lambda: gen_cases(model, cfg, n, rng, fixed_mode="stealth_full"),
    }

def autoencoder_part(model, cfg, n, rng, test_cases, full_stealth=None):
    from sklearn.neural_network import MLPRegressor
    from sklearn.preprocessing import StandardScaler
    clean_tr = gen_cases(model, cfg, n, rng, fixed_mode="none")
    clean_ca = gen_cases(model, cfg, n, rng, fixed_mode="none")
    Z = np.array([c["z"] for c in clean_tr]); sc = StandardScaler().fit(Z)
    ae = MLPRegressor(hidden_layer_sizes=(64, 16, 64), max_iter=400, random_state=0, early_stopping=True)
    Zs = sc.transform(Z); ae.fit(Zs, Zs)
    err = lambda z: float(np.mean((ae.predict(sc.transform(z[None, :]))[0] - sc.transform(z[None, :])[0]) ** 2))
    thr = float(np.quantile([err(c["z"]) for c in clean_ca], 0.95))
    rows = {"GridSentinel": {"tp": 0, "fn": 0, "fp": 0, "tn": 0}, "Autoencoder (clean-trained)": {"tp": 0, "fn": 0, "fp": 0, "tn": 0}}
    per = {}
    for c in test_cases:
        att = c["mode"] != "none"
        f = {"GridSentinel": c["res"].hard_fail or c["n_iso"] > 0,
             "Autoencoder (clean-trained)": err(c["z"]) > thr}
        for k, v in f.items():
            rows[k][("tp" if v else "fn") if att else ("fp" if v else "tn")] += 1
            if att:
                per.setdefault(c["mode"], {}).setdefault(k, []).append(bool(v))
    for c in (full_stealth or []):
        for k, v in {"GridSentinel": c["res"].hard_fail or c["n_iso"] > 0, "Autoencoder (clean-trained)": err(c["z"]) > thr}.items():
            per.setdefault("stealth_full (out-of-model)", {}).setdefault(k, []).append(bool(v))
    shifted = with_load((0.6, 1.4), lambda: gen_cases(model, cfg, n, rng, fixed_mode="none"))
    fa_shift = {"GridSentinel": float(np.mean([c["res"].hard_fail or c["n_iso"] > 0 for c in shifted])),
                "Autoencoder (clean-trained)": float(np.mean([err(c["z"]) > thr for c in shifted]))}
    out = {k: {"recall": d["tp"] / max(d["tp"] + d["fn"], 1), "false_alarm": d["fp"] / max(d["fp"] + d["tn"], 1)} for k, d in rows.items()}
    out["fa_load_shift"] = fa_shift
    out["per_mode_recall"] = {m: {k: float(np.mean(v)) for k, v in d.items()} for m, d in per.items()}
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", nargs="+", type=int, default=[7, 11, 13, 17, 19])
    ap.add_argument("--n", type=int, default=300)
    ap.add_argument("--out", default="results_robustness")
    ap.add_argument("--quick", action="store_true")
    a = ap.parse_args()
    if a.quick: a.n, a.seeds = 60, [7]
    os.makedirs(a.out, exist_ok=True)
    cfg = PipelineConfig(); model = GridModel()
    shift, ae_res = {}, []
    for s in a.seeds:
        rng = np.random.default_rng(s)
        tr = gen_cases(model, cfg, a.n, rng); ca = gen_cases(model, cfg, a.n, rng)
        cal = GateCalibration.fit(feats(tr), errs(tr), feats(ca), errs(ca), cfg.tol_pu, cfg.alpha)
        for name, fn in conds(model, cfg, a.n, rng).items():
            cases = fn(); gm = gate_metrics(cal, cases, cfg)
            shift.setdefault(name, []).append({k: gm[k] for k in ("bound_coverage", "proceed_rate", "unsafe_given_proceed", "unsafe_proceed_joint")})
            if name.startswith("in-distribution"):
                ae_res.append(autoencoder_part(model, cfg, a.n, rng, cases, gen_cases(model, cfg, a.n, rng, fixed_mode="stealth_full")))
        print("seed", s, "done", flush=True)
    m = lambda L, k: float(np.nanmean([d[k] for d in L]))
    ci = lambda L, k: float(1.96 * np.nanstd([d[k] for d in L], ddof=1) / np.sqrt(len(L))) if len(L) > 1 else float("nan")
    summ = {"shift": {n: {k: {"mean": m(L, k), "ci95": ci(L, k)} for k in L[0]} for n, L in shift.items()},
            "autoencoder": {k: {kk: {"mean": m([r[k] for r in ae_res], kk), "ci95": ci([r[k] for r in ae_res], kk)} for kk in ("recall", "false_alarm")}
                            for k in ("GridSentinel", "Autoencoder (clean-trained)")},
            "clean_false_alarm_load_shift": {k: {"mean": float(np.mean([r["fa_load_shift"][k] for r in ae_res])), "ci95": ci([{"v": r["fa_load_shift"][k]} for r in ae_res], "v")} for k in ("GridSentinel", "Autoencoder (clean-trained)")},
            "per_mode_recall": {mode: {k: float(np.mean([r["per_mode_recall"].get(mode, {}).get(k, np.nan) for r in ae_res])) for k in ("GridSentinel", "Autoencoder (clean-trained)")}
                                for mode in sorted({mm for r in ae_res for mm in r["per_mode_recall"]})},
            "seeds": a.seeds, "n": a.n}
    json.dump(summ, open(os.path.join(a.out, "summary.json"), "w"), indent=1)
    L = ["# Robustness results (IEEE 14, simulation only)", "", f"Gate calibrated on the default generator; seeds {a.seeds}, n = {a.n} per split. Mean +/- 95% CI.", "",
         "## 1. Distribution shift", "", "| Test condition | Bound coverage (target 0.90) | PROCEED rate | Unsafe given PROCEED |", "|---|---|---|---|"]
    f = lambda d: f"{d['mean']:.3f} +/- {d['ci95']:.3f}" if d['ci95'] == d['ci95'] else f"{d['mean']:.3f}"
    for n, d in summ["shift"].items():
        L.append(f"| {n} | {f(d['bound_coverage'])} | {f(d['proceed_rate'])} | {f(d['unsafe_given_proceed'])} |")
    L += ["", "## 2. Autoencoder baseline vs GridSentinel (detection only)", "", "| Method | Attack recall | False alarm on clean |", "|---|---|---|"]
    for k, d in summ["autoencoder"].items():
        L.append(f"| {k} | {f(d['recall'])} | {f(d['false_alarm'])} |")
    L += ["", "False alarm on CLEAN snapshots when the load range shifts to 0.6-1.4 (the autoencoder was trained on 0.8-1.2):", "", "| Method | False alarm on clean, shifted load |", "|---|---|"]
    for k, d in summ["clean_false_alarm_load_shift"].items():
        L.append(f"| {k} | {f(d)} |")
    L += ["", "Recall per attack type:", "", "| Attack | GridSentinel | Autoencoder |", "|---|---|---|"]
    for mode, d in summ["per_mode_recall"].items():
        L.append(f"| {mode} | {d['GridSentinel']:.2f} | {d['Autoencoder (clean-trained)']:.2f} |")
    open(os.path.join(a.out, "results_robustness.md"), "w").write("\n".join(L) + "\n")
    print("\n".join(L))

if __name__ == "__main__":
    main()


In [ ]:
!python multi_grid_eval.py --cases case14 case30 case57 --quick --out results_mg_quick | tail -12

## Robustness tests (IEEE 14, quick version about 1 minute)
1. Distribution shift: calibrate on the default generator, test on a different load range and attack mix.
2. Autoencoder baseline trained only on clean data, compared with GridSentinel on the same scenarios.

In [ ]:
!python robustness_eval.py --quick --out results_rob_quick | tail -45

In [ ]:
%%writefile hybrid_gate.py
#!/usr/bin/env python
"""GridSentinel AI - HybridGate (NEW file; no existing file is modified).

Wraps the shipped pipeline and adds ONE feature to the Trust Gate: the log reconstruction error of an
autoencoder trained only on clean measurements. Everything else is the shipped code.

    from hybrid_gate import HybridGate
    hg = HybridGate(GridModel()).fit(n=300, seed=7)      # about 1-2 minutes
    out = hg.assess(z)                                   # z = measurement vector
    out["hybrid"], out["shipped"]                        # ("PROCEED"|"CAUTION"|"VALIDATE")

    python hybrid_gate.py --demo                         # side-by-side on 5 attack types

EXPERIMENT (IEEE 14, simulation). See docs/HYBRID.md for results and limits."""
import argparse
import warnings
import numpy as np
warnings.filterwarnings("ignore")
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from gridsentinel_core import (GateCalibration, GridModel, PipelineConfig, apply_attack, decide_level,
                               make_scenarios, run_pipeline, true_error)
from batch_eval import errs, feats, gen_cases

NAME = {"green": "PROCEED", "yellow": "CAUTION", "red": "VALIDATE"}


class HybridGate:
    def __init__(self, model, cfg=None):
        self.model, self.cfg = model, cfg or PipelineConfig()

    def _ae_score(self, Z):
        Zs = self.sc.transform(np.atleast_2d(Z))
        return np.log10(np.mean((self.ae.predict(Zs) - Zs) ** 2, axis=1) + 1e-12)

    def fit(self, n=300, seed=7):
        rng = np.random.default_rng(seed)
        clean = gen_cases(self.model, self.cfg, n, rng, fixed_mode="none")
        Z = np.array([c["z"] for c in clean])
        self.sc = StandardScaler().fit(Z)
        Zs = self.sc.transform(Z)
        self.ae = MLPRegressor(hidden_layer_sizes=(64, 16, 64), max_iter=400, random_state=0, early_stopping=True).fit(Zs, Zs)
        tr, ca = gen_cases(self.model, self.cfg, n, rng), gen_cases(self.model, self.cfg, n, rng)
        aug = lambda cs: np.column_stack([feats(cs), self._ae_score(np.array([c["z"] for c in cs]))])
        t, a = self.cfg.tol_pu, self.cfg.alpha
        self.calA = GateCalibration.fit(feats(tr), errs(tr), feats(ca), errs(ca), t, a)
        self.calH = GateCalibration.fit(aug(tr), errs(tr), aug(ca), errs(ca), t, a)
        return self

    def assess(self, z):
        res = run_pipeline(self.model, z, self.cfg)
        out = {"result": res}
        for name, cal, f in (("shipped", self.calA, res.fvec),
                             ("hybrid", self.calH, np.append(res.fvec, self._ae_score(z)[0]))):
            U = cal.bound(f, self.cfg.alpha)
            conf = 100 * cal.prob_within(f, self.cfg.tol_pu)
            out[name] = NAME[decide_level(U, conf, res.hard_fail, self.cfg)[0]]
            out[name + "_bound_pu"] = float(U)
        return out


def demo(n=200, k=20, seed=3):
    model, cfg = GridModel(), PipelineConfig()
    hg = HybridGate(model, cfg).fit(n=n, seed=7)
    rng = np.random.default_rng(seed)
    print(f"\n{k} scenarios per attack type. Counts: PROCEED/CAUTION/VALIDATE; 'bad' = PROCEED although the true error exceeds {cfg.tol_pu} pu.\n")
    print(f"{'attack':18s} {'shipped P/C/V (bad)':24s} {'hybrid P/C/V (bad)':24s}")
    for mode in ("none", "random_single", "stealth_partial", "adaptive", "stealth_full"):
        cnt = {"shipped": [0, 0, 0], "hybrid": [0, 0, 0]}; bad = {"shipped": 0, "hybrid": 0}
        for sc in make_scenarios(model, k, rng):
            z, _ = apply_attack(model, sc, mode, rng, cfg)
            o = hg.assess(z)
            err = true_error(model, sc.x_true, o["result"].rec_V)
            for g in ("shipped", "hybrid"):
                i = ("PROCEED", "CAUTION", "VALIDATE").index(o[g]); cnt[g][i] += 1
                bad[g] += int(o[g] == "PROCEED" and err > cfg.tol_pu)
        sh = "/".join(map(str, cnt["shipped"])) + " (" + str(bad["shipped"]) + ")"
        hy = "/".join(map(str, cnt["hybrid"])) + " (" + str(bad["hybrid"]) + ")"
        print(f"{mode:18s} {sh:24s} {hy:24s}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", action="store_true")
    ap.add_argument("--n", type=int, default=200)
    a = ap.parse_args()
    demo(a.n) if a.demo else print(__doc__)


In [ ]:
%%writefile hybrid_gate_eval.py
#!/usr/bin/env python
"""GridSentinel AI - hybrid gate experiment (NEW file; no existing file is modified).

Question: if a learned anomaly score (autoencoder trained on clean measurements) is added as one extra
feature to the Trust Gate, do the weaknesses found in robustness_eval.py shrink?
Gate A = the shipped gate (11 physics features). Gate H = the same gate + 1 feature (log10 autoencoder error).
IEEE 14, simulation only. Truth is used only for scoring."""
import argparse, json, os, warnings
import numpy as np
warnings.filterwarnings("ignore")
import batch_eval as be
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from gridsentinel_core import GateCalibration, GridModel, PipelineConfig, decide_level
from batch_eval import feats, errs, gen_cases
from robustness_eval import with_load, HEAVY

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", nargs="+", type=int, default=[7, 11, 13, 17, 19])
    ap.add_argument("--n", type=int, default=300)
    ap.add_argument("--out", default="results_hybrid")
    ap.add_argument("--quick", action="store_true")
    a = ap.parse_args()
    if a.quick: a.n, a.seeds = 60, [7]
    os.makedirs(a.out, exist_ok=True)
    cfg, model = PipelineConfig(), GridModel()
    res = {}
    for s in a.seeds:
        rng = np.random.default_rng(s)
        clean = gen_cases(model, cfg, a.n, rng, fixed_mode="none")
        Z = np.array([c["z"] for c in clean]); sc = StandardScaler().fit(Z); Zs = sc.transform(Z)
        ae = MLPRegressor(hidden_layer_sizes=(64, 16, 64), max_iter=400, random_state=0, early_stopping=True).fit(Zs, Zs)
        def aescore(cases):
            Zc = sc.transform(np.array([c["z"] for c in cases]))
            return np.log10(np.mean((ae.predict(Zc) - Zc) ** 2, axis=1) + 1e-12)
        aug = lambda cases: np.column_stack([feats(cases), aescore(cases)])
        tr = gen_cases(model, cfg, a.n, rng); ca = gen_cases(model, cfg, a.n, rng)
        calA = GateCalibration.fit(feats(tr), errs(tr), feats(ca), errs(ca), cfg.tol_pu, cfg.alpha)
        calH = GateCalibration.fit(aug(tr), errs(tr), aug(ca), errs(ca), cfg.tol_pu, cfg.alpha)
        tests = {
            "default mix": lambda: gen_cases(model, cfg, a.n, rng),
            "load range 0.6-1.4": lambda: with_load((0.6, 1.4), lambda: gen_cases(model, cfg, a.n, rng)),
            "90% adaptive / stealth-partial": lambda: gen_cases(model, cfg, a.n, rng, mix=HEAVY),
            "perfect stealth (out-of-model)": lambda: gen_cases(model, cfg, a.n, rng, fixed_mode="stealth_full"),
        }
        for tn, fn in tests.items():
            cases = fn(); E = errs(cases)
            for gn, cal, F in (("A: shipped gate (11 features)", calA, feats(cases)), ("H: hybrid gate (+ autoencoder score)", calH, aug(cases))):
                U = np.array([cal.bound(f, cfg.alpha) for f in F])
                conf = np.array([100 * cal.prob_within(f, cfg.tol_pu) for f in F])
                lv = np.array([decide_level(u, cf, c["res"].hard_fail, cfg)[0] for u, cf, c in zip(U, conf, cases)])
                pr = lv == "green"; unsafe = E > cfg.tol_pu
                res.setdefault((gn, tn), []).append({"cov": float(np.mean(E <= U)), "proceed": float(pr.mean()),
                    "unsafe_given_proceed": float(unsafe[pr].mean()) if pr.any() else float("nan")})
        print("seed", s, flush=True)
    f = lambda L, k: (lambda v: f"{np.nanmean(v):.3f} +/- {1.96*np.nanstd(v, ddof=1)/np.sqrt(max(len(v),1)):.3f}" if len(v) > 1 else f"{np.nanmean(v):.3f}")([d[k] for d in L])
    L = ["# Hybrid gate vs shipped gate (IEEE 14, simulation only)", "", f"seeds {a.seeds}, n = {a.n}. Mean +/- 95% CI. 'unsafe given PROCEED' is nan when the gate never says PROCEED.", "",
         "| Test set | Gate | Bound coverage (target 0.90) | PROCEED rate | Unsafe given PROCEED |", "|---|---|---|---|---|"]
    for tn in ("default mix", "load range 0.6-1.4", "90% adaptive / stealth-partial", "perfect stealth (out-of-model)"):
        for gn in ("A: shipped gate (11 features)", "H: hybrid gate (+ autoencoder score)"):
            v = res[(gn, tn)]; L.append(f"| {tn} | {gn} | {f(v,'cov')} | {f(v,'proceed')} | {f(v,'unsafe_given_proceed')} |")
    open(os.path.join(a.out, "results_hybrid.md"), "w").write("\n".join(L) + "\n")
    json.dump({f"{k[0]} | {k[1]}": v for k, v in res.items()}, open(os.path.join(a.out, "runs.json"), "w"), indent=1)
    print("\n".join(L))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile stealth_magnitude_sweep.py
#!/usr/bin/env python
"""GridSentinel AI - how small can a perfect-stealth attack be before the hybrid gate stops noticing it?
NEW file; no existing file is modified. IEEE 14, simulation only.
The attacker shifts the voltage at one bus by dV (per unit) with a measurement bias a = H c (chi-square unchanged).
Reported per dV: share of attacks whose recovered-state error exceeds tolerance (0.01 pu), PROCEED rate,
and the share of ALL attacks that are both accepted (PROCEED) and unsafe (joint risk), for gate A and gate H."""
import argparse, json, os, warnings
import numpy as np
warnings.filterwarnings("ignore")
import batch_eval as be
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from gridsentinel_core import GateCalibration, GridModel, PipelineConfig, decide_level
from batch_eval import feats, errs, gen_cases

_orig = be.apply_attack
def set_dv(dv):
    if dv is None:
        be.apply_attack = _orig
    else:
        be.apply_attack = lambda model, sc, mode, rng, cfg=None, **kw: _orig(model, sc, "custom_stealth" if mode == "stealth_full" else mode, rng, cfg, size=dv if mode == "stealth_full" else None, **kw)

ap = argparse.ArgumentParser()
ap.add_argument("--seeds", nargs="+", type=int, default=[7, 11, 13, 17, 19])
ap.add_argument("--n", type=int, default=200)
ap.add_argument("--out", default="results_stealth_sweep")
ap.add_argument("--quick", action="store_true")
a = ap.parse_args()
DVS = [0.005, 0.01, 0.02, 0.03, 0.05]
if a.quick: a.n, a.seeds, DVS = 40, [7], [0.01, 0.05]
os.makedirs(a.out, exist_ok=True)
cfg, model = PipelineConfig(), GridModel()
res = {}
for s in a.seeds:
    rng = np.random.default_rng(s)
    set_dv(None)
    clean = gen_cases(model, cfg, a.n * 2, rng, fixed_mode="none")
    Z = np.array([c["z"] for c in clean]); sc = StandardScaler().fit(Z); Zs = sc.transform(Z)
    ae = MLPRegressor(hidden_layer_sizes=(64, 16, 64), max_iter=400, random_state=0, early_stopping=True).fit(Zs, Zs)
    aescore = lambda cases: np.log10(np.mean((ae.predict(sc.transform(np.array([c["z"] for c in cases]))) - sc.transform(np.array([c["z"] for c in cases]))) ** 2, axis=1) + 1e-12)
    aug = lambda cases: np.column_stack([feats(cases), aescore(cases)])
    tr = gen_cases(model, cfg, a.n * 2, rng); ca = gen_cases(model, cfg, a.n * 2, rng)
    gates = {"A (shipped)": (GateCalibration.fit(feats(tr), errs(tr), feats(ca), errs(ca), cfg.tol_pu, cfg.alpha), feats),
             "H (hybrid)": (GateCalibration.fit(aug(tr), errs(tr), aug(ca), errs(ca), cfg.tol_pu, cfg.alpha), aug)}
    for dv in DVS:
        set_dv(dv)
        cases = gen_cases(model, cfg, a.n, rng, fixed_mode="stealth_full"); set_dv(None)
        E = errs(cases); unsafe = E > cfg.tol_pu
        for gn, (cal, fx) in gates.items():
            F = fx(cases); U = np.array([cal.bound(f, cfg.alpha) for f in F]); conf = np.array([100 * cal.prob_within(f, cfg.tol_pu) for f in F])
            pr = np.array([decide_level(u, cf, c["res"].hard_fail, cfg)[0] == "green" for u, cf, c in zip(U, conf, cases)])
            res.setdefault((dv, gn), []).append({"attack_unsafe": float(unsafe.mean()), "proceed": float(pr.mean()), "joint": float(np.mean(pr & unsafe))})
    print("seed", s, flush=True)
m = lambda L, k: float(np.mean([d[k] for d in L]))
L = ["# Perfect-stealth magnitude sweep (IEEE 14, simulation only)", "", f"seeds {a.seeds}, {a.n} attacks per magnitude per seed.", "",
     "dV = voltage shift forced on one bus by the attacker. 'Attack matters' = share of attacks whose recovered-state error exceeds 0.01 pu. 'Accepted and unsafe' = share of ALL attacks where the gate said PROCEED although the error exceeds tolerance.", "",
     "| dV (pu) | Gate | Attack matters (error > 0.01 pu) | PROCEED rate | Accepted and unsafe |", "|---|---|---|---|---|"]
for (dv, gn), v in res.items():
    L.append(f"| {dv} | {gn} | {m(v,'attack_unsafe'):.2f} | {m(v,'proceed'):.2f} | {m(v,'joint'):.2f} |")
open(os.path.join(a.out, "results_stealth_sweep.md"), "w").write("\n".join(L) + "\n")
json.dump({f"{k[0]} | {k[1]}": v for k, v in res.items()}, open(os.path.join(a.out, "runs.json"), "w"), indent=1)
print("\n".join(L))


## Hybrid gate (experiment): physics + learned anomaly score
One extra feature is added to the Trust Gate: the reconstruction error of an autoencoder trained only on clean measurements. Saved results (IEEE 14, 5 seeds, n = 300):

| | Shipped gate | Hybrid gate |
|---|---|---|
| PROCEED rate | 35% | 69% |
| Unsafe after PROCEED | 4.5% | 1.3% |
| 90% adaptive / stealth-partial mix: unsafe after PROCEED | 32% | 5% |
| Large perfect-stealth attacks: PROCEED rate | 33% | 0.3% |

Limits: small perfect-stealth attacks (0.01 to 0.02 pu) still pass in 22 to 30% of cases, the bound coverage on a shifted attack mix is 0.85, and everything is simulation. Details: `docs/HYBRID.md`.

The next cell trains both gates (about 1 minute) and prints, for 20 scenarios per attack type, how often each says PROCEED / CAUTION / VALIDATE and how many PROCEEDs were unsafe.

In [ ]:
!python hybrid_gate.py --demo

In [ ]:
!python hybrid_gate_eval.py --quick --out results_hybrid_quick | tail -14

In [ ]:
%%writefile activsg_eval.py
#!/usr/bin/env python
"""GridSentinel AI - run the SAME pipeline on any MATPOWER case file, e.g. Texas A&M ACTIVSg200 (NEW file).

  py -3.12 activsg_eval.py --file case_ACTIVSg200.m --check            # convert + sanity report only
  py -3.12 activsg_eval.py --file case_ACTIVSg200.m --n 100 --seeds 7 11 --out results_activsg

ACTIVSg cases are SYNTHETIC but realistic grids (Texas A&M), not real utility data. Attacks are injected synthetically.
Truth is used only for scoring. Same settings as every other grid (no tuning)."""
import argparse, copy, json, os, warnings
import numpy as np
warnings.filterwarnings("ignore")
import pandapower as pp
from pandapower.converter.matpower import from_mpc
import multi_grid_eval as mg
from gridsentinel_core import PipelineConfig


def load_net(path):
    net = from_mpc(path, f_hz=60)
    # light cleaning: drop out-of-service elements, keep one slack
    for tbl in ("gen", "sgen", "load", "line", "trafo", "shunt", "ext_grid"):
        t = getattr(net, tbl, None)
        if t is not None and len(t) and "in_service" in t:
            t.drop(t.index[~t["in_service"].astype(bool)], inplace=True)
    return net


def report(net):
    pp.runpp(net, init="dc", max_iteration=50)
    vm = net.res_bus.vm_pu
    return {"buses": int(len(net.bus)), "lines": int(len(net.line)), "trafos": int(len(net.trafo)), "gens": int(len(net.gen)),
            "sgens": int(len(net.sgen)), "loads": int(len(net.load)), "shunts": int(len(net.shunt)), "ext_grids": int(len(net.ext_grid)),
            "power_flow_converged": bool(net.converged), "vm_min": float(vm.min()), "vm_max": float(vm.max()),
            "total_load_mw": float(net.res_load.p_mw.sum())}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--file", required=True)
    ap.add_argument("--check", action="store_true")
    ap.add_argument("--n", type=int, default=100)
    ap.add_argument("--seeds", nargs="+", type=int, default=[7, 11])
    ap.add_argument("--out", default="results_activsg")
    a = ap.parse_args()
    base = load_net(a.file)
    rep = report(copy.deepcopy(base))
    print(json.dumps(rep, indent=1))
    os.makedirs(a.out, exist_ok=True)
    json.dump(rep, open(os.path.join(a.out, "case_report.json"), "w"), indent=1)
    if a.check:
        return
    mg._ORIG["custom"] = lambda: copy.deepcopy(base)
    cfg, rows = PipelineConfig(), []
    for s in a.seeds:
        r = mg.one_run("custom", s, a.n, cfg); r["case"] = os.path.basename(a.file); rows.append(r)
        print(f"seed {s}: cov={r['bound_coverage']:.3f} proceed={r['proceed_rate']:.2f} unsafe|proceed={r['unsafe_given_proceed']:.3f} "
              f"iso_rec={r['isolation_recall']:.2f} clean_FA={r['clean_false_alarm']:.2f} err {r['median_err_naive']:.4f}->{r['median_err_recovered']:.4f} "
              f"{r['median_ms_pipeline']:.0f}ms/case", flush=True)
        json.dump(rows, open(os.path.join(a.out, "runs.json"), "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile case_ACTIVSg200.m
function mpc = case_ACTIVSg200
%CASE_ACTIVSG200  Synthetic Illinois 200-bus power system model.
%
%   This is an entirely synthetic 200 bus case, geographically situated
%   in the central part of the US state of Illinois. The case is
%   designed with a 230 and 115 kV transmission network to serve
%   a load that roughly mimics the actual population of its geographic
%   footprint. The synthetic transmission system was designed by
%   algorithms described in [1] to be statistically similar to actual
%   transmission system models but without modeling any actual lines.
%
%   When publishing results based on this data, please cite:
%
%   [1] A.B. Birchfield, T. Xu, K.M. Gegner, K.S. Shetye, T.J. Overbye,
%       "Grid Structural Characteristics as Validation Criteria for
%       Synthetic Networks," IEEE Transactions on Power Systems, 2017.
%
%   This is a synthetic power system model that does not represent the
%   actual grid. It was developed as part of the US ARPA-E GRID DATA
%   research project and contains no CEII.
%
%   One-line diagrams and other data formats available at:
%       https://electricgrids.engr.tamu.edu
%
%   May 16, 2017

%   Created from ACTIV_SG_200.pwb, saved by
%   PowerWorld Simulator, version 20 Beta, build date May 19, 2017,
%   then by MATPOWER 6.

%   MATPOWER
%   Copyright (c) 2017 by A.B. Birchfield, T. Xu, K.M. Gegner, K.S. Shetye,
%   and T.J. Overbye
%   Licensed under the Creative Commons Attribution 4.0 International license,
%   http://creativecommons.org/licenses/by/4.0/

%% MATPOWER Case Format : Version 2
mpc.version = '2';

%%-----  Power Flow Data  -----%%
%% system MVA base
mpc.baseMVA = 100;

%% bus data
%	bus_i	type	Pd	Qd	Gs	Bs	area	Vm	Va	baseKV	zone	Vmax	Vmin	lam_P	lam_Q	mu_Vmax	mu_Vmin
mpc.bus = [
	1	1	0	0	0	0	1	1.0191524	-7.085196	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	2	1	7.39	2.1	0	0	1	1.0190346	-7.098018	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	3	1	0	0	0	0	1	1.0300537	-10.029102	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	4	1	1.7	0.48	0	0	1	1.0300287	-10.031977	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	5	1	0	0	0	0	1	1.0372733	-3.546274	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	6	1	7.95	2.26	0	0	1	1.0371472	-3.559563	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	7	1	0	0	0	0	1	1.0130981	-7.793195	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	8	1	23.74	6.77	0	0	1	1.0127362	-7.833073	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	9	1	0	0	0	0	1	1.0157742	-7.471766	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	10	1	42.68	12.16	0	0	1	1.0150515	-7.561889	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	11	1	0	0	0	0	1	1.0464424	-7.604683	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	12	1	0.3	0.09	0	0	1	1.046438	-7.60522	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	13	1	2.52	0.72	0	0	1	1.0464027	-7.609214	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	14	1	0	0	0	0	1	1.044617	-6.245575	230	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	15	1	0	0	0	30	1	1.0491072	-7.105388	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	16	1	44.76	12.76	0	0	1	1.048481	-7.176325	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	17	1	0	0	0	0	1	1.0247944	-8.305174	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	18	1	0.71	0.2	0	0	1	1.0247831	-8.306647	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	19	1	0.77	0.22	0	0	1	1.0247839	-8.306404	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	20	1	1.13	0.32	0	0	1	1.024777	-8.307127	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	21	1	4.61	1.31	0	0	1	1.024716	-8.31463	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	22	1	0	0	0	0	1	1.0446532	-8.021549	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	23	1	1.41	0.4	0	0	1	1.0446325	-8.023805	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	24	1	2.4	0.68	0	0	1	1.0446174	-8.025162	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	25	1	0	0	0	0	1	1.0425139	-4.003351	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	26	1	1.35	0.39	0	0	1	1.0424947	-4.005417	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	27	1	0	0	0	0	1	1.0457273	-7.975065	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	28	1	2.33	0.67	0	0	1	1.045689	-7.979706	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	29	1	0	0	0	0	1	1.0181686	-7.054086	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	30	1	59.09	16.84	0	0	1	1.0172458	-7.149969	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	31	1	0	0	0	0	1	1.0215416	-5.832082	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	32	1	0.51	0.15	0	0	1	1.0215327	-5.833121	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	33	1	6.44	1.84	0	0	1	1.0214429	-5.844415	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	34	1	0	0	0	0	1	1.0327185	-9.326455	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	35	1	20.76	5.92	0	0	1	1.032392	-9.365615	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	36	1	0	0	0	0	1	1.0280697	-6.178381	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	37	1	3.57	1.02	0	0	1	1.0280176	-6.18452	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	38	1	29.81	8.5	0	0	1	1.0275847	-6.23205	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	39	1	7.41	2.11	0	0	1	1.024585	-10.493094	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	40	1	2.04	0.58	0	0	1	1.0245573	-10.496396	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	41	1	38.34	10.93	0	0	1	1.0462799	-7.96017	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	42	1	10.62	3.03	0	0	1	1.0301465	-11.069993	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	43	1	2.09	0.59	0	0	1	1.017201	-6.400585	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	44	1	4.15	1.18	0	0	1	1.0280137	-10.39886	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	45	1	0	0	0	0	1	1.0384367	-2.068677	230	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	46	1	6.34	1.81	0	0	1	1.0378124	-2.889424	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	47	1	1.54	0.44	0	0	1	1.0392618	-8.19532	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	48	1	1.82	0.52	0	0	1	1.0351468	-4.17501	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	49	2	0	0	0	0	1	1.04	-3.80269	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	50	2	0	0	0	0	1	1.04	-3.896095	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	51	2	0	0	0	0	1	1.04	-3.742621	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	52	2	0	0	0	0	1	1.04	-3.914588	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	53	2	0	0	0	0	1	1.04	-3.349326	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	54	1	2.66	0.76	0	0	1	1.0375005	-8.493516	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	55	1	0	0	0	0	1	1.0370718	-2.298751	230	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	56	1	1.31	0.37	0	0	1	1.0364415	-2.736582	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	57	1	2.82	0.8	0	0	1	1.0281336	-10.449132	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	58	1	4.23	1.21	0	0	1	1.0349414	-6.030088	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	59	1	4.73	1.35	0	0	1	1.0187885	-7.693149	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	60	1	3.8	1.08	0	0	1	1.0291097	-6.554687	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	61	1	20.55	5.86	0	0	1	1.0351527	-3.102719	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	62	1	21.41	6.1	0	0	1	1.0267194	-11.318015	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	63	1	7.98	2.27	0	0	1	1.0437777	-8.233032	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	64	1	0	0	0	0	1	1.0415417	-2.532761	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	65	2	0	0	0	0	1	1.04	2.119	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	66	1	0	0	0	0	1	1.0410702	-7.846441	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	67	2	0	0	0	0	1	1.0405589	-7.73215	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	68	2	0	0	0	0	1	1.04	-6.669196	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	69	2	0	0	0	0	1	1.04	-7.421629	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	70	2	0	0	0	0	1	1.04	-6.959081	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	71	2	0	0	0	0	1	1.04	-7.48165	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	72	2	0	0	0	0	1	1.04	-7.512144	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	73	2	0	0	0	0	1	1.04	-7.193069	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	74	1	18.76	5.35	0	0	1	1.0294816	-5.12964	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	75	1	3.6	1.03	0	0	1	1.028977	-6.116296	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	76	2	0	0	0	0	1	1.04	-5.641354	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	77	2	0	0	0	0	1	1.04	-5.292751	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	78	2	0	0	0	0	1	1.028977	-6.116296	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	79	2	0	0	0	0	1	1.028977	-6.116296	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	80	1	1.48	0.42	0	0	1	1.0475622	-7.25821	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	81	1	0	0	0	0	1	1.038152	-3.896426	230	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	82	1	7.6	2.17	0	0	1	1.0353106	-4.72467	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	83	1	3.14	0.89	0	0	1	1.0305903	-5.147124	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	84	1	3.75	1.07	0	0	1	1.0192369	-5.836735	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	85	1	4.83	1.38	0	0	1	1.0241032	-9.891841	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	86	1	8.25	2.35	0	0	1	1.0142895	-7.633369	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	87	1	0	0	0	0	1	1.0382378	-9.281203	230	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	88	1	49.34	14.06	0	0	1	1.0367741	-10.475486	115	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	89	1	62.85	17.91	0	0	1	1.0367956	-6.001781	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	90	2	0	0	0	0	1	1.04	-4.84285	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	91	2	0	0	0	0	1	1.04	-5.608287	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	92	2	0	0	0	0	1	1.0367956	-6.001781	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	93	1	3.06	0.87	0	0	1	1.0467635	-7.705981	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	94	2	0	0	0	0	1	1.0412845	-6.897097	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	95	1	3	0.85	0	30	1	1.0452705	-6.143695	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	96	1	2.96	0.84	0	0	1	1.0399843	-4.622369	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	97	1	7.33	2.09	0	0	1	1.0266279	-8.070874	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	98	1	0	0	0	0	1	1.0259732	-5.693301	230	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	99	1	12.17	3.47	0	0	1	1.021759	-6.532648	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	100	1	37.39	10.66	0	80	1	1.05559	-7.845164	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	101	1	0.52	0.15	0	0	1	1.0153419	-7.486883	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	102	1	0	0	0	0	1	1.0375884	-1.855349	230	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	103	1	0	0	0	0	1	1.0361908	-2.820525	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	104	2	0	0	0	0	1	1.04	1.292356	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	105	2	0	0	0	0	1	1.04	3.030149	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	106	1	2.06	0.59	0	0	1	1.0333053	-3.587524	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	107	1	52.56	14.98	0	0	1	1.0164253	-5.897801	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	108	1	9.42	2.68	0	0	1	1.02631	-6.590908	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	109	1	5.16	1.47	0	0	1	1.0266885	-7.119037	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	110	1	10.86	3.09	0	0	1	1.0153998	-7.494693	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	111	1	8.82	2.51	0	0	1	1.0422996	-8.069078	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	112	1	0	0	0	0	1	1.0338868	-2.38601	230	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	113	1	0	0	0	0	1	1.0214787	-5.246294	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	114	2	0	0	0	0	1	1.0362036	-2.072597	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	115	2	0	0	0	0	1	1.04	2.075284	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	116	1	0	0	0	0	1	1.0260581	-5.40757	230	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	117	1	34.08	9.71	0	0	1	1.0194057	-6.761358	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	118	1	3.94	1.12	0	0	1	1.0220863	-6.761277	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	119	1	17.55	5	0	0	1	1.0162966	-7.890283	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	120	1	1.8	0.51	0	0	1	1.0244523	-9.26074	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	121	1	0	0	0	0	1	1.0413751	-2.550667	230	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	122	1	15.44	4.4	0	0	1	1.0392113	-3.476706	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	123	1	0	0	0	0	1	1.0317706	-4.799729	230	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	124	1	0	0	0	0	1	1.0241995	-6.311793	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	125	2	0	0	0	0	1	1.04	-2.638002	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	126	2	0	0	0	0	1	1.04	-3.433514	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	127	2	0	0	0	0	1	1.04	-4.153305	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	128	1	0	0	0	0	1	1.0336624	-2.641873	230	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	129	1	77.24	22.01	0	0	1	1.0199071	-5.280933	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	130	1	4.4	1.25	0	0	1	1.0308537	-4.215383	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	131	1	30.25	8.62	0	0	1	1.0159516	-7.483658	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	132	1	26.96	7.68	0	0	1	1.0142019	-7.143976	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	133	1	0	0	0	0	1	1.033428	-4.376173	230	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	134	1	0	0	0	0	1	1.0315652	-5.231244	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	135	2	0	0	0	0	1	1.04	-1.987898	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	136	2	0	0	0	0	1	1.04	-2.419807	13.8	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	137	1	1.16	0.33	0	0	1	1.0313258	-9.719095	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	138	1	2.09	0.6	0	0	1	1.0307923	-5.699318	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	139	1	6.84	1.95	0	0	1	1.0214166	-7.361468	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	140	1	8.21	2.34	0	0	1	1.0231103	-6.390899	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	141	1	1.42	0.41	0	0	1	1.0153587	-7.521902	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	142	1	37.26	10.62	0	0	1	1.0140717	-7.617783	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	143	1	26.78	7.63	0	0	1	1.0372589	-6.549447	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	144	1	9.05	2.58	0	0	1	1.0236389	-5.928282	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	145	1	17.54	5	0	0	1	1.0320168	-11.164121	115	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	146	1	0	0	0	0	1	1.0331184	-4.010275	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	147	2	0	0	0	0	1	1.04	1.000906	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	148	1	32.02	9.13	0	0	1	1.0102299	-8.189318	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	149	1	0	0	0	0	1	1.0395559	-8.879345	230	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	150	1	0	0	0	0	1	1.039664	-9.999925	115	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	151	2	0	0	0	0	1	1.04	-8.34258	13.8	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	152	2	0	0	0	0	1	1.04	-7.139644	13.8	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	153	2	0	0	0	0	1	1.04	-6.612046	13.8	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	154	2	0	0	0	0	1	1.04	-7.61542	13.8	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	155	2	0	0	0	0	1	1.04	-7.653539	13.8	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	156	1	0	0	0	0	1	1.0321033	-4.724707	230	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	157	1	14.76	4.21	0	0	1	1.0305206	-5.439245	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	158	1	8.97	2.56	0	0	1	1.0434577	-7.900145	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	159	1	1.39	0.4	0	0	1	1.0265896	-10.742416	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	160	1	0	0	0	0	1	1.0311472	-11.239582	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	161	2	0	0	0	0	1	1.0311472	-11.239582	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	162	1	1.45	0.41	0	0	1	1.0225791	-6.221058	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	163	1	45.8	13.05	0	0	1	1.0431643	-7.926244	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	164	2	0	0	0	0	1	1.0431643	-7.926244	13.8	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	165	2	0	0	0	0	1	1.0431643	-7.926244	13.8	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	166	2	0	0	0	0	1	1.0431643	-7.926244	13.8	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	167	2	0	0	0	0	1	1.0414716	-7.628263	13.8	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	168	2	0	0	0	0	1	1.0431643	-7.926244	13.8	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	169	2	0	0	0	0	1	1.0431643	-7.926244	13.8	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	170	2	0	0	0	0	1	1.04	-7.359358	13.8	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	171	1	7.94	2.26	0	0	1	1.0293606	-5.544471	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	172	1	5.38	1.53	0	0	1	1.0446624	-6.737828	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	173	1	8.71	2.48	0	0	1	1.0438694	-7.824896	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	174	1	2.77	0.79	0	0	1	1.048556	-6.975714	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	175	1	7.31	2.08	0	0	1	1.042508	-8.051141	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	176	1	19.01	5.42	0	0	1	1.034057	-10.848998	115	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	177	1	11.29	3.22	0	0	1	1.0279083	-5.445402	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	178	1	0	0	0	0	1	1.0414926	-4.889523	230	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	179	1	23.61	6.73	0	0	1	1.0462277	-6.907941	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	180	1	7.74	2.21	0	0	1	1.045211	-7.124937	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	181	1	47.98	13.67	0	0	1	1.0333892	-11.195233	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	182	2	0	0	0	0	1	1.04	-10.37446	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	183	2	0	0	0	0	1	1.04	-10.723282	13.8	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	184	1	10.69	3.05	0	0	1	1.0492205	-8.289414	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	185	1	10.79	3.07	0	0	1	1.0341291	-10.871111	115	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	186	1	2.1	0.6	0	0	1	1.0294683	-5.915141	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	187	1	0	0	0	0	1	1.0411088	-1.59351	230	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	188	1	0	0	0	0	1	1.0422271	-4.274235	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	189	3	0	0	0	0	1	1.04	0	13.8	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	190	1	16.79	4.78	0	0	1	1.0275107	-5.607695	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	191	1	7.23	2.06	0	0	1	1.0460774	-7.762654	115	5	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	192	1	52.74	15.03	0	0	1	1.0166276	-5.99268	115	7	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	193	1	21.99	6.27	0	0	1	1.0164747	-7.379588	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	194	1	37.57	10.71	0	50	1	1.0400826	-10.703489	115	3	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	195	1	0	0	0	0	1	1.0332601	-5.127426	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	196	2	0	0	0	0	1	1.0332601	-5.127426	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	197	2	0	0	0	0	1	1.0332601	-5.127426	13.8	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	198	1	5.37	1.53	0	0	1	1.0232234	-6.849765	115	2	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	199	1	4.26	1.21	0	0	1	1.0438791	-5.721465	115	6	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
	200	1	9.34	2.66	0	0	1	1.0259148	-9.367606	115	4	1.1	0.9	6.8700	0.0000	0.0000	0.0000;
];

%% generator data
%	bus	Pg	Qg	Qmax	Qmin	Vg	mBase	status	Pmax	Pmin	Pc1	Pc2	Qc1min	Qc1max	Qc2min	Qc2max	ramp_agc	ramp_10	ramp_30	ramp_q	apf	mu_Pmax	mu_Pmin	mu_Qmax	mu_Qmin
mpc.gen = [
	49	1.36	0.88	2.11	-0.55	1.04	5.44	1	4.53	1.36	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	50	1.36	1.18	2.11	-0.55	1.04	5.44	1	4.53	1.36	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	51	1.36	0.77	2.11	-0.55	1.04	5.44	1	4.53	1.36	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	52	1.36	1.25	2.11	-0.55	1.04	5.44	1	4.53	1.36	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	53	2.72	0.79	4.23	-1.11	1.04	10.88	1	9.07	2.72	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	65	86.5	0.84	32.04	-21.66	1.04	180.48	1	86.5	86.5	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	67	1.41	-0.57	2.19	-0.57	1.04	5.64	1	4.7	1.41	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	68	8.38	-0.55	13.01	-3.41	1.04	33.5	1	27.92	8.38	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	69	8.38	-1.53	13.01	-3.41	1.04	33.5	1	27.92	8.38	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	70	8.38	-0.78	13.01	-3.41	1.04	33.5	1	27.92	8.38	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	71	8.38	-1.7	13.01	-3.41	1.04	33.5	1	27.92	8.38	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	72	8.38	-1.86	13.01	-3.41	1.04	33.5	1	27.92	8.38	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	73	8.38	-1.11	13.01	-3.41	1.04	33.5	1	27.92	8.38	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	76	1.2	1.41	2.04	-0.44	1.04	4.8	1	4	1.2	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	77	0.72	0.5	1.22	-0.27	1.04	2.88	1	2.4	0.72	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	78	0	0	9.16	-2	1.04	21.6	0	18	5.4	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	79	0	0	9.16	-2	1.04	21.6	0	18	5.4	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	90	0.96	0.13	1.63	-0.36	1.04	3.84	1	3.2	0.96	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	91	1.5	0.59	2.55	-0.55	1.04	6	1	5	1.5	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	92	0	0	3.21	-0.7	1.04	7.56	0	6.3	1.89	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	94	5.4	-2.2	8.39	-2.2	1.04	21.6	1	18	5.4	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	104	67.6	3.1	21.09	-14.26	1.04	118.8	1	67.6	67.6	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	105	154.8	8.29	42.17	-28.51	1.04	237.6	1	154.8	154.8	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	114	1.4	0.36	0.36	-0.24	1.04	2.04	1	1.4	1.4	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	115	133.5	12.6	31.95	-21.6	1.04	180	1	133.5	133.5	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	125	39.02	7.24	60.6	-15.87	1.04	156.06	1	130.05	39.02	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	126	39.02	12.55	60.6	-15.87	1.04	156.06	1	130.05	39.02	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	127	39.02	25.99	60.6	-15.87	1.04	156.06	1	130.05	39.02	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	135	133.92	21.46	208.02	-54.46	1.04	535.68	1	446.4	133.92	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	136	133.92	25.06	208.02	-54.46	1.04	535.68	1	446.4	133.92	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	147	92.4	8.79	21.41	-14.47	1.04	120.6	1	92.4	92.4	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	151	1.62	0.02	2.52	-0.66	1.04	6.48	1	5.4	1.62	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	152	23.17	0.27	35.99	-9.42	1.04	92.67	1	77.22	23.17	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	153	23.17	0.02	35.99	-9.42	1.04	92.67	1	77.22	23.17	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	154	23.17	0.14	35.99	-9.42	1.04	92.67	1	77.22	23.17	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	155	23.17	0.13	35.99	-9.42	1.04	92.67	1	77.22	23.17	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	161	0	0	70.55	-15.38	1.04	166.32	0	138.6	41.58	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	164	0	0	6.11	-1.33	1.04	14.4	0	12	3.6	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	165	0	0	13.23	-2.89	1.04	31.2	0	26	7.8	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	166	0	0	4.78	-1.04	1.04	11.28	0	9.4	2.82	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	167	2.82	-1.04	4.78	-1.04	1.04	11.28	1	9.4	2.82	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	168	0	0	4.78	-1.04	1.04	11.28	0	9.4	2.82	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	169	0	0	4.78	-1.04	1.04	11.28	0	9.4	2.82	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	170	2.82	-0.99	4.78	-1.04	1.04	11.28	1	9.4	2.82	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	182	5.25	2.15	8.16	-2.14	1.04	21	1	17.5	5.25	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	183	7.98	5.73	12.4	-3.25	1.04	31.92	1	26.6	7.98	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	189	384.37	-24.15	209.45	-46.67	1.04	682.98	1	569.15	170.75	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	196	0	0	34.36	-7.49	1.04	81	0	67.5	20.25	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
	197	0	0	34.36	-7.49	1.04	81	0	67.5	20.25	0	0	0	0	0	0	0	0	0	0	0	0.0000	0.0000	0.0000	0.0000;
];

%% branch data
%	fbus	tbus	r	x	b	rateA	rateB	rateC	ratio	angle	status	angmin	angmax	Pf	Qf	Pt	Qt	mu_Sf	mu_St	mu_angmin	mu_angmax
mpc.branch = [
	2	1	0.000673	0.003339	0	100	0	0	0	0	1	0	0	-7.3900	-2.1000	7.3900	2.1100	0.0000	0.0000	0.0000	0.0000;
	1	119	0.018542	0.119758	0.02285	221.1	0	0	0	0	1	0	0	12.2500	-0.5700	-12.2200	-1.6300	0.0000	0.0000	0.0000	0.0000;
	124	1	0.005615	0.036268	0.00692	221.1	0	0	0	0	1	0	0	40.1300	7.9400	-40.0400	-8.0800	0.0000	0.0000	0.0000	0.0000;
	193	1	0.004258	0.027501	0.00525	221.1	0	0	0	0	1	0	0	-20.3900	-6.9600	20.4100	6.5400	0.0000	0.0000	0.0000	0.0000;
	4	3	0.000573	0.003303	0	100	0	0	0	0	1	0	0	-1.7000	-0.4800	1.7000	0.4800	0.0000	0.0000	0.0000	0.0000;
	57	3	0.008505	0.058098	0.00988	221.1	0	0	0	0	1	0	0	-13.5600	-1.8900	13.5800	0.9400	0.0000	0.0000	0.0000	0.0000;
	3	137	0.005889	0.038035	0.00726	221.1	0	0	0	0	1	0	0	-15.2700	-1.4200	15.2900	0.7400	0.0000	0.0000	0.0000	0.0000;
	6	5	0.000694	0.003337	0	100	0	0	0	0	1	0	0	-7.9500	-2.2600	7.9500	2.2700	0.0000	0.0000	0.0000	0.0000;
	48	5	0.006246	0.043513	0.00711	221.1	0	0	0	0	1	0	0	-27.2200	-1.3800	27.2700	0.9200	0.0000	0.0000	0.0000	0.0000;
	5	64	0.007877	0.054875	0.00897	221.1	0	0	0	0	1	0	0	-35.2100	-3.1900	35.3000	2.8600	0.0000	0.0000	0.0000	0.0000;
	8	7	0.000634	0.003189	0	100	0	0	0	0	1	0	0	-23.7400	-6.7700	23.7400	6.7800	0.0000	0.0000	0.0000	0.0000;
	7	86	0.003756	0.024259	0.00463	221.1	0	0	0	0	1	0	0	-12.2900	-3.2900	12.2900	2.8600	0.0000	0.0000	0.0000	0.0000;
	7	101	0.003597	0.023232	0.00443	221.1	0	0	0	0	1	0	0	-24.5900	-6.1400	24.6100	5.8300	0.0000	0.0000	0.0000	0.0000;
	7	148	0.008667	0.05598	0.01068	221.1	0	0	0	0	1	0	0	13.1400	2.6500	-13.1200	-3.6500	0.0000	0.0000	0.0000	0.0000;
	10	9	0.000585	0.003967	0	100	0	0	0	0	1	0	0	-42.6800	-12.1600	42.6900	12.2400	0.0000	0.0000	0.0000	0.0000;
	9	124	0.008069	0.055123	0.00938	221.1	0	0	0	0	1	0	0	-39.5800	-9.8300	39.7000	9.7400	0.0000	0.0000	0.0000	0.0000;
	131	9	0.004139	0.026732	0.0051	221.1	0	0	0	0	1	0	0	-0.6800	0.5200	0.6800	-1.0400	0.0000	0.0000	0.0000	0.0000;
	9	141	0.004446	0.03097	0.00506	221.1	0	0	0	0	1	0	0	3.0500	0.6700	-3.0500	-1.1800	0.0000	0.0000	0.0000	0.0000;
	9	193	0.003695	0.025244	0.00429	221.1	0	0	0	0	1	0	0	-6.8500	-2.0300	6.8500	1.6000	0.0000	0.0000	0.0000	0.0000;
	12	11	0.000534	0.003533	0	100	0	0	0	0	1	0	0	-0.3000	-0.0900	0.3000	0.0900	0.0000	0.0000	0.0000	0.0000;
	13	11	0.000619	0.003606	0	100	0	0	0	0	1	0	0	-2.5200	-0.7200	2.5200	0.7200	0.0000	0.0000	0.0000	0.0000;
	11	15	0.010408	0.06722	0.01282	221.1	0	0	0	0	1	0	0	-14.5200	-2.5400	14.5400	1.2600	0.0000	0.0000	0.0000	0.0000;
	11	93	0.018394	0.128141	0.02095	221.1	0	0	0	0	1	0	0	1.4400	-1.6200	-1.4400	-0.6800	0.0000	0.0000	0.0000	0.0000;
	158	11	0.008519	0.058199	0.0099	221.1	0	0	0	0	1	0	0	-10.2400	-4.3700	10.2400	3.3500	0.0000	0.0000	0.0000	0.0000;
	15	14	0.000637	0.027526	0	300	0	0	1	0	1	0	0	-59.3100	18.9400	59.3300	-17.9700	0.0000	0.0000	0.0000	0.0000;
	14	121	0.007013	0.050098	0.09134	402.36	0	0	0	0	1	0	0	-135.7000	25.2800	136.9400	-26.3500	0.0000	0.0000	0.0000	0.0000;
	14	149	0.009133	0.065244	0.11895	402.36	0	0	0	0	1	0	0	76.3700	-7.3200	-75.8800	-2.1100	0.0000	0.0000	0.0000	0.0000;
	16	15	0.000553	0.0032	0	100	0	0	0	0	1	0	0	-44.7600	-12.7600	44.7700	12.8200	0.0000	0.0000	0.0000	0.0000;
	18	17	0.000503	0.003926	0	100	0	0	0	0	1	0	0	-0.7100	-0.2000	0.7100	0.2000	0.0000	0.0000	0.0000	0.0000;
	19	17	0.000514	0.003079	0	100	0	0	0	0	1	0	0	-0.7700	-0.2200	0.7700	0.2200	0.0000	0.0000	0.0000	0.0000;
	20	17	0.000623	0.003339	0	100	0	0	0	0	1	0	0	-1.1300	-0.3200	1.1300	0.3200	0.0000	0.0000	0.0000	0.0000;
	21	17	0.00062	0.00394	0	100	0	0	0	0	1	0	0	-4.6100	-1.3100	4.6100	1.3100	0.0000	0.0000	0.0000	0.0000;
	109	17	0.009789	0.068198	0.01115	221.1	0	0	0	0	1	0	0	31.7400	-1.9600	-31.6500	1.4400	0.0000	0.0000	0.0000	0.0000;
	17	120	0.010316	0.070473	0.01199	221.1	0	0	0	0	1	0	0	24.4200	-3.5000	-24.3600	2.6500	0.0000	0.0000	0.0000	0.0000;
	23	22	0.000616	0.003222	0	100	0	0	0	0	1	0	0	-1.4100	-0.4000	1.4100	0.4000	0.0000	0.0000	0.0000	0.0000;
	24	22	0.000687	0.003067	0	100	0	0	0	0	1	0	0	-2.4000	-0.6800	2.4000	0.6800	0.0000	0.0000	0.0000	0.0000;
	22	27	0.011379	0.073494	0.01402	221.1	0	0	0	0	1	0	0	-1.4100	-2.0700	1.4100	0.5400	0.0000	0.0000	0.0000	0.0000;
	158	22	0.012701	0.086769	0.01476	221.1	0	0	0	0	1	0	0	2.4000	-2.5900	-2.4000	0.9900	0.0000	0.0000	0.0000	0.0000;
	26	25	0.00061	0.003071	0	100	0	0	0	0	1	0	0	-1.3500	-0.3900	1.3500	0.3900	0.0000	0.0000	0.0000	0.0000;
	25	64	0.007643	0.053244	0.0087	221.1	0	0	0	0	1	0	0	-50.9200	9.4100	51.1100	-9.0400	0.0000	0.0000	0.0000	0.0000;
	199	25	0.009909	0.064003	0.01221	221.1	0	0	0	0	1	0	0	-49.3300	9.9600	49.5700	-9.8000	0.0000	0.0000	0.0000	0.0000;
	28	27	0.000587	0.003964	0	100	0	0	0	0	1	0	0	-2.3300	-0.6700	2.3300	0.6700	0.0000	0.0000	0.0000	0.0000;
	27	93	0.025172	0.175362	0.02867	221.1	0	0	0	0	1	0	0	-2.9600	-1.7500	2.9600	-1.3700	0.0000	0.0000	0.0000	0.0000;
	27	158	0.020429	0.131945	0.02517	221.1	0	0	0	0	1	0	0	-0.7800	0.5400	0.7800	-3.2900	0.0000	0.0000	0.0000	0.0000;
	30	29	0.000694	0.003131	0	100	0	0	0	0	1	0	0	-59.0900	-16.8400	59.1100	16.9500	0.0000	0.0000	0.0000	0.0000;
	124	29	0.005561	0.038742	0.00633	221.1	0	0	0	0	1	0	0	36.4400	10.6100	-36.3600	-10.7300	0.0000	0.0000	0.0000	0.0000;
	29	140	0.007895	0.055004	0.00899	221.1	0	0	0	0	1	0	0	-22.7500	-6.2200	22.7900	5.5800	0.0000	0.0000	0.0000	0.0000;
	32	31	0.000667	0.003884	0	100	0	0	0	0	1	0	0	-0.5100	-0.1500	0.5100	0.1500	0.0000	0.0000	0.0000	0.0000;
	33	31	0.000528	0.003639	0	100	0	0	0	0	1	0	0	-6.4400	-1.8400	6.4400	1.8400	0.0000	0.0000	0.0000	0.0000;
	177	31	0.008406	0.05856	0.00957	221.1	0	0	0	0	1	0	0	13.4300	8.7800	-13.4100	-9.6400	0.0000	0.0000	0.0000	0.0000;
	192	31	0.008088	0.05525	0.0094	221.1	0	0	0	0	1	0	0	-6.4500	-8.5800	6.4600	7.6600	0.0000	0.0000	0.0000	0.0000;
	35	34	0.000575	0.003674	0	100	0	0	0	0	1	0	0	-20.7600	-5.9200	20.7600	5.9300	0.0000	0.0000	0.0000	0.0000;
	34	54	0.006268	0.042819	0.00729	221.1	0	0	0	0	1	0	0	-37.2300	-6.2100	37.3100	6.0000	0.0000	0.0000	0.0000	0.0000;
	34	137	0.006911	0.044639	0.00852	221.1	0	0	0	0	1	0	0	16.4600	0.2700	-16.4500	-1.0700	0.0000	0.0000	0.0000	0.0000;
	37	36	0.000552	0.003328	0	100	0	0	0	0	1	0	0	-3.5700	-1.0200	3.5700	1.0200	0.0000	0.0000	0.0000	0.0000;
	38	36	0.00067	0.003511	0	100	0	0	0	0	1	0	0	-29.8100	-8.5000	29.8100	8.5300	0.0000	0.0000	0.0000	0.0000;
	58	36	0.010889	0.07439	0.01266	221.1	0	0	0	0	1	0	0	4.9900	8.1600	-4.9800	-9.4300	0.0000	0.0000	0.0000	0.0000;
	83	36	0.009615	0.066986	0.01095	221.1	0	0	0	0	1	0	0	28.4700	-0.5300	-28.4000	-0.1100	0.0000	0.0000	0.0000	0.0000;
	40	39	0.000505	0.003103	0	100	0	0	0	0	1	0	0	-2.0400	-0.5800	2.0400	0.5800	0.0000	0.0000	0.0000	0.0000;
	39	85	0.008702	0.060625	0.00991	221.1	0	0	0	0	1	0	0	-17.6700	2.9300	17.6900	-3.7800	0.0000	0.0000	0.0000	0.0000;
	159	39	0.007332	0.05108	0.00835	221.1	0	0	0	0	1	0	0	-8.2100	4.7900	8.2200	-5.6200	0.0000	0.0000	0.0000	0.0000;
	41	100	0.004595	0.029676	0.00566	221.1	0	0	0	0	1	0	0	-12.2600	-31.2300	12.3000	30.9000	0.0000	0.0000	0.0000	0.0000;
	41	163	0.003165	0.021623	0.00368	221.1	0	0	0	0	1	0	0	-0.7700	14.9900	0.7700	-15.3400	0.0000	0.0000	0.0000	0.0000;
	41	180	0.009408	0.060766	0.01159	221.1	0	0	0	0	1	0	0	-25.3100	5.3200	25.3700	-6.2100	0.0000	0.0000	0.0000	0.0000;
	44	42	0.012256	0.085383	0.01396	221.1	0	0	0	0	1	0	0	13.8900	-5.2100	-13.8600	3.9100	0.0000	0.0000	0.0000	0.0000;
	181	42	0.008008	0.05579	0.00912	221.1	0	0	0	0	1	0	0	-3.2400	5.9900	3.2500	-6.9300	0.0000	0.0000	0.0000	0.0000;
	43	84	0.007509	0.051295	0.00873	221.1	0	0	0	0	1	0	0	-20.0400	-1.4600	20.0700	0.7500	0.0000	0.0000	0.0000	0.0000;
	43	132	0.011052	0.075503	0.01285	221.1	0	0	0	0	1	0	0	17.9500	0.8600	-17.9200	-1.9500	0.0000	0.0000	0.0000	0.0000;
	44	200	0.014813	0.101195	0.01722	221.1	0	0	0	0	1	0	0	-18.0300	4.0300	18.0800	-5.5100	0.0000	0.0000	0.0000	0.0000;
	46	45	0.000936	0.049195	0	160	0	0	1	0	1	0	0	-31.3900	-0.4900	31.4000	0.9500	0.0000	0.0000	0.0000	0.0000;
	55	45	0.002593	0.018527	0.03378	402.36	0	0	0	0	1	0	0	-23.9400	-6.0600	23.9500	2.5200	0.0000	0.0000	0.0000	0.0000;
	102	45	0.0029	0.020715	0.03777	402.36	0	0	0	0	1	0	0	18.4200	-8.8200	-18.4100	4.8300	0.0000	0.0000	0.0000	0.0000;
	45	187	0.003474	0.024821	0.04525	402.36	0	0	0	0	1	0	0	-36.9400	-8.3000	36.9900	3.7300	0.0000	0.0000	0.0000	0.0000;
	61	46	0.006971	0.048561	0.00794	221.1	0	0	0	0	1	0	0	-8.8600	-4.8100	8.8700	4.0000	0.0000	0.0000	0.0000	0.0000;
	122	46	0.010129	0.065419	0.01248	221.1	0	0	0	0	1	0	0	-16.1500	4.1400	16.1800	-5.3100	0.0000	0.0000	0.0000	0.0000;
	47	54	0.006678	0.045623	0.00776	221.1	0	0	0	0	1	0	0	12.6200	1.7800	-12.6100	-2.5400	0.0000	0.0000	0.0000	0.0000;
	47	66	0.007339	0.047401	0.00904	221.1	0	0	0	0	1	0	0	-14.1700	-2.2200	14.1800	1.3300	0.0000	0.0000	0.0000	0.0000;
	49	48	0.027149	0.532021	0	28.9	0	0	1	0	1	0	0	1.3600	0.8800	-1.3600	-0.8700	0.0000	0.0000	0.0000	0.0000;
	50	48	0.021719	0.404167	0	28.7	0	0	1	0	1	0	0	1.3600	1.1800	-1.3600	-1.1700	0.0000	0.0000	0.0000	0.0000;
	51	48	0.027149	0.612711	0	24.3	0	0	1	0	1	0	0	1.3600	0.7700	-1.3600	-0.7500	0.0000	0.0000	0.0000	0.0000;
	52	48	0.021719	0.379817	0	24.3	0	0	1	0	1	0	0	1.3600	1.2500	-1.3600	-1.2400	0.0000	0.0000	0.0000	0.0000;
	53	48	0.02121	0.576549	0	35.3	0	0	1	0	1	0	0	2.7200	0.7900	-2.7200	-0.7500	0.0000	0.0000	0.0000	0.0000;
	48	74	0.007959	0.054369	0.00925	221.1	0	0	0	0	1	0	0	33.5600	5.6500	-33.4700	-6.0500	0.0000	0.0000	0.0000	0.0000;
	54	66	0.007051	0.045544	0.00869	221.1	0	0	0	0	1	0	0	-27.3600	-4.2100	27.4100	3.6000	0.0000	0.0000	0.0000	0.0000;
	56	55	0.005152	0.124653	0	160	0	0	1	0	1	0	0	-6.6000	-0.2300	6.6000	0.2800	0.0000	0.0000	0.0000	0.0000;
	81	55	0.005467	0.039054	0.0712	402.36	0	0	0	0	1	0	0	-74.8400	10.5800	75.1400	-16.1400	0.0000	0.0000	0.0000	0.0000;
	55	102	0.001246	0.008902	0.01623	402.36	0	0	0	0	1	0	0	-92.5200	6.4200	92.6200	-7.4500	0.0000	0.0000	0.0000	0.0000;
	55	112	0.004166	0.029763	0.05426	402.36	0	0	0	0	1	0	0	6.9100	7.2200	-6.9000	-12.9900	0.0000	0.0000	0.0000	0.0000;
	55	128	0.003415	0.024395	0.04448	402.36	0	0	0	0	1	0	0	27.8100	8.2900	-27.7800	-12.8500	0.0000	0.0000	0.0000	0.0000;
	56	103	0.004282	0.02983	0.00488	221.1	0	0	0	0	1	0	0	5.2900	-0.1500	-5.2900	-0.3700	0.0000	0.0000	0.0000	0.0000;
	57	159	0.00752	0.051369	0.00874	221.1	0	0	0	0	1	0	0	10.7400	1.0800	-10.7400	-1.9500	0.0000	0.0000	0.0000	0.0000;
	95	58	0.009836	0.063526	0.01212	221.1	0	0	0	0	1	0	0	-0.7300	16.4500	0.7500	-17.5900	0.0000	0.0000	0.0000	0.0000;
	177	58	0.014049	0.095972	0.01633	221.1	0	0	0	0	1	0	0	10.0000	-9.8000	-9.9800	8.2300	0.0000	0.0000	0.0000	0.0000;
	59	119	0.011209	0.072397	0.01381	221.1	0	0	0	0	1	0	0	5.3400	1.9700	-5.3300	-3.3800	0.0000	0.0000	0.0000	0.0000;
	59	139	0.009665	0.062426	0.01191	221.1	0	0	0	0	1	0	0	-10.0700	-3.3200	10.0800	2.1500	0.0000	0.0000	0.0000	0.0000;
	60	97	0.012274	0.079279	0.01512	221.1	0	0	0	0	1	0	0	34.9900	-2.5300	-34.8500	1.8500	0.0000	0.0000	0.0000	0.0000;
	134	60	0.009007	0.062749	0.01026	221.1	0	0	0	0	1	0	0	38.9200	-1.6400	-38.7900	1.4500	0.0000	0.0000	0.0000	0.0000;
	61	103	0.006539	0.045556	0.00745	221.1	0	0	0	0	1	0	0	-11.6900	-1.0500	11.7000	0.3100	0.0000	0.0000	0.0000	0.0000;
	159	62	0.008634	0.058979	0.01003	221.1	0	0	0	0	1	0	0	17.5600	-3.2300	-17.5300	2.3500	0.0000	0.0000	0.0000	0.0000;
	160	62	0.007628	0.053144	0.00869	221.1	0	0	0	0	1	0	0	3.8800	7.5700	-3.8700	-8.4500	0.0000	0.0000	0.0000	0.0000;
	63	66	0.011949	0.083244	0.01361	221.1	0	0	0	0	1	0	0	-8.1500	3.8500	8.1600	-5.2600	0.0000	0.0000	0.0000	0.0000;
	63	184	0.017963	0.116023	0.02214	221.1	0	0	0	0	1	0	0	0.1700	-6.1300	-0.1600	3.7300	0.0000	0.0000	0.0000	0.0000;
	65	64	0.00128	0.10157	0	195.5	0	0	1	0	1	0	0	86.5000	0.8400	-86.4100	6.1800	0.0000	0.0000	0.0000	0.0000;
	67	66	0.020951	0.144786	0	30	0	0	1	0	1	0	0	1.4100	-0.5700	-1.4100	0.5700	0.0000	0.0000	0.0000	0.0000;
	68	66	0.00689	0.264997	0	73	0	0	1	0	1	0	0	8.3800	-0.5500	-8.3800	0.7200	0.0000	0.0000	0.0000	0.0000;
	69	66	0.00441	0.09499	0	70.5	0	0	1	0	1	0	0	8.3800	-1.5300	-8.3800	1.5900	0.0000	0.0000	0.0000	0.0000;
	70	66	0.00689	0.199449	0	70.5	0	0	1	0	1	0	0	8.3800	-0.7800	-8.3800	0.9100	0.0000	0.0000	0.0000	0.0000;
	71	66	0.003528	0.081544	0	68	0	0	1	0	1	0	0	8.3800	-1.7000	-8.3800	1.7600	0.0000	0.0000	0.0000	0.0000;
	72	66	0.003528	0.074599	0	68	0	0	1	0	1	0	0	8.3800	-1.8600	-8.3800	1.9100	0.0000	0.0000	0.0000	0.0000;
	73	66	0.00689	0.146423	0	65	0	0	1	0	1	0	0	8.3800	-1.1100	-8.3800	1.2000	0.0000	0.0000	0.0000	0.0000;
	66	158	0.004993	0.032251	0.00615	221.1	0	0	0	0	1	0	0	1.9200	-8.3400	-1.9200	7.6900	0.0000	0.0000	0.0000	0.0000;
	74	190	0.008894	0.06076	0.01034	221.1	0	0	0	0	1	0	0	14.7100	0.7000	-14.6900	-1.6700	0.0000	0.0000	0.0000	0.0000;
	76	75	0.038464	0.784328	0	24.3	0	0	1	0	1	0	0	1.2000	1.4100	-1.2000	-1.3800	0.0000	0.0000	0.0000	0.0000;
	77	75	0.08013	2.192105	0	7.4	0	0	1	0	1	0	0	0.7200	0.5000	-0.7200	-0.4900	0.0000	0.0000	0.0000	0.0000;
	78	75	0.008544	0.181406	0	65	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	79	75	0.01068	0.277774	0	62.1	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	108	75	0.010988	0.070969	0.01354	221.1	0	0	0	0	1	0	0	-12.6100	-2.5700	12.6300	1.2500	0.0000	0.0000	0.0000	0.0000;
	75	157	0.013487	0.087112	0.01662	221.1	0	0	0	0	1	0	0	-14.3100	-0.4000	14.3400	-1.1900	0.0000	0.0000	0.0000	0.0000;
	80	100	0.021463	0.149525	0.02444	221.1	0	0	0	0	1	0	0	6.6400	-7.8800	-6.6200	5.2900	0.0000	0.0000	0.0000	0.0000;
	80	143	0.020594	0.14347	0.02345	221.1	0	0	0	0	1	0	0	-8.1100	7.4600	8.1400	-9.8200	0.0000	0.0000	0.0000	0.0000;
	82	81	0.00097	0.042371	0	250	0	0	1	0	1	0	0	-36.8000	-5.8400	36.8100	6.3800	0.0000	0.0000	0.0000	0.0000;
	81	178	0.006598	0.047137	0.08594	402.36	0	0	0	0	1	0	0	38.0300	-16.9700	-37.9300	8.3700	0.0000	0.0000	0.0000	0.0000;
	82	195	0.003845	0.026265	0.00447	221.1	0	0	0	0	1	0	0	29.2000	3.6700	-29.1700	-3.9300	0.0000	0.0000	0.0000	0.0000;
	83	146	0.005382	0.036767	0.00626	221.1	0	0	0	0	1	0	0	-57.1800	1.5200	57.3500	-1.0600	0.0000	0.0000	0.0000	0.0000;
	83	186	0.00792	0.055174	0.00902	221.1	0	0	0	0	1	0	0	25.5700	-1.8800	-25.5200	1.2700	0.0000	0.0000	0.0000	0.0000;
	84	113	0.006652	0.04544	0.00773	221.1	0	0	0	0	1	0	0	-23.8200	-1.8200	23.8600	1.2700	0.0000	0.0000	0.0000	0.0000;
	85	120	0.007786	0.050291	0.00959	221.1	0	0	0	0	1	0	0	-22.5300	2.4000	22.5700	-3.1600	0.0000	0.0000	0.0000	0.0000;
	86	101	0.00512	0.033072	0.00631	221.1	0	0	0	0	1	0	0	-8.2600	-2.2600	8.2700	1.6400	0.0000	0.0000	0.0000	0.0000;
	142	86	0.003387	0.023141	0.00394	221.1	0	0	0	0	1	0	0	1.0500	-1.3100	-1.0500	0.9100	0.0000	0.0000	0.0000	0.0000;
	86	193	0.006606	0.042664	0.00814	221.1	0	0	0	0	1	0	0	-11.2300	-3.8500	11.2400	3.0700	0.0000	0.0000	0.0000	0.0000;
	88	87	0.000832	0.034189	0	350	0	0	1	0	1	0	0	-65.6700	-2.1600	65.7100	3.5300	0.0000	0.0000	0.0000	0.0000;
	149	87	0.001621	0.011579	0.02111	402.36	0	0	0	0	1	0	0	65.7700	1.7200	-65.7100	-3.5300	0.0000	0.0000	0.0000	0.0000;
	88	150	0.00557	0.035973	0.00686	221.1	0	0	0	0	1	0	0	-25.5300	-4.6400	25.5700	4.1300	0.0000	0.0000	0.0000	0.0000;
	176	88	0.003859	0.024922	0.00475	221.1	0	0	0	0	1	0	0	-29.0800	-6.9300	29.1100	6.6300	0.0000	0.0000	0.0000	0.0000;
	88	194	0.004258	0.029088	0.00495	221.1	0	0	0	0	1	0	0	12.7600	-13.9000	-12.7400	13.4600	0.0000	0.0000	0.0000	0.0000;
	90	89	0.0601	2.279922	0	8.6	0	0	1	0	1	0	0	0.9600	0.1300	-0.9600	-0.1100	0.0000	0.0000	0.0000	0.0000;
	91	89	0.024614	0.503424	0	32	0	0	1	0	1	0	0	1.5000	0.5900	-1.5000	-0.5800	0.0000	0.0000	0.0000	0.0000;
	92	89	0.019539	0.36204	0	35.3	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	89	95	0.008734	0.056409	0.01076	221.1	0	0	0	0	1	0	0	2.2900	-16.5000	-2.2700	15.4700	0.0000	0.0000	0.0000	0.0000;
	188	89	0.00805	0.051996	0.00992	221.1	0	0	0	0	1	0	0	62.9700	1.5400	-62.6800	-0.7100	0.0000	0.0000	0.0000	0.0000;
	94	93	0.01068	0.280602	0	62.1	0	0	1	0	1	0	0	5.4000	-2.2000	-5.4000	2.2900	0.0000	0.0000	0.0000	0.0000;
	93	191	0.022088	0.142662	0.02722	221.1	0	0	0	0	1	0	0	0.8200	-1.1100	-0.8200	-1.8700	0.0000	0.0000	0.0000	0.0000;
	122	96	0.01062	0.073986	0.01209	221.1	0	0	0	0	1	0	0	28.5100	-5.5400	-28.4200	4.8000	0.0000	0.0000	0.0000	0.0000;
	96	143	0.015858	0.102423	0.01954	221.1	0	0	0	0	1	0	0	35.1000	-3.1300	-34.9200	2.1900	0.0000	0.0000	0.0000	0.0000;
	96	188	0.010903	0.070421	0.01343	221.1	0	0	0	0	1	0	0	-9.6300	-2.5200	9.6400	1.1300	0.0000	0.0000	0.0000	0.0000;
	97	200	0.01318	0.085125	0.01624	221.1	0	0	0	0	1	0	0	27.5200	-3.9400	-27.4300	2.8500	0.0000	0.0000	0.0000	0.0000;
	99	98	0.000643	0.030516	0	250	0	0	1	0	1	0	0	-50.5900	-12.6800	50.6100	13.4700	0.0000	0.0000	0.0000	0.0000;
	98	123	0.0047	0.033574	0.06121	402.36	0	0	0	0	1	0	0	-50.6100	-13.4700	50.7200	7.8400	0.0000	0.0000	0.0000	0.0000;
	99	142	0.007605	0.052982	0.00866	221.1	0	0	0	0	1	0	0	38.4200	9.2100	-38.3100	-9.3100	0.0000	0.0000	0.0000	0.0000;
	100	174	0.010698	0.074531	0.01218	221.1	0	0	0	0	1	0	0	-20.6500	12.4200	20.7100	-13.3700	0.0000	0.0000	0.0000	0.0000;
	100	179	0.007093	0.049413	0.00808	221.1	0	0	0	0	1	0	0	-32.9600	24.5800	33.0700	-24.7100	0.0000	0.0000	0.0000	0.0000;
	100	184	0.012745	0.08879	0.01451	221.1	0	0	0	0	1	0	0	10.5500	5.2900	-10.5300	-6.7800	0.0000	0.0000	0.0000	0.0000;
	110	101	0.00196	0.012657	0.00241	221.1	0	0	0	0	1	0	0	-1.0100	0.5000	1.0100	-0.7500	0.0000	0.0000	0.0000	0.0000;
	101	117	0.005245	0.035833	0.0061	221.1	0	0	0	0	1	0	0	-37.4300	-6.1200	37.5000	5.9900	0.0000	0.0000	0.0000	0.0000;
	101	141	0.003144	0.020305	0.00387	221.1	0	0	0	0	1	0	0	3.0200	-0.7500	-3.0200	0.3500	0.0000	0.0000	0.0000	0.0000;
	103	102	0.001709	0.054578	0	250	0	0	1	0	1	0	0	-33.2200	-1.3300	33.2400	1.9000	0.0000	0.0000	0.0000	0.0000;
	104	102	0.0021	0.087749	0	130	0	0	1	0	1	0	0	67.6000	3.1000	-67.5100	0.6200	0.0000	0.0000	0.0000	0.0000;
	105	102	0.00097	0.059419	0	260	0	0	1	0	1	0	0	154.8000	8.2900	-154.5800	4.9100	0.0000	0.0000	0.0000	0.0000;
	102	128	0.0027	0.019292	0.03517	402.36	0	0	0	0	1	0	0	77.8200	8.8500	-77.6600	-11.5200	0.0000	0.0000	0.0000	0.0000;
	103	106	0.007749	0.053987	0.00883	221.1	0	0	0	0	1	0	0	26.8200	1.3900	-26.7600	-1.9700	0.0000	0.0000	0.0000	0.0000;
	130	106	0.006991	0.047761	0.00813	221.1	0	0	0	0	1	0	0	-24.6700	-1.9800	24.7100	1.3900	0.0000	0.0000	0.0000	0.0000;
	107	113	0.006748	0.046101	0.00784	221.1	0	0	0	0	1	0	0	-26.6500	-7.5000	26.7000	7.0300	0.0000	0.0000	0.0000	0.0000;
	107	129	0.005168	0.035307	0.00601	221.1	0	0	0	0	1	0	0	-32.3600	-5.4300	32.4100	5.1700	0.0000	0.0000	0.0000	0.0000;
	192	107	0.003939	0.025441	0.00485	221.1	0	0	0	0	1	0	0	-6.4500	1.5600	6.4500	-2.0500	0.0000	0.0000	0.0000	0.0000;
	108	198	0.024792	0.160126	0.03055	221.1	0	0	0	0	1	0	0	3.1900	-0.1200	-3.1900	-3.0700	0.0000	0.0000	0.0000	0.0000;
	186	109	0.008774	0.059942	0.0102	221.1	0	0	0	0	1	0	0	37.0100	-0.7900	-36.9000	0.4900	0.0000	0.0000	0.0000	0.0000;
	110	193	0.003444	0.022243	0.00424	221.1	0	0	0	0	1	0	0	-9.8400	-3.5900	9.8500	3.1800	0.0000	0.0000	0.0000	0.0000;
	163	111	0.005281	0.034111	0.00651	221.1	0	0	0	0	1	0	0	8.1600	1.0400	-8.1600	-1.7200	0.0000	0.0000	0.0000	0.0000;
	111	175	0.00795	0.055386	0.00905	221.1	0	0	0	0	1	0	0	-0.6600	-0.7900	0.6600	-0.1900	0.0000	0.0000	0.0000	0.0000;
	113	112	0.002077	0.058615	0	350	0	0	1	0	1	0	0	-90.4800	-16.1700	90.6500	20.9200	0.0000	0.0000	0.0000	0.0000;
	114	112	0.06	0.434013	0	20	0	0	1	0	1	0	0	1.4000	0.3600	-1.4000	-0.3500	0.0000	0.0000	0.0000	0.0000;
	115	112	0.00128	0.062771	0	200	0	0	1	0	1	0	0	133.5000	12.6000	-133.2900	-2.1600	0.0000	0.0000	0.0000	0.0000;
	128	112	0.001295	0.009255	0.01687	402.36	0	0	0	0	1	0	0	-50.9000	3.8300	50.9400	-5.4100	0.0000	0.0000	0.0000	0.0000;
	113	192	0.005418	0.034995	0.00668	221.1	0	0	0	0	1	0	0	39.9300	7.8800	-39.8400	-8.0200	0.0000	0.0000	0.0000	0.0000;
	117	116	0.000517	0.034943	0	300	0	0	1	0	1	0	0	-70.9800	-17.5200	71.0100	19.3200	0.0000	0.0000	0.0000	0.0000;
	133	116	0.00389	0.027793	0.05067	402.36	0	0	0	0	1	0	0	71.2000	15.3500	-71.0100	-19.3200	0.0000	0.0000	0.0000	0.0000;
	117	132	0.011912	0.082984	0.01357	221.1	0	0	0	0	1	0	0	9.0500	4.4200	-9.0400	-5.7300	0.0000	0.0000	0.0000	0.0000;
	117	162	0.01617	0.104442	0.01993	221.1	0	0	0	0	1	0	0	-9.6500	-2.5900	9.6700	0.6100	0.0000	0.0000	0.0000	0.0000;
	131	118	0.00718	0.046377	0.00885	221.1	0	0	0	0	1	0	0	-29.5700	-9.1400	29.6400	8.6500	0.0000	0.0000	0.0000	0.0000;
	118	134	0.011598	0.0808	0.01321	221.1	0	0	0	0	1	0	0	-35.7600	-7.0800	35.9100	6.7100	0.0000	0.0000	0.0000	0.0000;
	118	198	0.009487	0.064809	0.01103	221.1	0	0	0	0	1	0	0	2.1800	-2.6900	-2.1800	1.5400	0.0000	0.0000	0.0000	0.0000;
	122	121	0.000786	0.063023	0	200	0	0	1	0	1	0	0	-27.7900	-3.0000	27.8000	3.4500	0.0000	0.0000	0.0000	0.0000;
	121	178	0.008903	0.063604	0.11596	402.36	0	0	0	0	1	0	0	68.4200	-14.6400	-68.0300	4.8500	0.0000	0.0000	0.0000	0.0000;
	187	121	0.001063	0.007593	0.01384	402.36	0	0	0	0	1	0	0	233.7000	-35.1200	-233.1500	37.5300	0.0000	0.0000	0.0000	0.0000;
	124	123	0.000329	0.017481	0	400	0	0	1	0	1	0	0	-160.2500	-39.2400	160.3400	43.7800	0.0000	0.0000	0.0000	0.0000;
	125	123	0.00448	0.104562	0	170	0	0	1	0	1	0	0	39.0200	7.2400	-38.9500	-5.7200	0.0000	0.0000	0.0000	0.0000;
	126	123	0.00148	0.066043	0	170	0	0	1	0	1	0	0	39.0200	12.5500	-39.0000	-11.5200	0.0000	0.0000	0.0000	0.0000;
	127	123	0.001	0.031691	0	170	0	0	1	0	1	0	0	39.0200	25.9900	-39.0000	-25.3500	0.0000	0.0000	0.0000	0.0000;
	123	133	0.001459	0.010423	0.019	402.36	0	0	0	0	1	0	0	-76.3800	-6.4500	76.4600	4.9900	0.0000	0.0000	0.0000	0.0000;
	156	123	0.001117	0.007981	0.01455	402.36	0	0	0	0	1	0	0	17.7300	1.0600	-17.7200	-2.5800	0.0000	0.0000	0.0000	0.0000;
	124	193	0.006577	0.04582	0.00749	221.1	0	0	0	0	1	0	0	43.9700	10.9600	-43.8400	-10.8400	0.0000	0.0000	0.0000	0.0000;
	129	128	0.000753	0.044453	0	250	0	0	1	0	1	0	0	-109.6600	-27.1900	109.7500	32.6400	0.0000	0.0000	0.0000	0.0000;
	128	133	0.009554	0.068257	0.12444	402.36	0	0	0	0	1	0	0	46.6000	-12.1000	-46.4000	0.2100	0.0000	0.0000	0.0000	0.0000;
	130	144	0.022678	0.157988	0.02583	221.1	0	0	0	0	1	0	0	20.2700	0.7200	-20.1800	-2.8300	0.0000	0.0000	0.0000	0.0000;
	134	133	0.0002	0.013328	0	400	0	0	1	0	1	0	0	-119.5400	-11.7300	119.5600	13.5400	0.0000	0.0000	0.0000	0.0000;
	135	133	0.00043	0.033512	0	580	0	0	1	0	1	0	0	133.9200	21.4600	-133.8500	-15.7600	0.0000	0.0000	0.0000	0.0000;
	136	133	0.00043	0.027478	0	580	0	0	1	0	1	0	0	133.9200	25.0600	-133.8500	-20.3400	0.0000	0.0000	0.0000	0.0000;
	156	133	0.001958	0.013985	0.0255	402.36	0	0	0	0	1	0	0	-46.8200	-4.4400	46.8700	2.0100	0.0000	0.0000	0.0000	0.0000;
	134	140	0.010445	0.071357	0.01214	221.1	0	0	0	0	1	0	0	31.1000	7.3300	-31.0000	-7.9200	0.0000	0.0000	0.0000	0.0000;
	134	186	0.014463	0.093412	0.01782	221.1	0	0	0	0	1	0	0	13.6100	-0.6600	-13.5900	-1.0700	0.0000	0.0000	0.0000	0.0000;
	138	139	0.026477	0.184455	0.03015	221.1	0	0	0	0	1	0	0	16.9900	1.4400	-16.9200	-4.1000	0.0000	0.0000	0.0000	0.0000;
	138	195	0.008728	0.05637	0.01075	221.1	0	0	0	0	1	0	0	-19.0900	-2.0300	19.1200	1.0800	0.0000	0.0000	0.0000	0.0000;
	141	148	0.01019	0.065817	0.01256	221.1	0	0	0	0	1	0	0	18.9400	4.4400	-18.9000	-5.4800	0.0000	0.0000	0.0000	0.0000;
	141	193	0.002896	0.018702	0.00357	221.1	0	0	0	0	1	0	0	-14.3000	-4.0100	14.3100	3.6800	0.0000	0.0000	0.0000	0.0000;
	162	144	0.006962	0.048504	0.00793	221.1	0	0	0	0	1	0	0	-11.1200	-1.0300	11.1200	0.2500	0.0000	0.0000	0.0000	0.0000;
	145	176	0.008799	0.060111	0.01023	221.1	0	0	0	0	1	0	0	-10.0600	-2.5500	10.0700	1.5200	0.0000	0.0000	0.0000	0.0000;
	185	145	0.010839	0.075509	0.01234	221.1	0	0	0	0	1	0	0	7.4900	1.1800	-7.4900	-2.4500	0.0000	0.0000	0.0000	0.0000;
	147	146	0.0025	0.10181	0	130.6	0	0	1	0	1	0	0	92.4000	8.7900	-92.2000	-0.6900	0.0000	0.0000	0.0000	0.0000;
	146	177	0.011064	0.07708	0.0126	221.1	0	0	0	0	1	0	0	34.8500	1.7400	-34.7200	-2.2000	0.0000	0.0000	0.0000	0.0000;
	150	149	0.000412	0.020251	0	400	0	0	1	0	1	0	0	-104.3000	3.7000	104.3400	-1.6600	0.0000	0.0000	0.0000	0.0000;
	151	149	0.02267	0.625521	0	20	0	0	1	0	1	0	0	1.6200	0.0200	-1.6200	-0.0100	0.0000	0.0000	0.0000	0.0000;
	152	149	0.00249	0.141687	0	100	0	0	1	0	1	0	0	23.1700	0.2700	-23.1600	0.4300	0.0000	0.0000	0.0000	0.0000;
	153	149	0.00549	0.184603	0	100	0	0	1	0	1	0	0	23.1700	0.0200	-23.1400	0.9000	0.0000	0.0000	0.0000	0.0000;
	154	149	0.00249	0.10294	0	100	0	0	1	0	1	0	0	23.1700	0.1400	-23.1600	0.3700	0.0000	0.0000	0.0000	0.0000;
	155	149	0.00249	0.099835	0	100	0	0	1	0	1	0	0	23.1700	0.1300	-23.1600	0.3600	0.0000	0.0000	0.0000	0.0000;
	185	150	0.013458	0.091934	0.01564	221.1	0	0	0	0	1	0	0	-18.2800	-4.2500	18.3200	2.8700	0.0000	0.0000	0.0000	0.0000;
	194	150	0.003138	0.021435	0.00365	221.1	0	0	0	0	1	0	0	-60.3000	11.0400	60.4100	-10.6900	0.0000	0.0000	0.0000	0.0000;
	157	156	0.000592	0.045654	0	300	0	0	1	0	1	0	0	-29.0900	-3.0100	29.1000	3.3800	0.0000	0.0000	0.0000	0.0000;
	161	160	0.00139	0.080288	0	180.2	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	160	181	0.004296	0.029345	0.00499	221.1	0	0	0	0	1	0	0	-3.8800	-7.5700	3.8800	7.0600	0.0000	0.0000	0.0000	0.0000;
	164	163	0.008207	0.11301	0	58.1	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	165	163	0.0074	0.301657	0	65	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	166	163	0.016368	0.382075	0	42	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	167	163	0.010476	0.196499	0	42	0	0	1	0	1	0	0	2.8200	-1.0400	-2.8200	1.0600	0.0000	0.0000	0.0000	0.0000;
	168	163	0.018	0.490983	0	40	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	169	163	0.016368	0.396417	0	40	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	170	163	0.016368	0.374902	0	35.6	0	0	1	0	1	0	0	2.8200	-0.9900	-2.8200	1.0200	0.0000	0.0000	0.0000	0.0000;
	163	179	0.00579	0.039557	0.00673	221.1	0	0	0	0	1	0	0	-49.1000	-0.8200	49.2300	0.9600	0.0000	0.0000	0.0000	0.0000;
	171	190	0.009866	0.0674	0.01147	221.1	0	0	0	0	1	0	0	2.1000	1.9100	-2.1000	-3.1200	0.0000	0.0000	0.0000	0.0000;
	195	171	0.011644	0.081116	0.01326	221.1	0	0	0	0	1	0	0	10.0500	2.8500	-10.0400	-4.1700	0.0000	0.0000	0.0000	0.0000;
	180	172	0.015296	0.106558	0.01742	221.1	0	0	0	0	1	0	0	-6.7000	0.5700	6.7100	-2.4300	0.0000	0.0000	0.0000	0.0000;
	199	172	0.024005	0.155042	0.02958	221.1	0	0	0	0	1	0	0	12.1200	-3.9100	-12.0900	0.9000	0.0000	0.0000	0.0000	0.0000;
	173	174	0.015373	0.099293	0.01894	221.1	0	0	0	0	1	0	0	-16.6800	-3.2600	16.7200	1.4400	0.0000	0.0000	0.0000	0.0000;
	173	175	0.008572	0.055363	0.01056	221.1	0	0	0	0	1	0	0	7.9700	0.7700	-7.9700	-1.8900	0.0000	0.0000	0.0000	0.0000;
	174	188	0.017952	0.12264	0.02087	221.1	0	0	0	0	1	0	0	-40.2000	11.1400	40.4900	-11.4500	0.0000	0.0000	0.0000	0.0000;
	179	178	0.000495	0.036157	0	250	0	0	1	0	1	0	0	-105.9100	17.0200	105.9600	-13.2200	0.0000	0.0000	0.0000	0.0000;
	180	191	0.026393	0.183866	0.03006	221.1	0	0	0	0	1	0	0	6.4200	-3.0200	-6.4100	-0.1900	0.0000	0.0000	0.0000	0.0000;
	180	199	0.012185	0.078698	0.01501	221.1	0	0	0	0	1	0	0	-32.8300	6.4500	32.9600	-7.2700	0.0000	0.0000	0.0000	0.0000;
	182	181	0.01099	0.297746	0	60.3	0	0	1	0	1	0	0	5.2500	2.1500	-5.2500	-2.0600	0.0000	0.0000	0.0000	0.0000;
	183	181	0.004627	0.114254	0	65	0	0	1	0	1	0	0	7.9800	5.7300	-7.9800	-5.6200	0.0000	0.0000	0.0000	0.0000;
	181	194	0.004395	0.028384	0.00542	221.1	0	0	0	0	1	0	0	-35.4000	-19.0400	35.4700	18.8800	0.0000	0.0000	0.0000	0.0000;
	188	187	0.001394	0.044762	0	400	0	0	1	0	1	0	0	-113.1000	8.7800	113.2700	-3.4800	0.0000	0.0000	0.0000	0.0000;
	189	187	0.0003	0.007815	0	740	0	0	1	0	1	0	0	384.3700	-24.1500	-383.9600	34.8700	0.0000	0.0000	0.0000	0.0000;
	196	195	0.00285	0.060038	0	87.7	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
	197	195	0.00285	0.191484	0	87.7	0	0	1	0	1	0	0	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000	0.0000;
];

%%-----  OPF Data  -----%%
%% generator cost data
%	1	startup	shutdown	n	x1	y1	...	xn	yn
%	2	startup	shutdown	n	c(n-1)	...	c0
mpc.gencost = [
	2	0	0	3	0.002	19	236.12;
	2	0	0	3	0.002	19	236.12;
	2	0	0	3	0.002	19	236.12;
	2	0	0	3	0.002	19	236.12;
	2	0	0	3	0.002	19	236.24;
	2	0	0	3	0	0	0;
	2	0	0	3	0.002	19	236.13;
	2	0	0	3	0.002	19	236.74;
	2	0	0	3	0.002	19	236.74;
	2	0	0	3	0.002	19	236.74;
	2	0	0	3	0.002	19	236.74;
	2	0	0	3	0.002	19	236.74;
	2	0	0	3	0.002	19	236.74;
	2	0	0	3	0.002	23.157	606;
	2	0	0	3	0.002	23.146	603.6;
	2	0	0	3	0.002	23.252	627;
	2	0	0	3	0.002	23.252	627;
	2	0	0	3	0.002	23.152	604.8;
	2	0	0	3	0.002	23.164	607.5;
	2	0	0	3	0.002	23.173	609.45;
	2	0	0	3	0.002	19	236.48;
	2	0	0	3	0	0	0;
	2	0	0	3	0	0	0;
	2	0	0	3	0	0	0;
	2	0	0	3	0	0	0;
	2	0	0	3	0.002	19	610.14;
	2	0	0	3	0.002	19	610.14;
	2	0	0	3	0.002	19	610.14;
	2	0	0	3	0.002	19	1261.33;
	2	0	0	3	0.002	19	1261.33;
	2	0	0	3	0	0	0;
	2	0	0	3	0.002	19	236.14;
	2	0	0	3	0.002	19	253.01;
	2	0	0	3	0.002	19	253.01;
	2	0	0	3	0.002	19	253.01;
	2	0	0	3	0.002	19	253.01;
	2	0	0	3	0.002	24.07	807.9;
	2	0	0	3	0.002	23.211	618;
	2	0	0	3	0.002	23.306	639;
	2	0	0	3	0.002	23.194	614.1;
	2	0	0	3	0.002	23.194	614.1;
	2	0	0	3	0.002	23.194	614.1;
	2	0	0	3	0.002	23.194	614.1;
	2	0	0	3	0.002	23.194	614.1;
	2	0	0	3	0.002	19	236.47;
	2	0	0	3	0.002	19	236.71;
	2	0	0	3	0	6.71	1272.13;
	2	0	0	3	0.002	23.588	701.25;
	2	0	0	3	0.002	23.588	701.25;
];

%% generator unit type (see GENTYPES)
mpc.gentype = {
	'ST';
	'ST';
	'ST';
	'ST';
	'ST';
	'W2';
	'ST';
	'ST';
	'ST';
	'ST';
	'ST';
	'ST';
	'ST';
	'GT';
	'GT';
	'GT';
	'GT';
	'GT';
	'GT';
	'GT';
	'ST';
	'W2';
	'W2';
	'W2';
	'W2';
	'ST';
	'ST';
	'ST';
	'ST';
	'ST';
	'W2';
	'ST';
	'ST';
	'ST';
	'ST';
	'ST';
	'GT';
	'GT';
	'GT';
	'GT';
	'GT';
	'GT';
	'GT';
	'GT';
	'ST';
	'ST';
	'NB';
	'GT';
	'GT';
};

%% generator fuel type (see GENFUELS)
mpc.genfuel = {
	'coal';
	'coal';
	'coal';
	'coal';
	'coal';
	'wind';
	'coal';
	'coal';
	'coal';
	'coal';
	'coal';
	'coal';
	'coal';
	'ng';
	'ng';
	'ng';
	'ng';
	'ng';
	'ng';
	'ng';
	'coal';
	'wind';
	'wind';
	'wind';
	'wind';
	'coal';
	'coal';
	'coal';
	'coal';
	'coal';
	'wind';
	'coal';
	'coal';
	'coal';
	'coal';
	'coal';
	'ng';
	'ng';
	'ng';
	'ng';
	'ng';
	'ng';
	'ng';
	'ng';
	'coal';
	'coal';
	'nuclear';
	'ng';
	'ng';
};

%% bus names
mpc.bus_name = {
	'CREVE COEUR 0';
	'CREVE COEUR 1';
	'ILLIOPOLIS 0';
	'ILLIOPOLIS 1';
	'PAXTON 2 0';
	'PAXTON 2 1';
	'PEORIA 9 0';
	'PEORIA 9 1';
	'PEORIA 8 0';
	'PEORIA 8 1';
	'LOVINGTON 0';
	'LOVINGTON 1';
	'LOVINGTON 2';
	'DECATUR 3 0';
	'DECATUR 3 1';
	'DECATUR 3 2';
	'MASON CITY 0';
	'MASON CITY 1';
	'MASON CITY 2';
	'MASON CITY 3';
	'MASON CITY 4';
	'BETHANY 0';
	'BETHANY 1';
	'BETHANY 2';
	'RANKIN 0';
	'RANKIN 1';
	'WINDSOR 0';
	'WINDSOR 1';
	'PEKIN 2 0';
	'PEKIN 2 1';
	'HEYWORTH 0';
	'HEYWORTH 1';
	'HEYWORTH 2';
	'DECATUR 2 0';
	'DECATUR 2 1';
	'LINCOLN 0';
	'LINCOLN 1';
	'LINCOLN 2';
	'SHERMAN 0';
	'SHERMAN 1';
	'URBANA 2 0';
	'SPRINGFIELD 8 0';
	'CARLOCK 0';
	'PLEASANT PLAINS 0';
	'LE ROY 0';
	'LE ROY 1';
	'DECATUR 1 0';
	'RANTOUL 2 0';
	'RANTOUL 2 1';
	'RANTOUL 2 2';
	'RANTOUL 2 3';
	'RANTOUL 2 4';
	'RANTOUL 2 5';
	'MACON 0';
	'ELLSWORTH 2 0';
	'ELLSWORTH 2 1';
	'MOUNT PULASKI 0';
	'WAPELLA 0';
	'ROANOKE 0';
	'GREEN VALLEY 0';
	'BLOOMINGTON 3 0';
	'SPRINGFIELD 7 0';
	'BEMENT 0';
	'PAXTON 1 0';
	'PAXTON 1 1';
	'MOUNT ZION 0';
	'MOUNT ZION 1';
	'MOUNT ZION 2';
	'MOUNT ZION 3';
	'MOUNT ZION 4';
	'MOUNT ZION 5';
	'MOUNT ZION 6';
	'MOUNT ZION 7';
	'RANTOUL 1 0';
	'BRIMFIELD 0';
	'BRIMFIELD 1';
	'BRIMFIELD 2';
	'BRIMFIELD 3';
	'BRIMFIELD 4';
	'WELDON 0';
	'GIBSON CITY 2 0';
	'GIBSON CITY 2 1';
	'MINIER 0';
	'HUDSON 0';
	'ATHENS 0';
	'PEORIA HEIGHTS 0';
	'SPRINGFIELD 6 0';
	'SPRINGFIELD 6 1';
	'CLINTON 3 0';
	'CLINTON 3 1';
	'CLINTON 3 2';
	'CLINTON 3 3';
	'TUSCOLA 2 0';
	'TUSCOLA 2 1';
	'KENNEY 0';
	'MANSFIELD 0';
	'MANITO 0';
	'DUNLAP 0';
	'DUNLAP 1';
	'CHAMPAIGN 3 0';
	'PEORIA 7 0';
	'ELLSWORTH 1 0';
	'ELLSWORTH 1 1';
	'ELLSWORTH 1 2';
	'ELLSWORTH 1 3';
	'TOWANDA 0';
	'BLOOMINGTON 2 0';
	'PRINCEVILLE 0';
	'DELAVAN 0';
	'PEORIA 6 0';
	'SAVOY 0';
	'NORMAL 2 0';
	'NORMAL 2 1';
	'NORMAL 2 2';
	'NORMAL 2 3';
	'EAST PEORIA 0';
	'EAST PEORIA 1';
	'HANNA CITY 0';
	'METAMORA 0';
	'GREENVIEW 0';
	'CLINTON 2 0';
	'CLINTON 2 1';
	'BARTONVILLE 0';
	'BARTONVILLE 1';
	'BARTONVILLE 2';
	'BARTONVILLE 3';
	'BARTONVILLE 4';
	'NORMAL 1 0';
	'NORMAL 1 1';
	'LEXINGTON 0';
	'PEORIA 5 0';
	'MORTON 0';
	'PEKIN 1 0';
	'PEKIN 1 1';
	'PEKIN 1 2';
	'PEKIN 1 3';
	'NIANTIC 0';
	'COLFAX 0';
	'EL PASO 0';
	'TREMONT 0';
	'PEORIA 4 0';
	'PEORIA 3 0';
	'CHAMPAIGN 2 0';
	'EUREKA 0';
	'CHATHAM 0';
	'HOPEDALE 2 0';
	'HOPEDALE 2 1';
	'WASHINGTON 0';
	'SPRINGFIELD 5 0';
	'SPRINGFIELD 5 1';
	'SPRINGFIELD 5 2';
	'SPRINGFIELD 5 3';
	'SPRINGFIELD 5 4';
	'SPRINGFIELD 5 5';
	'SPRINGFIELD 5 6';
	'PEORIA 2 0';
	'PEORIA 2 1';
	'MT ZION 0';
	'BUFFALO 0';
	'SPRINGFIELD 4 0';
	'SPRINGFIELD 4 1';
	'CONGERVILLE 0';
	'CHAMPAIGN 1 0';
	'CHAMPAIGN 1 1';
	'CHAMPAIGN 1 2';
	'CHAMPAIGN 1 3';
	'CHAMPAIGN 1 4';
	'CHAMPAIGN 1 5';
	'CHAMPAIGN 1 6';
	'CHAMPAIGN 1 7';
	'FISHER 0';
	'HOMER 0';
	'TUSCOLA 1 0';
	'WHITE HEATH 0';
	'TOLONO 0';
	'SPRINGFIELD 3 0';
	'MACKINAW 0';
	'URBANA 1 0';
	'URBANA 1 1';
	'SAINT JOSEPH 0';
	'SPRINGFIELD 2 0';
	'SPRINGFIELD 2 1';
	'SPRINGFIELD 2 2';
	'MONTICELLO 0';
	'AUBURN 0';
	'HOPEDALE 1 0';
	'CLINTON 1 0';
	'CLINTON 1 1';
	'CLINTON 1 2';
	'MAHOMET 0';
	'VILLA GROVE 0';
	'BLOOMINGTON 1 0';
	'PEORIA 1 0';
	'SPRINGFIELD 1 0';
	'GIBSON CITY 1 0';
	'GIBSON CITY 1 1';
	'GIBSON CITY 1 2';
	'MAPLETON 0';
	'GIFFORD 0';
	'PETERSBURG 0';
};


## ACTIVSg200: a 200-bus synthetic Texas-style grid (Texas A&M, free dataset)
Synthetic but realistic, not utility data. Same pipeline and settings, gate re-calibrated on this grid. Saved results (1 to 2 seeds, small samples; one scenario takes about 2.7 s):

| Run | Bound coverage | PROCEED rate | Clean false alarm | Median error naive -> recovered (pu) |
|---|---|---|---|---|
| n = 60, seed 7 | 0.950 | 0.00 | 0.11 | 0.0080 -> 0.0073 |
| n = 60, seed 11 | 0.950 | 0.00 | 0.14 | 0.0067 -> 0.0067 |
| n = 150, seed 7 | 0.920 | 0.00 | 0.23 | 0.0063 -> 0.0062 |

Honest reading: the bound holds, but the gate refuses every case and recovery barely helps at 200 buses with settings tuned on IEEE 14. The cell below only converts the grid and checks that the power flow converges (about 20 seconds). The full run takes more than 30 minutes, so it is not run here.

In [ ]:
!pip -q install matpowercaseframes
!python activsg_eval.py --file case_ACTIVSg200.m --check | tail -n 16

## Launch the demo UI

In [ ]:
import subprocess, time, socket, re, os, urllib.request, stat
os.chdir("/content/gridsentinel") if os.path.exists("/content/gridsentinel") else None
subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
log = open("streamlit.log", "w")
subprocess.Popen(["streamlit","run","gridsentinel_app.py","--server.port","8501","--server.headless","true",
                  "--server.enableCORS","false","--server.enableXsrfProtection","false"], stdout=log, stderr=log)
# wait until Streamlit really listens (this was the cause of "connection refused")
for _ in range(120):
    try:
        socket.create_connection(("127.0.0.1", 8501), timeout=1).close(); break
    except OSError: time.sleep(1)
else:
    print(open("streamlit.log").read()); raise SystemExit("Streamlit did not start - see log above")
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)
tun = subprocess.Popen(["./cloudflared","tunnel","--url","http://127.0.0.1:8501","--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
t0 = time.time()
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: print("OPEN THIS LINK:", m.group(0)); break
    if time.time()-t0 > 90: print("no URL yet; check connection"); break


## ACTIVSg200 tuning (new): false alarms and operating tolerance
Pre-computed scenario pools (500 scenarios, z_iso=5.0, ~25 min to generate) are embedded, so this runs in seconds. Selection of z_iso and tolerance used VALIDATION data only. Simulation on a synthetic-but-realistic grid, not real data.

In [ ]:
%%writefile activsg_tune.py
#!/usr/bin/env python
"""NEW: per-grid operating-tolerance tuning for a large grid, done on a VALIDATION split, reported once on a held-out TEST split.
Why: on 200 buses the max-over-buses voltage error of even a clean estimate is larger than on 14 buses, so the fixed 0.01 pu tolerance
is almost never certifiable. Bound U does not depend on tol; only the PROCEED/CAUTION decision and the 'unsafe' label do.
Usage: py -3.12 activsg_tune.py --pools pool_101.npz pool_202.npz --out results_activsg200_tuned"""
import argparse, json, os, numpy as np
from gridsentinel_core import GateCalibration, PipelineConfig, decide_level
ap = argparse.ArgumentParser(); ap.add_argument("--pools", nargs="+", required=True); ap.add_argument("--out", default="results_activsg200_tuned")
ap.add_argument("--splits", type=int, default=20); ap.add_argument("--min_proceed", type=float, default=0.10); ap.add_argument("--max_unsafe", type=float, default=0.10)
a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
P = [np.load(p, allow_pickle=True) for p in a.pools]
F = np.vstack([p["F"] for p in P]); E = np.concatenate([p["E"] for p in P]); Ep = np.concatenate([p["Ep"] for p in P])
mode = np.concatenate([p["mode"] for p in P]); hard = np.concatenate([p["hard"] for p in P]); n_iso = np.concatenate([p["n_iso"] for p in P])
N = len(E); clean = mode == "none"
TOLS = [0.008, 0.010, 0.012, 0.015, 0.020, 0.025, 0.030]
info = {"N": int(N), "n_clean": int(clean.sum()), "err_clean_pct(50,90,95)": [float(x) for x in np.percentile(E[clean], [50, 90, 95])],
        "err_all_pct(50,90)": [float(x) for x in np.percentile(E, [50, 90])], "clean_false_alarm_default": float(np.mean((n_iso[clean] > 0) | hard[clean]))}
print(info)

def evaluate(cal, idx, tol, cfg):
    cfg.tol_pu = tol
    U = np.array([cal.bound(F[i], cfg.alpha) for i in idx]); cf = np.array([100 * cal.prob_within(F[i], tol) for i in idx])
    lv = np.array([decide_level(u, c, hard[i], cfg)[0] for u, c, i in zip(U, cf, idx)])
    pr = lv == "green"; uns = E[idx] > tol
    return {"proceed": float(pr.mean()), "unsafe_given_proceed": float(uns[pr].mean()) if pr.any() else float("nan"),
            "coverage": float(np.mean(E[idx] <= U)), "unsafe_rate": float(uns.mean())}

rows, chosen = [], []
for s in range(a.splits):
    rng = np.random.default_rng(1000 + s); perm = rng.permutation(N)
    k = N // 5; tr, ca, va, te = perm[:2 * k], perm[2 * k:3 * k], perm[3 * k:4 * k], perm[4 * k:]
    cal = GateCalibration.fit(F[tr], E[tr], F[ca], E[ca], 0.01, 0.10)
    best = None
    for t in TOLS:
        v = evaluate(cal, va, t, PipelineConfig())
        if v["proceed"] >= a.min_proceed and v["unsafe_given_proceed"] <= a.max_unsafe: best = t; break
    t_use = best if best is not None else TOLS[-1]
    tst = evaluate(cal, te, t_use, PipelineConfig()); tst0 = evaluate(cal, te, 0.01, PipelineConfig())
    sweep = {str(t): evaluate(cal, te, t, PipelineConfig()) for t in TOLS}
    rows.append({"split": s, "chosen_tol": t_use, "found": best is not None, "test_tuned": tst, "test_default_tol": tst0, "test_sweep": sweep})
    chosen.append(t_use)

def agg(key, sub):
    v = [r[sub][key] for r in rows if r[sub][key] == r[sub][key]]; return (float(np.mean(v)), float(np.std(v))) if v else (float("nan"), float("nan"))
summ = {"info": info, "chosen_tol_values": {str(t): int(chosen.count(t)) for t in TOLS}, "found_in_splits": int(sum(r["found"] for r in rows)),
        "default_tol_0.01": {k: agg(k, "test_default_tol") for k in ["proceed", "unsafe_given_proceed", "coverage"]},
        "tuned_tol": {k: agg(k, "test_tuned") for k in ["proceed", "unsafe_given_proceed", "coverage"]},
        "sweep_mean": {str(t): {k: float(np.nanmean([r["test_sweep"][str(t)][k] for r in rows])) for k in ["proceed", "unsafe_given_proceed", "coverage"]} for t in TOLS}}
json.dump({"summary": summ, "rows": rows}, open(os.path.join(a.out, "tuning.json"), "w"), indent=1)
print(json.dumps(summ, indent=1))


In [ ]:
import base64
open('poolz5_101.npz','wb').write(base64.b64decode('...'))
print('poolz5_101.npz written')

In [ ]:
import base64
open('poolz5_202.npz','wb').write(base64.b64decode('...'))
print('poolz5_202.npz written')

In [ ]:
!python activsg_tune.py --pools poolz5_101.npz poolz5_202.npz --out results_activsg200_tuned_z5 | tail -60

## Bring your own grid (MVP tool)
Give the tool a standard grid file and a CSV of meter readings; it returns PROCEED / CAUTION / VALIDATE with a calibrated error bound for every snapshot. Below: IEEE 14-bus example (about a minute). Simulation-validated prototype.

In [ ]:
%%writefile gridsentinel_cli.py
#!/usr/bin/env python
"""GridSentinel AI - "bring your own grid" command-line tool (NEW file; no existing file is changed).

Takes a standard grid model (MATPOWER .m, or case14/30/57/118) and a CSV of measurement snapshots and returns, for every snapshot:
attack status, isolated meters, recovered voltages, calibrated error bound, and PROCEED / CAUTION / VALIDATE.

  1) template : py -3.12 gridsentinel_cli.py template --grid case14 --out meas_template.csv
                (writes the exact column list, with units, plus 5 example snapshots incl. attacks)
  2) calibrate: py -3.12 gridsentinel_cli.py calibrate --grid case14 --cal cal_case14.json [--n 200] [--tol 0.01]
                (offline, uses the grid MODEL only - builds the Trust Gate; time grows with grid size)
  3) run      : py -3.12 gridsentinel_cli.py run --grid case14 --meas meas_template.csv --cal cal_case14.json --out decisions.csv

Measurement CSV: one row per snapshot, one column per meter (names exactly as in the template). Units: V in per unit, P in MW,
Q in MVAr, injections positive = generation, flows measured at the 'from' or 'to' end as named.
ASSUMPTIONS (state them): static AC snapshot, the grid model matches the real grid, meter noise sd = 0.004 pu (V) and 1 MW/MVAr (P/Q)
unless you change them with --sigma_v / --sigma_pq. Research prototype; simulation-validated only."""
import argparse, copy, json, os, sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import pandapower.networks as pn
import multi_grid_eval as mg
from gridsentinel_core import (GridModel, PipelineConfig, GateCalibration, build_calibration, run_pipeline,
                               make_scenarios, apply_attack)

BUILTIN = {"case14", "case30", "case57", "case118"}


def load_model(grid, sigma_v=0.004, sigma_pq=1.0):
    if grid in BUILTIN:
        fn = mg._ORIG[grid]
    elif os.path.exists(grid):
        from activsg_eval import load_net
        base = load_net(grid)
        fn = lambda: copy.deepcopy(base)
    else:
        sys.exit(f"grid '{grid}' not found: use case14/30/57/118 or a path to a MATPOWER .m file")
    saved = pn.case14
    pn.case14 = fn
    try:
        return GridModel(sigma_v=sigma_v, sigma_pq=sigma_pq)
    finally:
        pn.case14 = saved


def cmd_template(a):
    model = load_model(a.grid)
    cfg = PipelineConfig()
    rng = np.random.default_rng(a.seed)
    rows, labels = [], []
    modes = ["none", "none", "random_single", "multi_coordinated", "stealth_partial"]
    for sc, mode in zip(make_scenarios(model, len(modes), rng), modes):
        z, _ = apply_attack(model, sc, mode, rng, cfg)
        rows.append(z); labels.append(mode)
    df = pd.DataFrame(rows, columns=model.names)
    df.insert(0, "snapshot_id", [f"example_{i}_{m}" for i, m in enumerate(labels)])
    df.to_csv(a.out, index=False)
    pd.DataFrame({"column": model.names, "kind": model.kinds,
                  "unit": ["pu" if k == "V" else ("MW" if k[0] == "P" else "MVAr") for k in model.kinds]}).to_csv(
        os.path.splitext(a.out)[0] + "_columns.csv", index=False)
    print(f"{model.nb} buses, {model.nl} branches, {model.m} meters per snapshot. Wrote {a.out} and *_columns.csv")


def cmd_calibrate(a):
    model = load_model(a.grid, a.sigma_v, a.sigma_pq)
    cfg = PipelineConfig(tol_pu=a.tol, z_iso=a.z_iso, z_zero=a.z_iso + 2.0)
    n = a.n
    print(f"Building Trust Gate for {model.nb} buses from {2 * n} synthetic scenarios (model only, no real data)...", flush=True)
    cal = build_calibration(model, n, n, a.seed, cfg,
                            progress=lambda i, t: print(f"  {i}/{t}", flush=True) if i % 50 == 0 else None)
    meta = {"grid": a.grid, "tol": a.tol, "z_iso": a.z_iso, "sigma_v": a.sigma_v, "sigma_pq": a.sigma_pq, "n": n}
    d = json.loads(cal.to_json()); d["cli_meta"] = meta
    json.dump(d, open(a.cal, "w"))
    print("saved", a.cal)


def cmd_run(a):
    meta = json.load(open(a.cal)).get("cli_meta", {})
    tol, z_iso = meta.get("tol", 0.01), meta.get("z_iso", 4.5)
    model = load_model(a.grid, meta.get("sigma_v", 0.004), meta.get("sigma_pq", 1.0))
    cfg = PipelineConfig(tol_pu=tol, z_iso=z_iso, z_zero=z_iso + 2.0)
    cal = GateCalibration.from_json(open(a.cal).read())
    df = pd.read_csv(a.meas)
    missing = [c for c in model.names if c not in df.columns]
    if missing:
        sys.exit(f"{len(missing)} required columns missing, e.g. {missing[:5]}. Run 'template' to see the exact names.")
    out = []
    for _, r in df.iterrows():
        z = r[model.names].to_numpy(float)
        if not np.isfinite(z).all():
            out.append({"snapshot_id": r.get("snapshot_id", ""), "decision": "VALIDATE", "reason": "missing/NaN readings"}); continue
        res = run_pipeline(model, z, cfg, cal)
        lv = {"green": "PROCEED", "yellow": "CAUTION", "red": "VALIDATE"}[res.level]
        out.append({"snapshot_id": r.get("snapshot_id", ""), "decision": lv, "error_bound_pu": round(res.bound_pu, 5),
                    "p_within_tol_pct": round(res.conf, 1), "converged": res.converged, "n_isolated": len(res.iso_idx),
                    "isolated_meters": "; ".join(res.iso_names), "min_V_recovered": round(float(res.rec_V.min()), 4),
                    "max_V_recovered": round(float(res.rec_V.max()), 4), "runtime_ms": round(res.ms, 1),
                    "reason": " | ".join(res.reasons) if res.reasons else ""})
    o = pd.DataFrame(out); o.to_csv(a.out, index=False)
    print(o[["snapshot_id", "decision", "error_bound_pu", "n_isolated", "isolated_meters"]].to_string(index=False))
    print(f"\nwrote {a.out}  (tolerance {tol} pu; 90% target bound; simulation-validated prototype)")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)
    t = sp.add_parser("template"); t.add_argument("--grid", required=True); t.add_argument("--out", default="meas_template.csv"); t.add_argument("--seed", type=int, default=5)
    c = sp.add_parser("calibrate"); c.add_argument("--grid", required=True); c.add_argument("--cal", required=True)
    c.add_argument("--n", type=int, default=200); c.add_argument("--tol", type=float, default=0.01); c.add_argument("--z_iso", type=float, default=4.5)
    c.add_argument("--sigma_v", type=float, default=0.004); c.add_argument("--sigma_pq", type=float, default=1.0); c.add_argument("--seed", type=int, default=7)
    r = sp.add_parser("run"); r.add_argument("--grid", required=True); r.add_argument("--meas", required=True); r.add_argument("--cal", required=True); r.add_argument("--out", default="decisions.csv")
    a = ap.parse_args()
    {"template": cmd_template, "calibrate": cmd_calibrate, "run": cmd_run}[a.cmd](a)


if __name__ == "__main__":
    main()


In [ ]:
!python gridsentinel_cli.py template --grid case14 --out meas_template.csv
!python gridsentinel_cli.py calibrate --grid case14 --cal cal_case14.json --n 100 | tail -2
!python gridsentinel_cli.py run --grid case14 --meas meas_template.csv --cal cal_case14.json --out decisions.csv

In [ ]:
import pandas as pd
pd.read_csv('decisions.csv')

## Bring-your-own-grid interface (live)
Opens the same tool with an interface: pick a grid, build the Trust Gate, load readings, see decisions. Run the cell and open the link.

In [ ]:
%%writefile gridsentinel_byog_app.py
"""GridSentinel AI - Bring-your-own-grid interface (NEW file; nothing existing is changed).
Run:  py -3.12 -m streamlit run gridsentinel_byog_app.py
Pick a grid (built-in or your own MATPOWER .m), build the Trust Gate, load a CSV of meter readings, get a decision per snapshot.
Simulation-validated research prototype."""
import os, tempfile, warnings
import numpy as np
import pandas as pd
import streamlit as st
warnings.filterwarnings("ignore")
import gridsentinel_cli as gc
from gridsentinel_core import PipelineConfig, GateCalibration, build_calibration, run_pipeline, make_scenarios, apply_attack

st.set_page_config(page_title="GridSentinel AI - your grid", page_icon="⚡", layout="wide")
COL = {"PROCEED": "#1b8a3a", "CAUTION": "#c98a00", "VALIDATE": "#c0392b"}
LV = {"green": "PROCEED", "yellow": "CAUTION", "red": "VALIDATE"}
st.title("⚡ GridSentinel AI: bring your own grid")
st.caption("Give it a grid model and meter readings. It scores each meter, isolates suspicious ones, recovers the grid state and says whether to PROCEED, take CAUTION or VALIDATE, with a calibrated error bound. Validated in simulation only.")

with st.sidebar:
    st.header("1 · Grid")
    choice = st.selectbox("Grid model", ["case14", "case30", "case57", "case118", "Upload my own (MATPOWER .m)"])
    grid = choice
    if choice.startswith("Upload"):
        up = st.file_uploader("MATPOWER case file (.m)", type=["m"])
        if up is None:
            st.info("Upload a .m file to continue."); st.stop()
        path = os.path.join(tempfile.gettempdir(), "gs_" + up.name); open(path, "wb").write(up.getvalue()); grid = path
    st.header("2 · Trust Gate")
    tol = st.number_input("Tolerance (pu): error you can accept", 0.005, 0.1, 0.01, 0.005, format="%.3f")
    z_iso = st.number_input("Isolation threshold (z)", 3.5, 8.0, 4.5, 0.5)
    n = st.slider("Calibration scenarios (each split)", 40, 300, 200, 20, help="More is slower and tighter. 14 buses: seconds. 200 buses: tens of minutes.")

@st.cache_resource(show_spinner="Loading grid model...")
def get_model(g):
    return gc.load_model(g)

model = get_model(grid)
cfg = PipelineConfig(tol_pu=tol, z_iso=z_iso, z_zero=z_iso + 2.0)
st.sidebar.success(f"{model.nb} buses · {model.nl} branches · {model.m} meters per snapshot")
key = (grid, tol, z_iso, n)
if st.sidebar.button("Build the Trust Gate", type="primary"):
    bar = st.sidebar.progress(0.0, text="Building from the grid model (no real data needed)...")
    st.session_state["cal"] = (key, build_calibration(model, n, n, 7, cfg, progress=lambda i, t: bar.progress(min(i / t, 1.0))))
    bar.empty()
ok = "cal" in st.session_state and st.session_state["cal"][0] == key
st.sidebar.write("Gate: " + ("✅ ready" if ok else "⚠️ not built for these settings"))

st.header("3 · Meter readings")
src = st.radio("Readings", ["Example snapshots (simulated, with attacks)", "Upload my CSV"], horizontal=True)
cols_df = pd.DataFrame({"column": model.names, "kind": model.kinds})
if src.startswith("Upload"):
    f = st.file_uploader("CSV: one row per snapshot, one column per meter (V in pu, P in MW, Q in MVAr)", type=["csv"])
    df = pd.read_csv(f) if f else None
    rng = np.random.default_rng(5); modes = ["none", "none", "random_single", "multi_coordinated", "stealth_partial"]
    ex = pd.DataFrame([apply_attack(model, sc, m, rng, cfg)[0] for sc, m in zip(make_scenarios(model, 5, rng), modes)], columns=model.names)
    ex.insert(0, "snapshot_id", [f"example_{i}_{m}" for i, m in enumerate(modes)])
    st.download_button("Download a template CSV (exact column names)", ex.to_csv(index=False), "meter_template.csv")
else:
    rng = np.random.default_rng(5); modes = ["none", "none", "random_single", "multi_coordinated", "stealth_partial"]
    df = pd.DataFrame([apply_attack(model, sc, m, rng, cfg)[0] for sc, m in zip(make_scenarios(model, 5, rng), modes)], columns=model.names)
    df.insert(0, "snapshot_id", [f"example_{i}_{m}" for i, m in enumerate(modes)])
if df is not None:
    st.dataframe(df.iloc[:, :9], use_container_width=True, height=180); st.caption(f"{len(df)} snapshot(s), {df.shape[1]} columns")

st.header("4 · Decisions")
if df is None:
    st.info("Upload a CSV or use the example snapshots.")
elif not ok:
    st.warning("Build the Trust Gate first (left panel).")
else:
    missing = [c for c in model.names if c not in df.columns]
    if missing:
        st.error(f"{len(missing)} required columns are missing, for example {missing[:4]}. Download the template to see the exact names."); st.stop()
    cal = st.session_state["cal"][1]; rows, results = [], []
    for k, r in df.iterrows():
        z = r[model.names].to_numpy(float)
        if not np.isfinite(z).all():
            rows.append({"snapshot": r.get("snapshot_id", k), "decision": "VALIDATE", "error_bound_pu": None, "isolated": "missing / NaN readings"}); results.append(None); continue
        res = run_pipeline(model, z, cfg, cal); results.append(res)
        rows.append({"snapshot": r.get("snapshot_id", k), "decision": LV[res.level], "error_bound_pu": round(res.bound_pu, 4),
                     "p_within_tol_%": round(res.conf), "n_isolated": len(res.iso_idx), "isolated": "; ".join(res.iso_names[:6]) + (" ..." if len(res.iso_names) > 6 else "")})
    out = pd.DataFrame(rows)
    c = st.columns(3)
    for col, d in zip(c, ["PROCEED", "CAUTION", "VALIDATE"]):
        col.markdown(f"<div style='background:{COL[d]};color:white;border-radius:12px;padding:12px;text-align:center'><b style='font-size:26px'>{(out.decision == d).sum()}</b><br>{d}</div>", unsafe_allow_html=True)
    st.dataframe(out.style.map(lambda v: f"background-color:{COL.get(v, 'white')};color:white" if v in COL else "", subset=["decision"]), use_container_width=True)
    st.download_button("Download decisions (CSV)", out.to_csv(index=False), "decisions.csv")
    pick = st.selectbox("Look inside one snapshot", [str(x) for x in out.snapshot])
    i = [str(x) for x in out.snapshot].index(pick); res = results[i]
    if res is not None:
        st.subheader(f"{pick}: {LV[res.level]}")
        st.write(f"Calibrated error bound **{res.bound_pu:.4f} pu** · chance the error is within {tol} pu: **{res.conf:.0f}%** · isolated meters: **{len(res.iso_idx)}**")
        if res.reasons: st.write("Reasons: " + " | ".join(res.reasons))
        st.bar_chart(pd.DataFrame({"recovered voltage (pu)": res.rec_V}, index=[f"bus {b + 1}" for b in range(len(res.rec_V))]))
st.caption("Research prototype validated in simulation. A pilot on utility data is the next step.")


In [ ]:
import subprocess, time, socket, re, os
os.chdir("/content/gridsentinel")
log2 = open("streamlit_byog.log", "w")
subprocess.Popen(["streamlit","run","gridsentinel_byog_app.py","--server.port","8502","--server.headless","true",
                  "--server.enableCORS","false","--server.enableXsrfProtection","false"], stdout=log2, stderr=log2)
for _ in range(120):
    try:
        socket.create_connection(("127.0.0.1", 8502), timeout=1).close(); break
    except OSError: time.sleep(1)
else:
    print(open("streamlit_byog.log").read()); raise SystemExit("Streamlit did not start - see log above")
tun2 = subprocess.Popen(["./cloudflared","tunnel","--url","http://127.0.0.1:8502","--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
t0 = time.time()
for line in tun2.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: print("OPEN THIS LINK (bring-your-own-grid interface):", m.group(0)); break
    if time.time()-t0 > 90: print("no URL yet; check connection"); break
